# Lumpy 02: Standalone Forecasting Workflow

This notebook contains the complete retained modelling process. It starts from the prepared sales and external data, builds each required forecast route, selects the locked segment process, consolidates the result, calibrates it, audits it and writes the final planning outputs.

It does **not** read another notebook or use a forecast file created by the archived experiments. Intermediate files are created by earlier sections of this notebook under `results/lumpy_02_forecasting/stages/` and `docs/lumpy_02_forecasting/stages/`.

Run `Lumpy 01: Data Check` first, then run this notebook from top to bottom. The full rebuild is computationally heavy because it includes the historical validation needed to show how each retained process was chosen.


## Libraries and project setup

All Python, modelling and project libraries are loaded once here. The later stages contain only the task-specific forecasting logic and short module-alias assignments where a stage uses a particular helper name.


In [ ]:
from __future__ import annotations

import importlib
import itertools
import json
import sys
import time
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [start, start.parent, start.parent.parent, Path(r"D:/Lumpy_Fellas-/Inchscape Repo")]
    for candidate in candidates:
        if (candidate / "src/lumpy_forecasting.py").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate the Lumpy Fellas project root.")


project_root = find_project_root()
src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import lumpy_ab_optimization
import lumpy_block_hybrid
import lumpy_candidate_router
import lumpy_cold_start_calibration
import lumpy_consolidated_scorecard
import lumpy_delivery_output
import lumpy_error_attribution
import lumpy_established_tournament
import lumpy_forecasting
import lumpy_group_occurrence
import lumpy_internal_hybrid
import lumpy_launch_hurdle
import lumpy_layered_ensemble
import lumpy_methodology_audit
import lumpy_new_sku_calibration
import lumpy_new_sku_tournament
import lumpy_occurrence_budget
import lumpy_online_experts
import lumpy_operational_policy
import lumpy_quantile_calibration
import lumpy_rare_event
import lumpy_segment_policy
import lumpy_sku_router
import lumpy_temporal_memory

print("Libraries loaded")
print("Project root:", project_root)


## Workflow map

Each task has its own retained process, but every process uses the same final dates, three-month information gap, six forecast blocks and per-SKU WMAPE decision order.

| Task | Retained process |
|---|---|
| Core benchmark | Three-month block intermittent-demand and hurdle models |
| Recurring | Cohort models plus personalised expert memory |
| Occasional | Calibrated hurdle with cautious temporal memory |
| Rare | Event hurdle with family/subfamily occurrence priors |
| Cold start | Genuine historical launch hurdle |
| New | Short-history and analogue blend with selective calibration |
| Developing | Hurdle/analogue blend with high-volume routing |
| Dormant | Governance forecast with manual lifecycle review |


## Workflow diagram

This shows how the shared foundation splits into specialist SKU routes and then returns to one final forecast and planning output.

```mermaid
flowchart TD
    D["Prepared data from Lumpy 01"] --> S1["1. Core three-month block forecasts"]
    S1 --> S2["2. SKU classification and cold-start candidates"]
    S2 --> S3["3. Chronology-safe baseline"]

    S3 --> R4["4. Recurring A/B route"]
    R4 --> R5["5. Recurring candidate optimisation"]
    R5 --> R6["6. Recurring layered ensemble"]
    R6 --> R7["7. Recurring personalised memory"]
    S3 --> R8["8. Recurring C strict-cutoff route"]
    R7 --> R9["9. Final recurring A/B/C result"]
    R8 --> R9

    S3 --> O10["10. Occasional baseline"]
    O10 --> O11["11. Occasional calibrated hurdle"]
    O11 --> O12["12. Occasional SKU router"]
    O12 --> O13["13. Occasional temporal memory"]

    S3 --> A14["14. Rare-demand event model"]
    A14 --> A15["15. Rare group occurrence priors"]

    S2 --> C16["16. Cold-start launch hurdle"]
    S2 --> N17["17. New-SKU short-history blend"]
    N17 --> N18["18. New-SKU level calibration"]
    S2 --> G19["19. Developing-SKU model route"]
    G19 --> G20["20. Developing high-volume routing"]

    R9 --> C21["21. Consolidated segment champions"]
    O13 --> C21
    A15 --> C21
    C16 --> C21
    N18 --> C21
    G20 --> C21
    S3 -->|"Dormant governance route"| C21

    C21 --> E22["22. Bias and error attribution"]
    E22 --> K23["23. Selective historical calibration"]
    K23 --> T24["24. SKU delivery tables"]
    T24 --> P25["25. Planning policy"]
    P25 --> M26["26. Final methodology audit"]
    M26 --> F["Final 690-SKU and 637-SKU outputs"]
```


## 1. Core three-month block forecasts

Builds the shared model library. Every SKU is forecast over an 18-month horizon, following a three-month information gap, as six three-month blocks. This tests intermittent-demand baselines and hurdle/XGBoost models using demand history only.


In [ ]:
lf = lumpy_forecasting
history_tools = lumpy_internal_hybrid
bh = lumpy_block_hybrid

candidate_roots = [Path.cwd(), Path.cwd().parent, Path(r"D:/Lumpy_Fellas-/Inchscape Repo")]
project_root = next(root.resolve() for root in candidate_roots if (root / "src" / "lumpy_forecasting.py").exists())
src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
importlib.reload(lf)
importlib.reload(history_tools)
importlib.reload(bh)

SKU_COLUMN = lf.SKU_COLUMN
DATE_COLUMN = lf.MONTH_COLUMN
TARGET_COLUMN = lf.TARGET_COLUMN

OUTPUT_DIRECTORY = project_root / "results" / "lumpy_02_forecasting/stages/core_block"
CHECKPOINT_DIRECTORY = OUTPUT_DIRECTORY / "checkpoints_v1"
REVIEW_DIRECTORY = project_root / "docs" / "lumpy_02_forecasting/stages/core_block"
for directory in (OUTPUT_DIRECTORY, CHECKPOINT_DIRECTORY, REVIEW_DIRECTORY):
    directory.mkdir(parents=True, exist_ok=True)

POLICIES = {
    "required_18m_gap3": {"horizon_months": 18, "gap_months": 3, "required": True},
    "diagnostic_18m_gap1": {"horizon_months": 18, "gap_months": 1, "required": False},
    "diagnostic_18m_gap6": {"horizon_months": 18, "gap_months": 6, "required": False},
    "diagnostic_9m_gap1": {"horizon_months": 9, "gap_months": 1, "required": False},
    "diagnostic_9m_gap3": {"horizon_months": 9, "gap_months": 3, "required": False},
    "diagnostic_9m_gap6": {"horizon_months": 9, "gap_months": 6, "required": False},
}

BASE_CONFIG = lf.LumpyConfig(
    variant="all_sku_history",
    train_months=48,
    gap_months=3,
    test_months=18,
    step_months=3,
    min_train_months=18,
    max_folds=6,
    external_mode="off",
)
RUN_BACKTEST = True
RESUME_FROM_CHECKPOINTS = True
SAVE_CHECKPOINTS = True
INNER_VALIDATION_MONTHS = 6

print("External features: OFF")
print("Policies:", len(POLICIES))
print("ML models:", len(bh.MODEL_SPECS))
print("Baselines eligible for selection:", list(bh.BASELINE_LABELS))

sales, external = lf.load_lumpy_inputs(project_root, BASE_CONFIG)
model_data, _ = lf.build_lumpy_model_frame(sales, external, BASE_CONFIG)
print(f"All lumpy SKUs retained: {model_data[SKU_COLUMN].nunique():,}")
print(f"History: {model_data[DATE_COLUMN].nunique():,} months | {model_data[DATE_COLUMN].min():%Y-%m} to {model_data[DATE_COLUMN].max():%Y-%m}")

display(pd.DataFrame([{"policy": key, **value} for key, value in POLICIES.items()]))
display(pd.DataFrame([{"model_key": key, **spec.__dict__} for key, spec in bh.MODEL_SPECS.items()]))

def make_inner_fold(train, gap_months, validation_months=INNER_VALIDATION_MONTHS):
    months = pd.Series(pd.to_datetime(train[DATE_COLUMN].dropna().unique())).sort_values().tolist()
    if len(months) < 12 + gap_months + validation_months:
        return None
    validation_end = months[-1]
    validation_start = validation_end - pd.DateOffset(months=validation_months - 1)
    inner_train_end = validation_start - pd.DateOffset(months=gap_months + 1)
    inner_train = train.loc[train[DATE_COLUMN].le(inner_train_end)].copy()
    validation = train.loc[train[DATE_COLUMN].between(validation_start, validation_end)].copy()
    if inner_train[DATE_COLUMN].nunique() < 12 or validation[DATE_COLUMN].nunique() < validation_months:
        return None
    return inner_train, validation


def add_split_metadata(frame, split, policy):
    frame = frame.copy()
    frame["policy"] = policy
    for column, value in split.items():
        frame[column] = value
    return frame


def monthly_rolling_diagnostic(block_forecast, actual_monthly):
    rows = []
    for row in block_forecast.itertuples(index=False):
        for offset in range(3):
            rows.append({
                SKU_COLUMN: row.sku_id,
                DATE_COLUMN: pd.Timestamp(row.block_start) + pd.DateOffset(months=offset),
                "forecast": float(row.forecast) / 3.0,
            })
    monthly = pd.DataFrame(rows).merge(
        actual_monthly[[SKU_COLUMN, DATE_COLUMN, TARGET_COLUMN]], on=[SKU_COLUMN, DATE_COLUMN], how="left"
    )
    monthly[TARGET_COLUMN] = monthly[TARGET_COLUMN].fillna(0.0)
    monthly["absolute_error"] = (monthly[TARGET_COLUMN] - monthly["forecast"]).abs()
    monthly = monthly.sort_values([SKU_COLUMN, DATE_COLUMN])
    monthly["rolling_abs"] = monthly.groupby(SKU_COLUMN).absolute_error.transform(lambda x: x.rolling(3, min_periods=1).sum())
    monthly["rolling_actual"] = monthly.groupby(SKU_COLUMN)[TARGET_COLUMN].transform(lambda x: x.rolling(3, min_periods=1).sum())
    monthly["rolling_wmape"] = np.where(monthly.rolling_actual.gt(0), 100 * monthly.rolling_abs / monthly.rolling_actual, np.nan)
    sku = monthly.groupby(SKU_COLUMN, as_index=False).agg(
        mean_rolling_3m_wmape=("rolling_wmape", "mean"), actual_total=(TARGET_COLUMN, "sum")
    )
    valid = sku.loc[sku.actual_total.gt(0) & sku.mean_rolling_3m_wmape.notna()]
    return {
        "monthly_rolling_valid_skus": len(valid),
        "monthly_rolling_under_50_skus": int(valid.mean_rolling_3m_wmape.lt(50).sum()),
        "monthly_rolling_under_70_skus": int(valid.mean_rolling_3m_wmape.lt(70).sum()),
        "monthly_rolling_under_100_skus": int(valid.mean_rolling_3m_wmape.lt(100).sum()),
        "monthly_rolling_median_wmape": valid.mean_rolling_3m_wmape.median() if len(valid) else np.nan,
    }, sku

if RUN_BACKTEST:
    all_forecasts = []
    all_recipe_scores = []
    errors = []
    policy_splits = {}
    policy_job_counts = {}
    for policy, settings in POLICIES.items():
        config = replace(
            BASE_CONFIG,
            gap_months=settings["gap_months"],
            test_months=settings["horizon_months"],
        )
        splits = lf.make_backtest_splits(model_data, config)
        policy_splits[policy] = splits
        policy_job_counts[policy] = len(splits) * len(bh.MODEL_SPECS)
    total_jobs = sum(policy_job_counts.values())
    job = 0
    started = time.perf_counter()

    for policy, settings in POLICIES.items():
        splits = policy_splits[policy]
        print(f"\n{policy} | {len(splits)} folds | {settings['horizon_months']}m horizon | {settings['gap_months']}m gap", flush=True)
        for fold_position, (_, split) in enumerate(splits.iterrows(), start=1):
            train, test = lf.split_train_test(model_data, split)
            prepared = bh.prepare_fold(
                train, test, SKU_COLUMN, DATE_COLUMN, TARGET_COLUMN,
                settings["gap_months"], settings["horizon_months"],
            )
            baseline_checkpoint = CHECKPOINT_DIRECTORY / f"{policy}__fold_{int(split['fold_id']):03d}__baselines.csv"
            if RESUME_FROM_CHECKPOINTS and baseline_checkpoint.exists():
                baselines = pd.read_csv(baseline_checkpoint, parse_dates=["block_start", "train_start", "train_end", "test_start", "test_end"])
            else:
                baselines = add_split_metadata(bh.baseline_forecasts(prepared), split, policy)
                if SAVE_CHECKPOINTS:
                    baselines.to_csv(baseline_checkpoint, index=False)
            all_forecasts.append(baselines)

            inner = make_inner_fold(train, settings["gap_months"])
            inner_prepared = None
            if inner is not None:
                inner_train, validation = inner
                try:
                    inner_prepared = bh.prepare_fold(
                        inner_train, validation, SKU_COLUMN, DATE_COLUMN, TARGET_COLUMN,
                        settings["gap_months"], INNER_VALIDATION_MONTHS,
                    )
                except ValueError as exc:
                    print(f"    inner tuning unavailable for this early fold: {exc}", flush=True)
                    inner_prepared = None

            for model_position, model_key in enumerate(bh.MODEL_SPECS, start=1):
                job += 1
                checkpoint = CHECKPOINT_DIRECTORY / f"{policy}__fold_{int(split['fold_id']):03d}__{model_key}.csv"
                recipe_checkpoint = CHECKPOINT_DIRECTORY / f"{policy}__fold_{int(split['fold_id']):03d}__{model_key}__recipe_scores.csv"
                model_started = time.perf_counter()
                print(f"  [{job}/{total_jobs}] fold {fold_position}/{len(splits)} | model {model_position}/{len(bh.MODEL_SPECS)} | {model_key}", flush=True)
                try:
                    if RESUME_FROM_CHECKPOINTS and checkpoint.exists():
                        forecast = pd.read_csv(checkpoint, parse_dates=["block_start", "train_start", "train_end", "test_start", "test_end"])
                        all_forecasts.append(forecast)
                        if recipe_checkpoint.exists():
                            all_recipe_scores.append(pd.read_csv(recipe_checkpoint))
                        print(f"      loaded checkpoint | rows: {len(forecast):,}", flush=True)
                        continue

                    if inner_prepared is not None:
                        inner_components = bh.fit_components(model_key, inner_prepared, BASE_CONFIG.random_state)
                        recipe, recipe_scores = bh.choose_recipe(model_key, inner_components)
                        recipe_scores["policy"] = policy
                        recipe_scores["fold_id"] = int(split["fold_id"])
                        all_recipe_scores.append(recipe_scores)
                        if SAVE_CHECKPOINTS:
                            recipe_scores.to_csv(recipe_checkpoint, index=False)
                    else:
                        architecture = bh.MODEL_SPECS[model_key].architecture
                        neutral_mode = "direct" if architecture == "direct" else "expected"
                        recipe = next(
                            candidate for candidate in bh.recipe_grid(architecture)
                            if candidate["mode"] == neutral_mode and candidate["scale"] == 1.0
                        )

                    components = bh.fit_components(model_key, prepared, BASE_CONFIG.random_state)
                    forecast = add_split_metadata(bh.compose(model_key, components, recipe), split, policy)
                    if SAVE_CHECKPOINTS:
                        forecast.to_csv(checkpoint, index=False)
                    all_forecasts.append(forecast)
                    print(f"      {bh.recipe_name(recipe)} | {time.perf_counter() - model_started:,.1f}s", flush=True)
                except Exception as exc:
                    errors.append({"policy": policy, "fold_id": split["fold_id"], "model_key": model_key, "error": repr(exc)})
                    print(f"      FAILED | {repr(exc)}", flush=True)

    block_forecasts = pd.concat(all_forecasts, ignore_index=True, sort=False)
    recipe_scores = pd.concat(all_recipe_scores, ignore_index=True, sort=False) if all_recipe_scores else pd.DataFrame()
    backtest_errors = pd.DataFrame(errors)
    block_forecasts.to_csv(OUTPUT_DIRECTORY / "core_block_block_backtest_forecasts.csv", index=False)
    recipe_scores.to_csv(REVIEW_DIRECTORY / "core_block_recipe_validation_scores.csv", index=False)
    backtest_errors.to_csv(OUTPUT_DIRECTORY / "core_block_backtest_errors.csv", index=False)
    print(f"Finished in {(time.perf_counter() - started) / 60:,.1f} minutes | errors: {len(backtest_errors)}")
else:
    block_forecasts = pd.read_csv(
        OUTPUT_DIRECTORY / "core_block_block_backtest_forecasts.csv",
        parse_dates=["block_start", "train_start", "train_end", "test_start", "test_end"],
    )
    recipe_scores = pd.read_csv(REVIEW_DIRECTORY / "core_block_recipe_validation_scores.csv")
    backtest_errors = pd.read_csv(OUTPUT_DIRECTORY / "core_block_backtest_errors.csv")

policy_summary_rows = []
holdout_sku_frames = []
selection_frames = []
selected_holdout_frames = []
monthly_sku_frames = []

for policy, policy_data in block_forecasts.groupby("policy", sort=False):
    latest_fold = int(
        policy_data[["fold_id", "test_end"]].drop_duplicates().sort_values(["test_end", "fold_id"]).iloc[-1].fold_id
    )
    development = policy_data.loc[policy_data.fold_id.ne(latest_fold)]
    holdout = policy_data.loc[policy_data.fold_id.eq(latest_fold)]

    development_scores = []
    development_global = []
    for model_key, group in development.groupby("model_key", sort=False):
        sku_scores, summary = bh.score_forecast(group)
        sku_scores["model_key"] = model_key
        sku_scores["model"] = group.model.iloc[0]
        development_scores.append(sku_scores)
        development_global.append({"model_key": model_key, "model": group.model.iloc[0], **summary})
    development_scores = pd.concat(development_scores, ignore_index=True)
    development_global = pd.DataFrame(development_global).sort_values(
        ["under_50_skus", "under_70_skus", "under_100_skus", "median_sku_block_wmape"],
        ascending=[False, False, False, True],
    )
    fallback_model = development_global.iloc[0].model_key

    sku_selection = (
        development_scores.assign(score=lambda x: x.block_wmape_percent.fillna(np.inf))
        .sort_values(["sku_id", "score", "absolute_error", "model_key"])
        .groupby("sku_id", as_index=False).head(1)[["sku_id", "model_key", "model", "score"]]
        .rename(columns={"model_key": "selected_model_key", "model": "selected_model"})
    )
    sku_selection["policy"] = policy
    selection_frames.append(sku_selection)

    holdout = holdout.merge(sku_selection[["sku_id", "selected_model_key"]], on="sku_id", how="left")
    holdout["selected_model_key"] = holdout.selected_model_key.fillna(fallback_model)
    selected = holdout.loc[holdout.model_key.eq(holdout.selected_model_key)].copy()
    selected_holdout_frames.append(selected)
    holdout_per_sku, summary = bh.score_forecast(selected)
    holdout_per_sku["policy"] = policy
    holdout_sku_frames.append(holdout_per_sku)

    settings = POLICIES[policy]
    actual_monthly = model_data.loc[
        model_data[DATE_COLUMN].between(selected.test_start.iloc[0], selected.test_end.iloc[0])
    ]
    monthly_summary, monthly_sku = monthly_rolling_diagnostic(selected, actual_monthly)
    monthly_sku["policy"] = policy
    monthly_sku_frames.append(monthly_sku)

    zero = selected.copy()
    zero["forecast"] = 0.0
    _, zero_summary = bh.score_forecast(zero)
    policy_summary_rows.append({
        "policy": policy,
        "required": settings["required"],
        "horizon_months": settings["horizon_months"],
        "gap_months": settings["gap_months"],
        "holdout_fold_id": latest_fold,
        **summary,
        **monthly_summary,
        "zero_audit_under_50_skus": zero_summary["under_50_skus"],
        "zero_audit_under_70_skus": zero_summary["under_70_skus"],
        "zero_audit_portfolio_wmape": zero_summary["portfolio_block_wmape"],
    })

policy_summary = pd.DataFrame(policy_summary_rows).sort_values(
    ["under_50_skus", "under_70_skus", "under_100_skus", "median_sku_block_wmape"],
    ascending=[False, False, False, True],
)
holdout_per_sku = pd.concat(holdout_sku_frames, ignore_index=True)
sku_selections = pd.concat(selection_frames, ignore_index=True)
selected_holdout_forecasts = pd.concat(selected_holdout_frames, ignore_index=True)
monthly_sku_results = pd.concat(monthly_sku_frames, ignore_index=True)

policy_summary.to_csv(REVIEW_DIRECTORY / "core_block_policy_holdout_summary.csv", index=False)
holdout_per_sku.to_csv(REVIEW_DIRECTORY / "core_block_policy_holdout_per_sku.csv", index=False)
sku_selections.to_csv(REVIEW_DIRECTORY / "core_block_development_selected_model_per_sku.csv", index=False)
monthly_sku_results.to_csv(REVIEW_DIRECTORY / "core_block_monthly_rolling_holdout_per_sku.csv", index=False)
selected_holdout_forecasts.to_csv(OUTPUT_DIRECTORY / "core_block_selected_holdout_block_forecasts.csv", index=False)

display(policy_summary)

required_policy = "required_18m_gap3"
required_data = block_forecasts.loc[block_forecasts.policy.eq(required_policy)]
required_holdout_fold = int(required_data[["fold_id", "test_end"]].drop_duplicates().sort_values(["test_end", "fold_id"]).iloc[-1].fold_id)
required_holdout = required_data.loc[required_data.fold_id.eq(required_holdout_fold)]
model_rows = []
for model_key, group in required_holdout.groupby("model_key", sort=False):
    _, summary = bh.score_forecast(group)
    model_rows.append({"model_key": model_key, "model": group.model.iloc[0], **summary})
required_model_summary = pd.DataFrame(model_rows).sort_values(
    ["under_50_skus", "under_70_skus", "under_100_skus", "median_sku_block_wmape"],
    ascending=[False, False, False, True],
)
required_model_summary.to_csv(REVIEW_DIRECTORY / "core_block_required_policy_model_summary.csv", index=False)
display(required_model_summary)

plot_data = selected_holdout_forecasts.loc[selected_holdout_forecasts.policy.eq("required_18m_gap3")]
top_skus = plot_data.groupby("sku_id").target.sum().sort_values(ascending=False).head(6).index
fig, axes = plt.subplots(3, 2, figsize=(15, 11))
for axis, sku in zip(axes.flat, top_skus):
    rows = plot_data.loc[plot_data.sku_id.eq(sku)].sort_values("block_start")
    axis.plot(rows.block_start, rows.target, marker="o", linewidth=2, label="Actual 3-month total")
    axis.plot(rows.block_start, rows.forecast, marker="o", linewidth=2, label="Forecast 3-month total")
    axis.set_title(str(sku))
    axis.grid(alpha=0.25)
    axis.legend()
for axis in axes.flat[len(top_skus):]:
    axis.set_visible(False)
fig.suptitle("Required 18-month / 3-month-gap holdout: actual vs forecast", fontsize=14)
fig.tight_layout()
plt.show()

print("Policy decision table:", REVIEW_DIRECTORY / "core_block_policy_holdout_summary.csv")
print("Individual SKU block results:", REVIEW_DIRECTORY / "core_block_policy_holdout_per_sku.csv")
print("Original rolling metric by SKU:", REVIEW_DIRECTORY / "core_block_monthly_rolling_holdout_per_sku.csv")
print("Required-policy model comparison:", REVIEW_DIRECTORY / "core_block_required_policy_model_summary.csv")


## 2. SKU classification and cold-start candidates

Classifies SKUs by lifecycle, demand frequency, recency, demand size and ABC tier. It also creates peer-based candidates for SKUs with little or no history.


In [ ]:
lf = lumpy_forecasting
bh = lumpy_block_hybrid
sr = lumpy_sku_router

candidate_roots = [Path.cwd(), Path.cwd().parent, Path(r"D:/Lumpy_Fellas-/Inchscape Repo")]
project_root = next(root.resolve() for root in candidate_roots if (root / "src" / "lumpy_forecasting.py").exists())
src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
importlib.reload(lf)
importlib.reload(bh)
importlib.reload(sr)

OUTPUT_DIRECTORY = project_root / "results" / "lumpy_02_forecasting/stages/sku_routing"
REVIEW_DIRECTORY = project_root / "docs" / "lumpy_02_forecasting/stages/sku_routing"
for directory in (OUTPUT_DIRECTORY, REVIEW_DIRECTORY):
    directory.mkdir(parents=True, exist_ok=True)

CONFIG = lf.LumpyConfig(
    variant="all_sku_history",
    train_months=48,
    gap_months=3,
    test_months=18,
    step_months=3,
    min_train_months=18,
    max_folds=6,
    external_mode="off",
)
POLICY = "required_18m_gap3"
RANDOM_STATE = CONFIG.random_state

print("This notebook reuses core three-month block forecasts forecasts; no block models will be retrained.")

sales, external = lf.load_lumpy_inputs(project_root, CONFIG)
model_data, _ = lf.build_lumpy_model_frame(sales, external, CONFIG)
metadata = sr.extract_static_metadata(sales)

forecast_path = project_root / "results" / "lumpy_02_forecasting/stages/core_block" / "core_block_block_backtest_forecasts.csv"
if not forecast_path.exists():
    raise FileNotFoundError(f"Core three-month block forecasts forecasts are required: {forecast_path}")
block_forecasts = pd.read_csv(
    forecast_path,
    parse_dates=["block_start", "train_start", "train_end", "test_start", "test_end"],
)
required_forecasts = block_forecasts.loc[block_forecasts.policy.eq(POLICY)].copy()
if required_forecasts.empty:
    raise ValueError(f"No core three-month block forecasts forecasts found for {POLICY}")

fold_inventory = (
    required_forecasts[["fold_id", "train_start", "train_end", "test_start", "test_end"]]
    .drop_duplicates()
    .sort_values(["test_end", "fold_id"])
)
holdout_fold_id = int(fold_inventory.iloc[-1].fold_id)
development_fold_ids = fold_inventory.loc[fold_inventory.fold_id.ne(holdout_fold_id), "fold_id"].astype(int).tolist()
universe_skus = sorted(model_data.sku_id.unique())

print(f"SKU universe: {len(universe_skus):,}")
print(f"Development folds: {development_fold_ids}")
print(f"Untouched holdout fold: {holdout_fold_id}")
display(fold_inventory)

holdout_meta = fold_inventory.loc[fold_inventory.fold_id.eq(holdout_fold_id)].iloc[0]
holdout_train = model_data.loc[model_data.month.between(holdout_meta.train_start, holdout_meta.train_end)].copy()
holdout_test = model_data.loc[model_data.month.between(holdout_meta.test_start, holdout_meta.test_end)].copy()
holdout_features = sr.history_feature_table(holdout_train, universe_skus, metadata)
holdout_features["classification_cutoff"] = holdout_meta.train_end
holdout_features["product_master_assumption"] = True

holdout_features.to_csv(REVIEW_DIRECTORY / "sku_routing_sku_classification_all_690.csv", index=False)

print("Lifecycle")
display(holdout_features.lifecycle_tier.value_counts(dropna=False).to_frame("sku_count"))
print("Demand frequency")
display(holdout_features.frequency_tier.value_counts(dropna=False).to_frame("sku_count"))
print("Potential stock status - this is a screening flag, not proof of lost demand")
display(holdout_features.potential_stock_status.value_counts(dropna=False).to_frame("sku_count"))

router_training_frames = []
for position, fold_id in enumerate(development_fold_ids, start=1):
    fold_meta = fold_inventory.loc[fold_inventory.fold_id.eq(fold_id)].iloc[0]
    fold_train = model_data.loc[model_data.month.between(fold_meta.train_start, fold_meta.train_end)].copy()
    fold_forecasts = required_forecasts.loc[required_forecasts.fold_id.eq(fold_id)].copy()
    fold_features = sr.history_feature_table(fold_train, universe_skus, metadata)
    fold_targets = sr.best_model_targets(fold_forecasts)
    training = fold_features.merge(
        fold_targets[["sku_id", "best_model_key", "best_model", "block_wmape_percent", "best_under_50", "best_under_70"]],
        on="sku_id",
        how="inner",
    )
    training["fold_id"] = fold_id
    training["train_end"] = fold_meta.train_end
    router_training_frames.append(training)
    print(f"Router fold {position}/{len(development_fold_ids)} | rows: {len(training):,}", flush=True)

router_training = pd.concat(router_training_frames, ignore_index=True)
router_training.to_csv(OUTPUT_DIRECTORY / "sku_routing_router_training_examples.csv", index=False)
print("Router training examples:", len(router_training))
display(router_training.best_model.value_counts().to_frame("training_examples"))

router_models = sr.fit_router(router_training, RANDOM_STATE)
established_features = holdout_features.loc[holdout_features.lifecycle_tier.ne("cold_start")].copy()
router_predictions = sr.predict_router(router_models, established_features)
router_predictions = router_predictions.merge(
    established_features[["sku_id", "lifecycle_tier", "frequency_tier", "recency_tier"]],
    on="sku_id",
    how="left",
)

development_global_rows = []
development_forecasts = required_forecasts.loc[required_forecasts.fold_id.isin(development_fold_ids)]
for model_key, group in development_forecasts.groupby("model_key", sort=False):
    scores = sr.score_sku_models(group)
    valid = scores.loc[scores.block_wmape_percent.notna()]
    development_global_rows.append({
        "model_key": model_key,
        "model": group.model.iloc[0],
        "under_50_skus": int(valid.block_wmape_percent.lt(50).sum()),
        "under_70_skus": int(valid.block_wmape_percent.lt(70).sum()),
        "under_100_skus": int(valid.block_wmape_percent.lt(100).sum()),
        "median_block_wmape": valid.block_wmape_percent.median(),
    })
development_global = pd.DataFrame(development_global_rows).sort_values(
    ["under_50_skus", "under_70_skus", "under_100_skus", "median_block_wmape"],
    ascending=[False, False, False, True],
)
fallback_model_key = development_global.iloc[0].model_key

holdout_candidates = required_forecasts.loc[required_forecasts.fold_id.eq(holdout_fold_id)].copy()
router_holdout = sr.route_forecasts(holdout_candidates, router_predictions, fallback_model_key)
router_holdout["selection_source"] = "development_trained_router"
router_per_sku, router_summary = bh.score_forecast(router_holdout)

core_block_path = project_root / "results" / "lumpy_02_forecasting/stages/core_block" / "core_block_selected_holdout_block_forecasts.csv"
core_block_selected = pd.read_csv(core_block_path, parse_dates=["block_start", "test_start", "test_end"])
core_block_selected = core_block_selected.loc[core_block_selected.policy.eq(POLICY)]
core_block_per_sku, core_block_summary = bh.score_forecast(core_block_selected)

router_comparison = pd.DataFrame([
    {"selection_method": "12 development-trained router", **router_summary},
    {"selection_method": "11 per-SKU development average", **core_block_summary},
])
router_comparison["decision"] = ["rejected_challenger", "retained_champion"]
router_predictions.to_csv(REVIEW_DIRECTORY / "sku_routing_router_predictions_established_skus.csv", index=False)
router_per_sku.to_csv(REVIEW_DIRECTORY / "sku_routing_router_holdout_per_sku.csv", index=False)
router_comparison.to_csv(REVIEW_DIRECTORY / "sku_routing_router_holdout_summary.csv", index=False)
router_holdout.to_csv(OUTPUT_DIRECTORY / "sku_routing_router_holdout_forecasts.csv", index=False)

display(development_global)
display(router_comparison)

cold_development_frames = []
cold_started = time.perf_counter()
for position, fold_id in enumerate(development_fold_ids, start=1):
    fold_meta = fold_inventory.loc[fold_inventory.fold_id.eq(fold_id)].iloc[0]
    fold_train = model_data.loc[model_data.month.between(fold_meta.train_start, fold_meta.train_end)].copy()
    fold_test = model_data.loc[model_data.month.between(fold_meta.test_start, fold_meta.test_end)].copy()
    eligible_targets = sorted(set(fold_train.sku_id).intersection(fold_test.sku_id))
    candidates = sr.cold_start_candidate_forecasts(
        fold_train,
        fold_test,
        eligible_targets,
        metadata,
        pd.Timestamp(fold_meta.test_start),
        CONFIG.test_months,
    )
    candidates["fold_id"] = fold_id
    cold_development_frames.append(candidates)
    print(
        f"Pseudo-cold fold {position}/{len(development_fold_ids)} | targets: {len(eligible_targets):,} | "
        f"rows: {len(candidates):,} | elapsed: {(time.perf_counter() - cold_started) / 60:,.1f}m",
        flush=True,
    )

cold_development_forecasts = pd.concat(cold_development_frames, ignore_index=True)
cold_method_scores = sr.score_cold_methods(cold_development_forecasts)
selected_cold_model_key = cold_method_scores.iloc[0].model_key

cold_development_forecasts.to_csv(OUTPUT_DIRECTORY / "sku_routing_pseudo_cold_start_forecasts.csv", index=False)
cold_method_scores.to_csv(REVIEW_DIRECTORY / "sku_routing_cold_start_method_development_summary.csv", index=False)
print("Selected cold-start method:", selected_cold_model_key)
display(cold_method_scores)

cold_start_skus = holdout_features.loc[holdout_features.lifecycle_tier.eq("cold_start"), "sku_id"].tolist()
real_cold_candidates = sr.cold_start_candidate_forecasts(
    holdout_train,
    holdout_test,
    cold_start_skus,
    metadata,
    pd.Timestamp(holdout_meta.test_start),
    CONFIG.test_months,
)
real_cold_selected = real_cold_candidates.loc[real_cold_candidates.model_key.eq(selected_cold_model_key)].copy()
real_cold_selected["selection_source"] = "development_selected_peer_cold_start"
real_cold_selected["block_naive_scale"] = np.nan
real_cold_per_sku, real_cold_summary = bh.score_forecast(real_cold_selected)
real_cold_candidate_summary = sr.score_cold_methods(real_cold_candidates)

real_cold_candidates.to_csv(OUTPUT_DIRECTORY / "sku_routing_real_cold_start_candidate_forecasts.csv", index=False)
real_cold_selected.to_csv(OUTPUT_DIRECTORY / "sku_routing_real_cold_start_selected_forecasts.csv", index=False)
real_cold_per_sku.to_csv(REVIEW_DIRECTORY / "sku_routing_real_cold_start_holdout_per_sku.csv", index=False)
pd.DataFrame([real_cold_summary]).to_csv(REVIEW_DIRECTORY / "sku_routing_real_cold_start_holdout_summary.csv", index=False)
real_cold_candidate_summary.to_csv(REVIEW_DIRECTORY / "sku_routing_real_cold_start_candidate_summary.csv", index=False)

print(f"Real cold starts scored: {real_cold_per_sku.sku_id.nunique():,}")
display(pd.DataFrame([real_cold_summary]))
display(real_cold_candidate_summary)

# The router is a rejected challenger on the untouched holdout, so retain core three-month block forecasts for established SKUs.
established_champion = core_block_selected.copy()
established_champion["selection_source"] = "11_per_sku_development_champion"
common_columns = sorted(set(established_champion.columns).union(real_cold_selected.columns))
final_forecasts = pd.concat(
    [established_champion.reindex(columns=common_columns), real_cold_selected.reindex(columns=common_columns)],
    ignore_index=True,
    sort=False,
)
final_per_sku, final_summary = bh.score_forecast(final_forecasts)
final_per_sku = final_per_sku.merge(
    holdout_features[[
        "sku_id", "lifecycle_tier", "frequency_tier", "recency_tier", "size_tier",
        "potential_stock_status", "abc_units_tier", "abc_value_tier",
    ]],
    on="sku_id",
    how="left",
)
final_per_sku = final_per_sku.merge(
    router_predictions[["sku_id", "probability_below_50", "probability_below_70"]],
    on="sku_id",
    how="left",
)

segment_rows = []
for segment_column in ("lifecycle_tier", "frequency_tier", "recency_tier", "abc_units_tier"):
    for segment, group in final_per_sku.groupby(segment_column, dropna=False):
        valid = group.loc[group.block_wmape_percent.notna()]
        segment_rows.append({
            "segment_type": segment_column,
            "segment": segment,
            "all_skus": group.sku_id.nunique(),
            "positive_demand_skus": len(valid),
            "under_50_skus": int(valid.block_wmape_percent.lt(50).sum()),
            "under_70_skus": int(valid.block_wmape_percent.lt(70).sum()),
            "under_100_skus": int(valid.block_wmape_percent.lt(100).sum()),
            "median_block_wmape": valid.block_wmape_percent.median() if len(valid) else np.nan,
            "actual_total": group.actual_total.sum(),
            "forecast_total": group.forecast_total.sum(),
        })
segment_summary = pd.DataFrame(segment_rows)

final_forecasts.to_csv(OUTPUT_DIRECTORY / "sku_routing_final_all_690_holdout_forecasts.csv", index=False)
final_per_sku.to_csv(REVIEW_DIRECTORY / "sku_routing_final_all_690_holdout_per_sku.csv", index=False)
pd.DataFrame([final_summary]).to_csv(REVIEW_DIRECTORY / "sku_routing_final_all_690_holdout_summary.csv", index=False)
segment_summary.to_csv(REVIEW_DIRECTORY / "sku_routing_final_segment_summary.csv", index=False)

coverage_audit = pd.DataFrame([{
    "expected_skus": len(universe_skus),
    "classification_skus": holdout_features.sku_id.nunique(),
    "established_forecast_skus": established_champion.sku_id.nunique(),
    "cold_start_forecast_skus": real_cold_selected.sku_id.nunique(),
    "final_forecast_skus": final_forecasts.sku_id.nunique(),
    "complete_coverage": final_forecasts.sku_id.nunique() == len(universe_skus),
}])
coverage_audit.to_csv(REVIEW_DIRECTORY / "sku_routing_coverage_audit.csv", index=False)

display(coverage_audit)
display(pd.DataFrame([final_summary]))
display(segment_summary)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for axis, (title, frame) in zip(
    axes.flat,
    [
        ("Established core three-month block forecasts champion", established_champion),
        ("Real cold starts", real_cold_selected),
    ],
):
    top_sku = frame.groupby("sku_id").target.sum().sort_values(ascending=False).index[0]
    rows = frame.loc[frame.sku_id.eq(top_sku)].sort_values("block_start")
    axis.plot(rows.block_start, rows.target, marker="o", linewidth=2, label="Actual")
    axis.plot(rows.block_start, rows.forecast, marker="o", linewidth=2, label="Forecast")
    axis.set_title(f"{title}: {top_sku}")
    axis.grid(alpha=0.25)
    axis.legend()
for axis in axes.flat[2:]:
    axis.set_visible(False)
fig.suptitle("SKU classification and cold-start candidates untouched holdout: actual versus forecast", fontsize=14)
fig.tight_layout()
plt.show()

print("All-SKU classification:", REVIEW_DIRECTORY / "sku_routing_sku_classification_all_690.csv")
print("Router comparison:", REVIEW_DIRECTORY / "sku_routing_router_holdout_summary.csv")
print("Cold-start development methods:", REVIEW_DIRECTORY / "sku_routing_cold_start_method_development_summary.csv")
print("Real cold-start result:", REVIEW_DIRECTORY / "sku_routing_real_cold_start_holdout_summary.csv")
print("Final all-690 result:", REVIEW_DIRECTORY / "sku_routing_final_all_690_holdout_summary.csv")
print("Segment result:", REVIEW_DIRECTORY / "sku_routing_final_segment_summary.csv")


## 3. Chronology-safe baseline

Corrects the validation chronology. A result can only influence model selection if the actual demand would have been known at that decision date. This creates the defensible 690-SKU baseline.


In [ ]:
lf = lumpy_forecasting
sr = lumpy_sku_router
sp = lumpy_segment_policy

candidate_roots = [Path.cwd(), Path.cwd().parent, Path(r"D:/Lumpy_Fellas-/Inchscape Repo")]
project_root = next(root.resolve() for root in candidate_roots if (root / "src" / "lumpy_forecasting.py").exists())
src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
importlib.reload(lf)
importlib.reload(sr)
importlib.reload(sp)

OUTPUT_DIRECTORY = project_root / "results" / "lumpy_02_forecasting/stages/chronology_strategy"
REVIEW_DIRECTORY = project_root / "docs" / "lumpy_02_forecasting/stages/chronology_strategy"
for directory in (OUTPUT_DIRECTORY, REVIEW_DIRECTORY):
    directory.mkdir(parents=True, exist_ok=True)

CONFIG = lf.LumpyConfig(
    variant="all_sku_history",
    train_months=48,
    gap_months=3,
    test_months=18,
    step_months=3,
    min_train_months=18,
    max_folds=6,
    external_mode="off",
)
POLICY = "required_18m_gap3"

print("Chronology-safe baseline reuses existing forecasts; it does not retrain XGBoost or Croston models.")

sales, external = lf.load_lumpy_inputs(project_root, CONFIG)
model_data, _ = lf.build_lumpy_model_frame(sales, external, CONFIG)
metadata = sr.extract_static_metadata(sales)
universe_skus = sorted(model_data.sku_id.unique())

candidate_path = project_root / "results" / "lumpy_02_forecasting/stages/core_block" / "core_block_block_backtest_forecasts.csv"
if not candidate_path.exists():
    raise FileNotFoundError(f"Core three-month block forecasts candidates are required: {candidate_path}")
all_candidates = pd.read_csv(
    candidate_path,
    parse_dates=["block_start", "train_start", "train_end", "test_start", "test_end"],
)
required_candidates = all_candidates.loc[all_candidates.policy.eq(POLICY)].copy()
fold_inventory = (
    required_candidates[["fold_id", "train_start", "train_end", "test_start", "test_end"]]
    .drop_duplicates()
    .sort_values(["test_end", "fold_id"])
)
holdout_meta = fold_inventory.iloc[-1]
holdout_fold_id = int(holdout_meta.fold_id)
holdout_cutoff = pd.Timestamp(holdout_meta.train_end)
holdout_candidates = required_candidates.loc[required_candidates.fold_id.eq(holdout_fold_id)].copy()

holdout_train = model_data.loc[model_data.month.between(holdout_meta.train_start, holdout_cutoff)].copy()
holdout_features = sp.add_strategy_tier(sr.history_feature_table(holdout_train, universe_skus, metadata))
holdout_features["classification_cutoff"] = holdout_cutoff

print(f"SKU universe: {len(universe_skus):,}")
print(f"Final decision cutoff: {holdout_cutoff:%Y-%m}")
print(f"Final holdout: {pd.Timestamp(holdout_meta.test_start):%Y-%m} to {pd.Timestamp(holdout_meta.test_end):%Y-%m}")
display(fold_inventory)
display(holdout_features.strategy_tier.value_counts().to_frame("sku_count"))

development_rows = required_candidates.loc[required_candidates.fold_id.ne(holdout_fold_id)].copy()
development_with_last = sp.add_block_last_month(development_rows)
strict_history = sp.known_forecasts_at_cutoff(development_rows, holdout_cutoff)

chronology_audit = pd.DataFrame([{
    "holdout_cutoff": holdout_cutoff,
    "old_development_rows": len(development_rows),
    "old_development_block_starts": development_rows.block_start.nunique(),
    "rows_with_outcome_known_at_cutoff_before_dedup": int(development_with_last.block_last_month.le(holdout_cutoff).sum()),
    "strict_rows_after_calendar_block_dedup": len(strict_history),
    "strict_known_block_starts": strict_history.block_start.nunique(),
    "strict_first_known_block": strict_history.block_start.min(),
    "strict_last_known_block": strict_history.block_start.max(),
    "future_or_overlapping_rows_removed": int(development_with_last.block_last_month.gt(holdout_cutoff).sum()),
}])
chronology_audit.to_csv(REVIEW_DIRECTORY / "chronology_strategy_chronology_audit.csv", index=False)

display(chronology_audit)
print("Known block starts:", ", ".join(sorted(strict_history.block_start.dt.strftime("%Y-%m").unique())))

validation_frames = []
validation_detail = []
calendar_blocks = sorted(strict_history.block_start.unique())

for block_start in calendar_blocks:
    block_candidates = required_candidates.loc[required_candidates.block_start.eq(block_start)].copy()
    if block_candidates.empty:
        continue
    latest_origin = block_candidates.train_end.max()
    block_candidates = (
        block_candidates.loc[block_candidates.train_end.eq(latest_origin)]
        .sort_values(["sku_id", "model_key", "fold_id"])
        .drop_duplicates(["sku_id", "model_key", "block_start"], keep="last")
    )
    history = sp.known_forecasts_at_cutoff(required_candidates, latest_origin)
    if history.empty:
        continue
    origin_row = fold_inventory.loc[fold_inventory.train_end.eq(latest_origin)].iloc[-1]
    origin_train = model_data.loc[model_data.month.between(origin_row.train_start, latest_origin)].copy()
    origin_features = sp.add_strategy_tier(sr.history_feature_table(origin_train, universe_skus, metadata))

    for spec in sp.POLICIES:
        selected = sp.apply_policy(history, block_candidates, origin_features, spec)
        selected["validation_block_start"] = block_start
        selected["validation_cutoff"] = latest_origin
        validation_frames.append(selected)
        _, summary = sp.score_forecasts(selected)
        validation_detail.append({"policy": spec.name, "block_start": block_start, "cutoff": latest_origin, **summary})
    print(f"Validation block {pd.Timestamp(block_start):%Y-%m} | cutoff {latest_origin:%Y-%m} | history blocks {history.block_start.nunique()}")

if not validation_frames:
    raise ValueError("No chronology-safe validation blocks were available.")
validation_forecasts = pd.concat(validation_frames, ignore_index=True)
validation_detail = pd.DataFrame(validation_detail)

development_rows = []
for policy_name, group in validation_forecasts.groupby("segmented_policy", sort=False):
    _, summary = sp.score_forecasts(group)
    development_rows.append({"policy": policy_name, **summary})
development_summary = sp.rank_policy_summary(pd.DataFrame(development_rows))
selected_policy_name = development_summary.iloc[0].policy
selected_spec = next(spec for spec in sp.POLICIES if spec.name == selected_policy_name)

validation_detail.to_csv(REVIEW_DIRECTORY / "chronology_strategy_development_policy_by_block.csv", index=False)
development_summary.to_csv(REVIEW_DIRECTORY / "chronology_strategy_development_policy_summary.csv", index=False)
validation_forecasts.to_csv(OUTPUT_DIRECTORY / "chronology_strategy_development_policy_forecasts.csv", index=False)

print("Development-selected policy:", selected_policy_name)
display(development_summary)

established_features = holdout_features.loc[holdout_features.lifecycle_tier.ne("cold_start")].copy()
holdout_policy_frames = []
holdout_policy_rows = []
for spec in sp.POLICIES:
    selected = sp.apply_policy(strict_history, holdout_candidates, established_features, spec)
    selected["selection_source"] = "13_strict_chronology_development"
    holdout_policy_frames.append(selected)
    _, summary = sp.score_forecasts(selected)
    holdout_policy_rows.append({"policy": spec.name, **summary})

holdout_policy_forecasts = pd.concat(holdout_policy_frames, ignore_index=True)
holdout_policy_summary = sp.rank_policy_summary(pd.DataFrame(holdout_policy_rows))
established_selected = holdout_policy_forecasts.loc[
    holdout_policy_forecasts.segmented_policy.eq(selected_policy_name)
].copy()
established_per_sku, established_summary = sp.score_forecasts(established_selected)

old_established_path = project_root / "results" / "lumpy_02_forecasting/stages/core_block" / "core_block_selected_holdout_block_forecasts.csv"
old_established = pd.read_csv(old_established_path, parse_dates=["block_start", "test_start", "test_end"])
old_established = old_established.loc[old_established.policy.eq(POLICY)]
_, old_established_summary = sp.score_forecasts(old_established)

comparison = pd.DataFrame([
    {"result": "11 original overlapping development selector", "chronology_safe": False, **old_established_summary},
    {"result": f"13 selected strict policy: {selected_policy_name}", "chronology_safe": True, **established_summary},
])
holdout_policy_summary.to_csv(REVIEW_DIRECTORY / "chronology_strategy_established_holdout_policy_comparison.csv", index=False)
comparison.to_csv(REVIEW_DIRECTORY / "chronology_strategy_old_vs_strict_comparison.csv", index=False)
holdout_policy_forecasts.to_csv(OUTPUT_DIRECTORY / "chronology_strategy_established_holdout_policy_forecasts.csv", index=False)

display(comparison)
print("Holdout results for every challenger are exploratory; the selected row was fixed by development results above.")
display(holdout_policy_summary)

pseudo_path = project_root / "results" / "lumpy_02_forecasting/stages/sku_routing" / "sku_routing_pseudo_cold_start_forecasts.csv"
real_path = project_root / "results" / "lumpy_02_forecasting/stages/sku_routing" / "sku_routing_real_cold_start_candidate_forecasts.csv"
if not pseudo_path.exists() or not real_path.exists():
    raise FileNotFoundError("SKU classification and cold-start candidates pseudo-cold and real-cold candidate forecasts are required.")

pseudo = pd.read_csv(pseudo_path, parse_dates=["block_start"])
pseudo = pseudo.merge(fold_inventory[["fold_id", "train_end"]], on="fold_id", how="left")
strict_pseudo = sp.known_forecasts_at_cutoff(
    pseudo,
    holdout_cutoff,
    identity_columns=("sku_id", "model_key", "block_start"),
)

cold_development_rows = []
for model_key, group in strict_pseudo.groupby("model_key", sort=False):
    _, summary = sp.score_forecasts(group)
    cold_development_rows.append({"model_key": model_key, "model": group.model.iloc[0], **summary})
cold_development_summary = sp.rank_policy_summary(
    pd.DataFrame(cold_development_rows).rename(columns={"model_key": "policy"})
).rename(columns={"policy": "model_key"})
selected_cold_key = cold_development_summary.iloc[0].model_key

real_candidates = pd.read_csv(real_path, parse_dates=["block_start"])
real_cold_selected = real_candidates.loc[real_candidates.model_key.eq(selected_cold_key)].copy()
real_cold_selected["selection_source"] = "13_strict_chronology_cold_method"
real_cold_per_sku, real_cold_summary = sp.score_forecasts(real_cold_selected)

cold_development_summary.to_csv(REVIEW_DIRECTORY / "chronology_strategy_strict_cold_method_development.csv", index=False)
pd.DataFrame([real_cold_summary]).to_csv(REVIEW_DIRECTORY / "chronology_strategy_real_cold_holdout_summary.csv", index=False)
real_cold_per_sku.to_csv(REVIEW_DIRECTORY / "chronology_strategy_real_cold_holdout_per_sku.csv", index=False)
real_cold_selected.to_csv(OUTPUT_DIRECTORY / "chronology_strategy_real_cold_selected_forecasts.csv", index=False)

print("Strictly selected cold-start method:", selected_cold_key)
display(cold_development_summary)
display(pd.DataFrame([real_cold_summary]))

development_winner_forecasts = validation_forecasts.loc[
    validation_forecasts.segmented_policy.eq(selected_policy_name)
].copy()
established_radii = sp.fit_interval_radius(development_winner_forecasts, holdout_features, quantile=0.80)
established_with_ranges = sp.attach_intervals(
    established_selected,
    established_features,
    established_radii,
)

selected_pseudo = strict_pseudo.loc[strict_pseudo.model_key.eq(selected_cold_key)].copy()
selected_pseudo["strategy_tier"] = "inventory_policy_priority"
cold_radius = float((selected_pseudo.target - selected_pseudo.forecast).abs().quantile(0.80))
cold_radii = pd.DataFrame([{
    "strategy_tier": "inventory_policy_priority",
    "interval_radius": cold_radius,
    "residual_rows": len(selected_pseudo),
}])
cold_features = holdout_features.loc[holdout_features.lifecycle_tier.eq("cold_start")].copy()
cold_with_ranges = sp.attach_intervals(real_cold_selected, cold_features, cold_radii)

common_columns = sorted(set(established_with_ranges.columns).union(cold_with_ranges.columns))
final_forecasts = pd.concat(
    [
        established_with_ranges.reindex(columns=common_columns),
        cold_with_ranges.reindex(columns=common_columns),
    ],
    ignore_index=True,
)
final_per_sku, final_summary = sp.score_forecasts(final_forecasts)
final_per_sku = final_per_sku.merge(
    holdout_features[[
        "sku_id", "lifecycle_tier", "frequency_tier", "recency_tier", "abc_units_tier",
        "abc_value_tier", "strategy_tier", "forecast_mode",
    ]],
    on="sku_id",
    how="left",
)

segment_rows = []
for segment_column in ("strategy_tier", "lifecycle_tier", "frequency_tier", "abc_units_tier"):
    for segment, group in final_per_sku.groupby(segment_column, dropna=False):
        valid = group.loc[group.block_wmape_percent.notna()]
        segment_rows.append({
            "segment_type": segment_column,
            "segment": segment,
            "all_skus": group.sku_id.nunique(),
            "positive_demand_skus": len(valid),
            "under_50_skus": int(valid.block_wmape_percent.lt(50).sum()),
            "under_70_skus": int(valid.block_wmape_percent.lt(70).sum()),
            "under_100_skus": int(valid.block_wmape_percent.lt(100).sum()),
            "median_block_wmape": valid.block_wmape_percent.median() if len(valid) else np.nan,
            "actual_total": group.actual_total.sum(),
            "forecast_total": group.forecast_total.sum(),
        })
segment_summary = pd.DataFrame(segment_rows)

coverage = pd.DataFrame([{
    "expected_skus": len(universe_skus),
    "established_skus": established_selected.sku_id.nunique(),
    "cold_start_skus": real_cold_selected.sku_id.nunique(),
    "final_skus": final_forecasts.sku_id.nunique(),
    "complete_coverage": final_forecasts.sku_id.nunique() == len(universe_skus),
}])

final_forecasts.to_csv(OUTPUT_DIRECTORY / "chronology_strategy_final_all_690_holdout_forecasts_with_ranges.csv", index=False)
final_per_sku.to_csv(REVIEW_DIRECTORY / "chronology_strategy_final_all_690_per_sku.csv", index=False)
pd.DataFrame([final_summary]).to_csv(REVIEW_DIRECTORY / "chronology_strategy_final_all_690_summary.csv", index=False)
segment_summary.to_csv(REVIEW_DIRECTORY / "chronology_strategy_final_segment_summary.csv", index=False)
coverage.to_csv(REVIEW_DIRECTORY / "chronology_strategy_coverage_audit.csv", index=False)
holdout_features.to_csv(REVIEW_DIRECTORY / "chronology_strategy_strategy_assignment_all_690.csv", index=False)

display(coverage)
display(pd.DataFrame([final_summary]))
display(segment_summary.loc[segment_summary.segment_type.eq("strategy_tier")])

plot_rows = final_forecasts.merge(
    holdout_features[["sku_id", "strategy_tier"]], on="sku_id", how="left", suffixes=("", "_feature")
)
if "strategy_tier_feature" in plot_rows:
    plot_rows["strategy_tier"] = plot_rows.strategy_tier.fillna(plot_rows.strategy_tier_feature)

examples = []
for tier, group in plot_rows.groupby("strategy_tier", sort=False):
    top = group.groupby("sku_id").target.sum().sort_values(ascending=False).head(2).index
    examples.extend([(tier, sku) for sku in top])

fig, axes = plt.subplots(3, 2, figsize=(15, 11))
for axis, (tier, sku) in zip(axes.flat, examples):
    rows = plot_rows.loc[plot_rows.sku_id.eq(sku)].sort_values("block_start")
    axis.plot(rows.block_start, rows.target, marker="o", linewidth=2, label="Actual 3-month total")
    axis.plot(rows.block_start, rows.forecast, marker="o", linewidth=2, label="Forecast 3-month total")
    axis.fill_between(
        rows.block_start,
        rows.forecast_lower_80,
        rows.forecast_upper_80,
        alpha=0.18,
        label="Empirical 80% range",
    )
    axis.set_title(f"{tier}: {sku}")
    axis.grid(alpha=0.25)
    axis.legend()
for axis in axes.flat[len(examples):]:
    axis.set_visible(False)
fig.suptitle("Chronology-safe baseline final holdout: actual, forecast, and empirical range", fontsize=14)
fig.tight_layout()
plt.show()

print("Chronology audit:", REVIEW_DIRECTORY / "chronology_strategy_chronology_audit.csv")
print("Development policy comparison:", REVIEW_DIRECTORY / "chronology_strategy_development_policy_summary.csv")
print("Old versus strict comparison:", REVIEW_DIRECTORY / "chronology_strategy_old_vs_strict_comparison.csv")
print("Strict cold-start comparison:", REVIEW_DIRECTORY / "chronology_strategy_strict_cold_method_development.csv")
print("Final all-690 summary:", REVIEW_DIRECTORY / "chronology_strategy_final_all_690_summary.csv")
print("Final SKU-level results:", REVIEW_DIRECTORY / "chronology_strategy_final_all_690_per_sku.csv")
print("Strategy assignments:", REVIEW_DIRECTORY / "chronology_strategy_strategy_assignment_all_690.csv")


## 4. Recurring A/B model route

Focuses on established, frequently demanded A and B SKUs. It compares shared classical and machine-learning models and selects a process separately for each cohort.


In [ ]:
lf = lumpy_forecasting
bh = lumpy_block_hybrid
sr = lumpy_sku_router
et = lumpy_established_tournament

candidate_roots = [Path.cwd(), Path.cwd().parent, Path(r"D:/Lumpy_Fellas-/Inchscape Repo")]
project_root = next(root.resolve() for root in candidate_roots if (root / "src" / "lumpy_forecasting.py").exists())
src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
importlib.reload(lf)
importlib.reload(bh)
importlib.reload(sr)
importlib.reload(et)

OUTPUT_DIRECTORY = project_root / "results" / "lumpy_02_forecasting/stages/recurring_ab"
CHECKPOINT_DIRECTORY = OUTPUT_DIRECTORY / "checkpoints_v1"
REVIEW_DIRECTORY = project_root / "docs" / "lumpy_02_forecasting/stages/recurring_ab"
for directory in (OUTPUT_DIRECTORY, CHECKPOINT_DIRECTORY, REVIEW_DIRECTORY):
    directory.mkdir(parents=True, exist_ok=True)

CONFIG = lf.LumpyConfig(
    variant="all_sku_history",
    train_months=48,
    gap_months=3,
    test_months=18,
    step_months=3,
    min_train_months=18,
    max_folds=6,
    external_mode="off",
)
RUN_ROLLING_BACKTEST = True
RESUME_FROM_CHECKPOINTS = True
RANDOM_STATE = CONFIG.random_state

print("Business ranking: below 70%, then below 50%, then below 100%.")
print("External features: OFF")
print("Model families:", len(et.BASE_MODEL_KEYS))
print("Candidate variants:", len(et.variant_grid()))

sales, external = lf.load_lumpy_inputs(project_root, CONFIG)
model_data, _ = lf.build_lumpy_model_frame(sales, external, CONFIG)
metadata = sr.extract_static_metadata(sales)
universe_skus = sorted(model_data.sku_id.unique())

core_block_path = project_root / "results" / "lumpy_02_forecasting/stages/core_block" / "core_block_block_backtest_forecasts.csv"
core_block = pd.read_csv(
    core_block_path,
    parse_dates=["block_start", "train_start", "train_end", "test_start", "test_end"],
)
required11 = core_block.loc[core_block.policy.eq("required_18m_gap3")]
fold_inventory = (
    required11[["fold_id", "train_start", "train_end", "test_start", "test_end"]]
    .drop_duplicates()
    .sort_values(["test_end", "fold_id"])
)
final_meta = fold_inventory.iloc[-1]
final_cutoff = pd.Timestamp(final_meta.train_end)
final_train = model_data.loc[model_data.month.between(final_meta.train_start, final_cutoff)].copy()
cutoff_features = sr.history_feature_table(final_train, universe_skus, metadata)

focus_features = cutoff_features.loc[
    cutoff_features.lifecycle_tier.eq("established")
    & cutoff_features.frequency_tier.eq("recurring_4_6")
    & cutoff_features.abc_units_tier.isin(["A", "B"])
].copy()
focus_features["tournament_cohort"] = "recurring_" + focus_features.abc_units_tier.str.lower()
cohort_skus = {
    cohort: set(group.sku_id)
    for cohort, group in focus_features.groupby("tournament_cohort", sort=True)
}
focus_skus = set().union(*cohort_skus.values())
focus_data = model_data.loc[model_data.sku_id.isin(focus_skus)].copy()

cohort_inventory = (
    focus_features.groupby("tournament_cohort", as_index=False)
    .agg(sku_count=("sku_id", "nunique"))
)
cohort_inventory.loc[len(cohort_inventory)] = ["combined", len(focus_skus)]
cohort_inventory.to_csv(REVIEW_DIRECTORY / "recurring_ab_cohort_inventory.csv", index=False)

print(f"Final cutoff: {final_cutoff:%Y-%m}")
print(f"Focus SKUs: {len(focus_skus):,}")
display(cohort_inventory)

first_month = pd.Timestamp(focus_data.month.min())
last_known_start = final_cutoff - pd.DateOffset(months=2)
starts = []
cursor = last_known_start
while cursor >= first_month:
    train_end = cursor - pd.DateOffset(months=CONFIG.gap_months + 1)
    train_start = max(first_month, train_end - pd.DateOffset(months=CONFIG.train_months - 1))
    observed = focus_data.loc[focus_data.month.between(train_start, train_end), "month"].nunique()
    if observed >= CONFIG.min_train_months:
        starts.append((cursor, train_start, train_end, observed))
    cursor -= pd.DateOffset(months=3)

rolling_rows = []
for origin_id, (test_start, train_start, train_end, observed) in enumerate(sorted(starts), start=1):
    rolling_rows.append({
        "origin_id": origin_id,
        "train_start": train_start,
        "train_end": train_end,
        "gap_months": CONFIG.gap_months,
        "test_start": test_start,
        "test_end": test_start + pd.DateOffset(months=2),
        "train_months": observed,
        "test_months": 3,
    })
rolling_splits = pd.DataFrame(rolling_rows)
if len(rolling_splits) < 5:
    raise ValueError(f"At least five rolling origins are required; found {len(rolling_splits)}")

validation_origin_count = 2
tuning_origin_ids = rolling_splits.origin_id.iloc[:-validation_origin_count].tolist()
validation_origin_ids = rolling_splits.origin_id.iloc[-validation_origin_count:].tolist()
rolling_splits["role"] = np.where(
    rolling_splits.origin_id.isin(validation_origin_ids), "validation", "tuning"
)
rolling_splits.to_csv(REVIEW_DIRECTORY / "recurring_ab_rolling_origin_inventory.csv", index=False)

print(f"Tuning origins: {tuning_origin_ids}")
print(f"Validation origins: {validation_origin_ids}")
display(rolling_splits)

def add_origin_metadata(frame, split):
    result = frame.copy()
    for column, value in split.items():
        result[column] = value
    return result


if RUN_ROLLING_BACKTEST:
    rolling_frames = []
    errors = []
    total_jobs = len(rolling_splits) * len(bh.MODEL_SPECS)
    completed_jobs = 0
    started = time.perf_counter()

    for fold_position, split in rolling_splits.iterrows():
        origin_id = int(split.origin_id)
        fold_checkpoint = CHECKPOINT_DIRECTORY / f"origin_{origin_id:02d}__all_variants.csv"
        if RESUME_FROM_CHECKPOINTS and fold_checkpoint.exists():
            frame = pd.read_csv(
                fold_checkpoint,
                parse_dates=["block_start", "train_start", "train_end", "test_start", "test_end"],
            )
            rolling_frames.append(frame)
            completed_jobs += len(bh.MODEL_SPECS)
            print(f"Origin {origin_id}/{len(rolling_splits)} loaded | rows: {len(frame):,}", flush=True)
            continue

        train, test = lf.split_train_test(focus_data, split)
        prepared = bh.prepare_fold(
            train, test, lf.SKU_COLUMN, lf.MONTH_COLUMN, lf.TARGET_COLUMN,
            CONFIG.gap_months, 3,
        )
        fold_frames = [add_origin_metadata(et.baseline_variants(prepared), split)]
        print(
            f"Origin {origin_id}/{len(rolling_splits)} | train {split.train_start:%Y-%m} to {split.train_end:%Y-%m} "
            f"| target {split.test_start:%Y-%m} to {split.test_end:%Y-%m}",
            flush=True,
        )
        for model_position, model_key in enumerate(bh.MODEL_SPECS, start=1):
            completed_jobs += 1
            model_started = time.perf_counter()
            print(
                f"  [{completed_jobs}/{total_jobs}] model {model_position}/{len(bh.MODEL_SPECS)} {model_key} ...",
                flush=True,
            )
            try:
                variants = et.model_variants(model_key, prepared, RANDOM_STATE)
                fold_frames.append(add_origin_metadata(variants, split))
                print(f"      done in {time.perf_counter() - model_started:,.1f}s", flush=True)
            except Exception as exc:
                errors.append({"origin_id": origin_id, "model_key": model_key, "error": repr(exc)})
                print(f"      FAILED: {repr(exc)}", flush=True)

        fold_forecasts = pd.concat(fold_frames, ignore_index=True, sort=False)
        fold_forecasts.to_csv(fold_checkpoint, index=False)
        rolling_frames.append(fold_forecasts)

    rolling_forecasts = pd.concat(rolling_frames, ignore_index=True, sort=False)
    backtest_errors = pd.DataFrame(errors)
    rolling_forecasts.to_csv(OUTPUT_DIRECTORY / "recurring_ab_rolling_variant_forecasts.csv", index=False)
    backtest_errors.to_csv(OUTPUT_DIRECTORY / "recurring_ab_backtest_errors.csv", index=False)
    print(f"Rolling tournament finished in {(time.perf_counter() - started) / 60:,.1f} minutes | errors: {len(errors)}")
else:
    rolling_forecasts = pd.read_csv(
        OUTPUT_DIRECTORY / "recurring_ab_rolling_variant_forecasts.csv",
        parse_dates=["block_start", "train_start", "train_end", "test_start", "test_end"],
    )
    backtest_errors = pd.read_csv(OUTPUT_DIRECTORY / "recurring_ab_backtest_errors.csv")

tuning_forecasts = rolling_forecasts.loc[rolling_forecasts.origin_id.isin(tuning_origin_ids)].copy()
validation_forecasts = rolling_forecasts.loc[rolling_forecasts.origin_id.isin(validation_origin_ids)].copy()

tuning_summary_frames = []
tuned_model_frames = []
for cohort, sku_ids in cohort_skus.items():
    summary = et.candidate_summary(tuning_forecasts, sku_ids)
    summary["cohort"] = cohort
    tuned = et.tuned_model_variants(summary)
    tuned["cohort"] = cohort
    tuning_summary_frames.append(summary)
    tuned_model_frames.append(tuned)

tuning_summary = pd.concat(tuning_summary_frames, ignore_index=True)
tuned_models = pd.concat(tuned_model_frames, ignore_index=True)
tuning_summary.to_csv(REVIEW_DIRECTORY / "recurring_ab_tuning_all_variants.csv", index=False)
tuned_models.to_csv(REVIEW_DIRECTORY / "recurring_ab_tuned_variant_per_model.csv", index=False)

print("One tuned variant per model family and cohort")
display(tuned_models[[
    "cohort", "base_model_key", "variant_mode", "calibration_scale",
    "under_70_skus", "under_50_skus", "median_sku_block_wmape", "bias_pct",
]])

validation_summary_frames = []
champion_rows = []
validation_selected_frames = []
for cohort, sku_ids in cohort_skus.items():
    tuned = tuned_models.loc[tuned_models.cohort.eq(cohort)]
    tuned_forecasts = et.filter_tuned_variants(validation_forecasts, tuned)
    summary = et.candidate_summary(tuned_forecasts, sku_ids)
    summary["cohort"] = cohort
    summary["selected_champion"] = False
    summary.loc[summary.index[0], "selected_champion"] = True
    validation_summary_frames.append(summary)
    champion = summary.iloc[0].to_dict()
    champion_rows.append(champion)
    chosen = tuned_forecasts.loc[
        tuned_forecasts.sku_id.isin(sku_ids)
        & tuned_forecasts.base_model_key.eq(champion["base_model_key"])
        & tuned_forecasts.variant_id.eq(champion["variant_id"])
    ].copy()
    chosen["cohort"] = cohort
    validation_selected_frames.append(chosen)

validation_summary = pd.concat(validation_summary_frames, ignore_index=True)
champions = pd.DataFrame(champion_rows)
validation_selected = pd.concat(validation_selected_frames, ignore_index=True)
_, validation_combined_summary = et.score_forecast(validation_selected)

validation_summary.to_csv(REVIEW_DIRECTORY / "recurring_ab_validation_model_summary.csv", index=False)
champions.to_csv(REVIEW_DIRECTORY / "recurring_ab_selected_champion_per_cohort.csv", index=False)
pd.DataFrame([validation_combined_summary]).to_csv(
    REVIEW_DIRECTORY / "recurring_ab_validation_combined_summary.csv", index=False
)

print("Locked champions")
display(champions[[
    "cohort", "base_model_key", "variant_mode", "calibration_scale",
    "under_70_skus", "under_50_skus", "median_sku_block_wmape", "bias_pct",
]])
print("Combined validation result")
display(pd.DataFrame([validation_combined_summary]))

final_checkpoint = CHECKPOINT_DIRECTORY / "final_18m__all_variants.csv"
if RESUME_FROM_CHECKPOINTS and final_checkpoint.exists():
    final_variants = pd.read_csv(final_checkpoint, parse_dates=["block_start"])
    print("Loaded final 18-month variant checkpoint")
else:
    final_test = focus_data.loc[focus_data.month.between(final_meta.test_start, final_meta.test_end)].copy()
    final_prepared = bh.prepare_fold(
        final_train.loc[final_train.sku_id.isin(focus_skus)],
        final_test,
        lf.SKU_COLUMN, lf.MONTH_COLUMN, lf.TARGET_COLUMN,
        CONFIG.gap_months, CONFIG.test_months,
    )
    final_frames = [et.baseline_variants(final_prepared)]
    for model_position, model_key in enumerate(bh.MODEL_SPECS, start=1):
        model_started = time.perf_counter()
        print(f"Final model {model_position}/{len(bh.MODEL_SPECS)} {model_key} ...", flush=True)
        final_frames.append(et.model_variants(model_key, final_prepared, RANDOM_STATE))
        print(f"    done in {time.perf_counter() - model_started:,.1f}s", flush=True)
    final_variants = pd.concat(final_frames, ignore_index=True, sort=False)
    final_variants.to_csv(final_checkpoint, index=False)

final_selected_frames = []
final_exploratory_frames = []
for cohort, sku_ids in cohort_skus.items():
    champion = champions.loc[champions.cohort.eq(cohort)].iloc[0]
    selected = final_variants.loc[
        final_variants.sku_id.isin(sku_ids)
        & final_variants.base_model_key.eq(champion.base_model_key)
        & final_variants.variant_id.eq(champion.variant_id)
    ].copy()
    selected["cohort"] = cohort
    selected["selection_source"] = "rolling_validation_locked_champion"
    final_selected_frames.append(selected)

    tuned = tuned_models.loc[tuned_models.cohort.eq(cohort)]
    exploratory = et.filter_tuned_variants(final_variants, tuned)
    exploratory = exploratory.loc[exploratory.sku_id.isin(sku_ids)].copy()
    exploratory["cohort"] = cohort
    final_exploratory_frames.append(exploratory)

final_selected = pd.concat(final_selected_frames, ignore_index=True)
final_exploratory = pd.concat(final_exploratory_frames, ignore_index=True)
final_per_sku, final_combined_summary = et.score_forecast(final_selected)
final_per_sku = final_per_sku.merge(
    focus_features[["sku_id", "tournament_cohort", "abc_units_tier"]], on="sku_id", how="left"
)

final_cohort_rows = []
for cohort, group in final_selected.groupby("cohort", sort=True):
    _, summary = et.score_forecast(group)
    final_cohort_rows.append({"cohort": cohort, **summary})
final_cohort_rows.append({"cohort": "combined", **final_combined_summary})
final_cohort_summary = pd.DataFrame(final_cohort_rows)

exploratory_rows = []
for (cohort, model_key, variant_id), group in final_exploratory.groupby(
    ["cohort", "base_model_key", "variant_id"], sort=False
):
    _, summary = et.score_forecast(group)
    exploratory_rows.append({
        "cohort": cohort, "base_model_key": model_key, "variant_id": variant_id, **summary
    })
final_exploratory_summary = pd.DataFrame(exploratory_rows)

final_selected.to_csv(OUTPUT_DIRECTORY / "recurring_ab_final_selected_18m_forecasts.csv", index=False)
final_per_sku.to_csv(REVIEW_DIRECTORY / "recurring_ab_final_selected_per_sku.csv", index=False)
final_cohort_summary.to_csv(REVIEW_DIRECTORY / "recurring_ab_final_selected_summary.csv", index=False)
final_exploratory_summary.to_csv(REVIEW_DIRECTORY / "recurring_ab_final_exploratory_model_summary.csv", index=False)

display(final_cohort_summary)

chronology_strategy_path = project_root / "results" / "lumpy_02_forecasting/stages/chronology_strategy" / "chronology_strategy_final_all_690_holdout_forecasts_with_ranges.csv"
chronology_strategy_forecasts = pd.read_csv(chronology_strategy_path, parse_dates=["block_start"])
comparison_rows = []
for label, forecasts in [
    ("13 frequency champion", chronology_strategy_forecasts),
    ("14 locked A/B champions", final_selected),
]:
    for cohort, sku_ids in {**cohort_skus, "combined": focus_skus}.items():
        rows = forecasts.loc[forecasts.sku_id.isin(sku_ids)].copy()
        _, summary = et.score_forecast(rows)
        comparison_rows.append({"result": label, "cohort": cohort, **summary})
comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(REVIEW_DIRECTORY / "recurring_ab_chronology_strategy_comparison_same_skus.csv", index=False)
display(comparison)

# Final champion-challenger retention: accept a cohort challenger only when it
# improves the primary below-70 count, using below-50 as the first tie-breaker.
retention_rows = []
recommended_frames = []
for cohort, sku_ids in cohort_skus.items():
    incumbent = chronology_strategy_forecasts.loc[chronology_strategy_forecasts.sku_id.isin(sku_ids)].copy()
    challenger = final_selected.loc[final_selected.cohort.eq(cohort)].copy()
    _, incumbent_summary = et.score_forecast(incumbent)
    _, challenger_summary = et.score_forecast(challenger)
    incumbent_rank = (
        incumbent_summary["under_70_skus"], incumbent_summary["under_50_skus"],
        incumbent_summary["under_100_skus"], -incumbent_summary["median_sku_block_wmape"],
    )
    challenger_rank = (
        challenger_summary["under_70_skus"], challenger_summary["under_50_skus"],
        challenger_summary["under_100_skus"], -challenger_summary["median_sku_block_wmape"],
    )
    accepted = challenger_rank > incumbent_rank
    selected = challenger if accepted else incumbent
    selected["cohort"] = cohort
    selected["retained_source"] = "14_challenger" if accepted else "13_incumbent"
    recommended_frames.append(selected)
    retention_rows.append({
        "cohort": cohort,
        "decision": "accept_14_challenger" if accepted else "retain_13_incumbent",
        "incumbent_under_70": incumbent_summary["under_70_skus"],
        "challenger_under_70": challenger_summary["under_70_skus"],
        "incumbent_under_50": incumbent_summary["under_50_skus"],
        "challenger_under_50": challenger_summary["under_50_skus"],
        "incumbent_median_wmape": incumbent_summary["median_sku_block_wmape"],
        "challenger_median_wmape": challenger_summary["median_sku_block_wmape"],
    })

retention_decisions = pd.DataFrame(retention_rows)
common_columns = sorted(set().union(*(frame.columns for frame in recommended_frames)))
recommended_forecasts = pd.concat(
    [frame.reindex(columns=common_columns) for frame in recommended_frames],
    ignore_index=True,
)
recommended_per_sku, recommended_combined_summary = et.score_forecast(recommended_forecasts)
recommended_per_sku = recommended_per_sku.merge(
    focus_features[["sku_id", "tournament_cohort", "abc_units_tier"]], on="sku_id", how="left"
)
recommended_rows = []
for cohort, group in recommended_forecasts.groupby("cohort", sort=True):
    _, summary = et.score_forecast(group)
    recommended_rows.append({"cohort": cohort, **summary})
recommended_rows.append({"cohort": "combined", **recommended_combined_summary})
recommended_summary = pd.DataFrame(recommended_rows)

retention_decisions.to_csv(REVIEW_DIRECTORY / "recurring_ab_champion_retention_decisions.csv", index=False)
recommended_forecasts.to_csv(OUTPUT_DIRECTORY / "recurring_ab_recommended_retained_forecasts.csv", index=False)
recommended_per_sku.to_csv(REVIEW_DIRECTORY / "recurring_ab_recommended_retained_per_sku.csv", index=False)
recommended_summary.to_csv(REVIEW_DIRECTORY / "recurring_ab_recommended_retained_summary.csv", index=False)

print("Champion-retention decisions")
display(retention_decisions)
print("Recommended retained result")
display(recommended_summary)

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
examples = []
for cohort, group in recommended_forecasts.groupby("cohort", sort=True):
    top = group.groupby("sku_id").target.sum().sort_values(ascending=False).head(2).index
    examples.extend([(cohort, sku) for sku in top])
for axis, (cohort, sku) in zip(axes.flat, examples):
    rows = recommended_forecasts.loc[recommended_forecasts.sku_id.eq(sku)].sort_values("block_start")
    axis.plot(rows.block_start, rows.target, marker="o", linewidth=2, label="Actual 3-month total")
    axis.plot(rows.block_start, rows.forecast, marker="o", linewidth=2, label="Forecast 3-month total")
    axis.set_title(f"{cohort}: {sku}")
    axis.grid(alpha=0.25)
    axis.legend()
fig.suptitle("Recurring A/B model route recommended retained champions: actual versus forecast", fontsize=14)
fig.tight_layout()
plt.show()

aggregate = recommended_forecasts.groupby(["cohort", "block_start"], as_index=False).agg(
    actual=("target", "sum"), forecast=("forecast", "sum")
)
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for axis, (cohort, group) in zip(axes, aggregate.groupby("cohort", sort=True)):
    axis.plot(group.block_start, group.actual, marker="o", linewidth=2, label="Actual")
    axis.plot(group.block_start, group.forecast, marker="o", linewidth=2, label="Forecast")
    axis.set_title(f"{cohort} aggregate")
    axis.grid(alpha=0.25)
    axis.legend()
fig.tight_layout()
plt.show()

print("Cohort inventory:", REVIEW_DIRECTORY / "recurring_ab_cohort_inventory.csv")
print("Rolling origins:", REVIEW_DIRECTORY / "recurring_ab_rolling_origin_inventory.csv")
print("Tuned variant per model:", REVIEW_DIRECTORY / "recurring_ab_tuned_variant_per_model.csv")
print("Validation comparison:", REVIEW_DIRECTORY / "recurring_ab_validation_model_summary.csv")
print("Locked champions:", REVIEW_DIRECTORY / "recurring_ab_selected_champion_per_cohort.csv")
print("Final A/B result:", REVIEW_DIRECTORY / "recurring_ab_final_selected_summary.csv")
print("Retention decisions:", REVIEW_DIRECTORY / "recurring_ab_champion_retention_decisions.csv")
print("Recommended retained result:", REVIEW_DIRECTORY / "recurring_ab_recommended_retained_summary.csv")
print("Final per-SKU result:", REVIEW_DIRECTORY / "recurring_ab_final_selected_per_sku.csv")
print("Chronology-safe baseline comparison:", REVIEW_DIRECTORY / "recurring_ab_chronology_strategy_comparison_same_skus.csv")


## 5. Recurring A/B candidate optimisation

Expands the recurring search across history lengths, XGBoost structures, Croston settings, calibration factors, subgroup routes and ensembles.


In [ ]:
lf = lumpy_forecasting
bh = lumpy_block_hybrid
sr = lumpy_sku_router
opt = lumpy_ab_optimization

roots=[Path.cwd(),Path.cwd().parent,Path(r"D:/Lumpy_Fellas-/Inchscape Repo")]
project_root=next(r.resolve() for r in roots if (r/"src"/"lumpy_forecasting.py").exists())
sys.path.insert(0,str(project_root/"src")) if str(project_root/"src") not in sys.path else None
for module in (lf,bh,sr,opt): importlib.reload(module)

OUT=project_root/"results"/"lumpy_02_forecasting/stages/recurring_optimisation"
CKPT=OUT/"checkpoints_v1"
DOC=project_root/"docs"/"lumpy_02_forecasting/stages/recurring_optimisation"
for p in (OUT,CKPT,DOC): p.mkdir(parents=True,exist_ok=True)
CONFIG=lf.LumpyConfig(variant="all_sku_history",train_months=48,gap_months=3,test_months=18,step_months=3,min_train_months=18,max_folds=6,external_mode="off")
RUN_COMPONENTS=True
RESUME=True
print("Structural/window trials:",len(opt.structural_trial_table()))
print("External features: OFF | final gap: 3 months | final horizon: 18 months")

sales,external=lf.load_lumpy_inputs(project_root,CONFIG)
model_data,_=lf.build_lumpy_model_frame(sales,external,CONFIG)
metadata=sr.extract_static_metadata(sales)
n11=pd.read_csv(project_root/"results/lumpy_02_forecasting/stages/core_block/core_block_block_backtest_forecasts.csv",parse_dates=["train_start","train_end","test_start","test_end"])
folds=n11.loc[n11.policy.eq("required_18m_gap3"),["fold_id","train_start","train_end","test_start","test_end"]].drop_duplicates().sort_values("test_end")
final_meta=folds.iloc[-1]
final_cutoff=pd.Timestamp(final_meta.train_end)
final_train=model_data.loc[model_data.month.between(final_meta.train_start,final_cutoff)]
features=sr.history_feature_table(final_train,sorted(model_data.sku_id.unique()),metadata)
focus=features.loc[features.lifecycle_tier.eq("established")&features.frequency_tier.eq("recurring_4_6")&features.abc_units_tier.isin(["A","B"])].copy()
focus["tournament_cohort"]="recurring_"+focus.abc_units_tier.str.lower()
focus=opt.add_optimization_segment(focus)
cohorts={name:set(g.sku_id) for name,g in focus.groupby("tournament_cohort")}
focus_ids=set(focus.sku_id); data=model_data.loc[model_data.sku_id.isin(focus_ids)].copy()

splits=pd.read_csv(project_root/"docs/lumpy_02_forecasting/stages/recurring_ab/recurring_ab_rolling_origin_inventory.csv",parse_dates=["train_start","train_end","test_start","test_end"])
tune_ids=splits.loc[splits.role.eq("tuning"),"origin_id"].tolist(); valid_ids=splits.loc[splits.role.eq("validation"),"origin_id"].tolist()
focus[["sku_id","tournament_cohort","optimization_segment"]].to_csv(DOC/"recurring_optimisation_focus_segments.csv",index=False)
display(focus.groupby(["tournament_cohort","optimization_segment"]).sku_id.nunique().to_frame("skus"))
display(splits)

trial_table=opt.structural_trial_table()
spec_lookup={spec.key:spec for spec in opt.STRUCTURAL_SPECS}
structural_frames=[]; classical_frames=[]; errors=[]
total=len(splits)*len(trial_table); job=0; started=time.perf_counter()
for _,split in splits.iterrows():
    oid=int(split.origin_id); spath=CKPT/f"origin_{oid:02d}__structural.pkl"; cpath=CKPT/f"origin_{oid:02d}__classical.pkl"
    if RESUME and spath.exists() and cpath.exists():
        structural_frames.append(pd.read_pickle(spath)); classical_frames.append(pd.read_pickle(cpath)); job+=len(trial_table)
        print(f"Origin {oid}/{len(splits)} loaded from checkpoints",flush=True); continue
    test=data.loc[data.month.between(split.test_start,split.test_end)].copy(); sf=[]; cf=[]
    print(f"Origin {oid}/{len(splits)} target {split.test_start:%Y-%m}",flush=True)
    for window in opt.WINDOW_OPTIONS:
        window_start=max(pd.Timestamp(data.month.min()),pd.Timestamp(split.train_end)-pd.DateOffset(months=window-1))
        train=data.loc[data.month.between(window_start,split.train_end)].copy()
        try:
            prepared=bh.prepare_fold(train,test,lf.SKU_COLUMN,lf.MONTH_COLUMN,lf.TARGET_COLUMN,3,3)
        except Exception as exc:
            errors.append({"origin_id":oid,"window":window,"trial_id":"prepare","error":repr(exc)}); continue
        classical=opt.classical_component_rows(train,prepared,window)
        classical["origin_id"]=oid; classical["history_window"]=window; cf.append(classical)
        window_trials=trial_table.loc[trial_table.history_window.eq(window)]
        for _,trial in window_trials.iterrows():
            job+=1; t0=time.perf_counter()
            try:
                frame=opt.fit_structural_components(prepared,spec_lookup[trial.key],trial.trial_id,CONFIG.random_state)
                frame["origin_id"]=oid; frame["history_window"]=window; sf.append(frame)
            except Exception as exc:
                errors.append({"origin_id":oid,"window":window,"trial_id":trial.trial_id,"error":repr(exc)})
            if job%10==0: print(f"  [{job}/{total}] elapsed {(time.perf_counter()-started)/60:.1f}m",flush=True)
    origin_s=pd.concat(sf,ignore_index=True); origin_c=pd.concat(cf,ignore_index=True)
    origin_s.to_pickle(spath); origin_c.to_pickle(cpath); structural_frames.append(origin_s); classical_frames.append(origin_c)
structural=pd.concat(structural_frames,ignore_index=True); classical=pd.concat(classical_frames,ignore_index=True)
pd.DataFrame(errors).to_csv(OUT/"recurring_optimisation_component_errors.csv",index=False)
print(f"Components complete in {(time.perf_counter()-started)/60:.1f}m | errors {len(errors)} | structural rows {len(structural):,}")

def score_candidate_variants(structural_part,classical_part,sku_ids):
    rows=[]
    for trial_id,g in structural_part.groupby("trial_id",sort=False):
        arch=g.architecture.iloc[0]
        for recipe in opt.recipe_grid(arch):
            f=opt.compose_components(g,recipe); f=f.loc[f.sku_id.isin(sku_ids)]
            _,s=opt.score_forecast(f); rows.append({"trial_id":trial_id,"recipe":opt.recipe_name(recipe),"candidate_id":f.candidate_id.iloc[0],"architecture":arch,**s})
    for trial_id,g in classical_part.groupby("trial_id",sort=False):
        for scale in opt.CALIBRATION_SCALES:
            f=opt.compose_classical(g,scale); f=f.loc[f.sku_id.isin(sku_ids)]
            _,s=opt.score_forecast(f); rows.append({"trial_id":trial_id,"recipe":f.recipe.iloc[0],"candidate_id":f.candidate_id.iloc[0],"architecture":"classical",**s})
    return opt.rank_summary(pd.DataFrame(rows))

tuning_s=structural.loc[structural.origin_id.isin(tune_ids)]; tuning_c=classical.loc[classical.origin_id.isin(tune_ids)]
validation_s=structural.loc[structural.origin_id.isin(valid_ids)]; validation_c=classical.loc[classical.origin_id.isin(valid_ids)]
tuning_tables=[]; tuned_tables=[]
for cohort,ids in cohorts.items():
    t0=time.perf_counter(); table=score_candidate_variants(tuning_s,tuning_c,ids); table["cohort"]=cohort
    tuned=table.sort_values(["trial_id","under_70","under_50","under_100","median_wmape"],ascending=[True,False,False,False,True]).groupby("trial_id",as_index=False).head(1); tuned["cohort"]=cohort
    tuning_tables.append(table); tuned_tables.append(tuned); print(f"{cohort} recipe tuning {time.perf_counter()-t0:.1f}s",flush=True)
tuning_summary=pd.concat(tuning_tables,ignore_index=True); tuned_candidates=pd.concat(tuned_tables,ignore_index=True)
tuning_summary.to_csv(DOC/"recurring_optimisation_tuning_all_candidates.csv",index=False); tuned_candidates.to_csv(DOC/"recurring_optimisation_tuned_recipe_per_trial.csv",index=False)
display(tuned_candidates.groupby(["cohort","architecture"]).head(3)[["cohort","architecture","candidate_id","under_70","under_50","median_wmape","bias_pct"]])

def materialize(candidate_ids,s_part,c_part):
    frames=[]
    for cid in candidate_ids:
        f=opt.candidate_forecast(s_part,c_part,cid).copy(); f["source_candidate_id"]=cid; frames.append(f)
    return pd.concat(frames,ignore_index=True)

validation_tables=[]; shortlists={}; validation_materialized={}
for cohort,ids in cohorts.items():
    candidate_ids=tuned_candidates.loc[tuned_candidates.cohort.eq(cohort),"candidate_id"].tolist()
    forecasts=materialize(candidate_ids,validation_s,validation_c); forecasts=forecasts.loc[forecasts.sku_id.isin(ids)]
    rows=[]
    for cid,g in forecasts.groupby("source_candidate_id",sort=False):
        _,s=opt.score_forecast(g); rows.append({"candidate_id":cid,**s})
    table=opt.rank_summary(pd.DataFrame(rows)); table["cohort"]=cohort; validation_tables.append(table)
    shortlists[cohort]=table.head(8).candidate_id.tolist(); validation_materialized[cohort]=forecasts
validation_summary=pd.concat(validation_tables,ignore_index=True); validation_summary.to_csv(DOC/"recurring_optimisation_validation_tuned_trials.csv",index=False)
display(validation_summary.groupby("cohort").head(10)[["cohort","candidate_id","under_70","under_50","under_100","median_wmape","bias_pct"]])

strategy_validation={}; strategy_rows=[]; subgroup_choices=[]; ensemble_choices=[]
for cohort,ids in cohorts.items():
    forecasts=validation_materialized[cohort]; shortlist=shortlists[cohort]
    global_id=shortlist[0]; global_f=forecasts.loc[forecasts.source_candidate_id.eq(global_id)].copy(); global_f["strategy"]="global"
    strategy_validation[(cohort,"global")]=global_f
    _,s=opt.score_forecast(global_f); strategy_rows.append({"cohort":cohort,"strategy":"global","detail":global_id,**s})

    routed=[]
    cohort_features=focus.loc[focus.tournament_cohort.eq(cohort)]
    for segment,segment_features in cohort_features.groupby("optimization_segment"):
        segment_ids=set(segment_features.sku_id); candidates=[]
        for cid in shortlist:
            g=forecasts.loc[forecasts.source_candidate_id.eq(cid)&forecasts.sku_id.isin(segment_ids)]
            _,ss=opt.score_forecast(g); candidates.append({"candidate_id":cid,**ss})
        best=opt.rank_summary(pd.DataFrame(candidates)).iloc[0]
        subgroup_choices.append({"cohort":cohort,"optimization_segment":segment,"candidate_id":best.candidate_id,"segment_skus":len(segment_ids)})
        routed.append(forecasts.loc[forecasts.source_candidate_id.eq(best.candidate_id)&forecasts.sku_id.isin(segment_ids)])
    routed=pd.concat(routed,ignore_index=True); routed["strategy"]="subgroup_router"; strategy_validation[(cohort,"subgroup_router")]=routed
    _,s=opt.score_forecast(routed); strategy_rows.append({"cohort":cohort,"strategy":"subgroup_router","detail":"size_volatility","underlying":len(shortlist),**s})

    ensemble_results=[]; top=shortlist[:4]
    for left,right in itertools.combinations(top,2):
        l=forecasts.loc[forecasts.source_candidate_id.eq(left)].set_index(["sku_id","block_start","origin_id"])
        r=forecasts.loc[forecasts.source_candidate_id.eq(right)].set_index(["sku_id","block_start","origin_id"])
        for weight in (0.25,0.50,0.75):
            e=l.reset_index().copy(); e["forecast"]=weight*l.forecast.to_numpy()+(1-weight)*r.forecast.to_numpy(); e["strategy"]="ensemble"
            _,ss=opt.score_forecast(e); ensemble_results.append({"left":left,"right":right,"left_weight":weight,"forecast_frame":e,**ss})
    erank=opt.rank_summary(pd.DataFrame([{k:v for k,v in row.items() if k!="forecast_frame"}|{"candidate_id":f"blend_{i}"} for i,row in enumerate(ensemble_results)]))
    best_index=int(erank.iloc[0].candidate_id.split("_")[1]); best=ensemble_results[best_index]; ensemble_choices.append({k:v for k,v in best.items() if k!="forecast_frame"}|{"cohort":cohort})
    ef=best["forecast_frame"]; strategy_validation[(cohort,"ensemble")]=ef
    _,s=opt.score_forecast(ef); strategy_rows.append({"cohort":cohort,"strategy":"ensemble","detail":f"{best['left_weight']:.2f} {best['left']} + {1-best['left_weight']:.2f} {best['right']}",**s})

pd.DataFrame(subgroup_choices).to_csv(DOC/"recurring_optimisation_subgroup_choices.csv",index=False)
pd.DataFrame(ensemble_choices).to_csv(DOC/"recurring_optimisation_ensemble_choices.csv",index=False)

def wide_candidate_frame(candidate_ids,s_part,c_part,ids):
    base=None; names=[]
    for position,cid in enumerate(candidate_ids):
        f=opt.candidate_forecast(s_part,c_part,cid); f=f.loc[f.sku_id.isin(ids)]
        keys=["sku_id","block_start","block_number","target","block_naive_scale","origin_id"]
        name=f"candidate_{position}"; names.append(name)
        piece=f[keys+["forecast"]].rename(columns={"forecast":name})
        if base is None: base=piece
        else: base=base.merge(piece[["sku_id","block_start","origin_id",name]],on=["sku_id","block_start","origin_id"],how="inner")
    base["month_sin"]=np.sin(2*np.pi*(pd.to_datetime(base.block_start).dt.month-1)/12)
    base["month_cos"]=np.cos(2*np.pi*(pd.to_datetime(base.block_start).dt.month-1)/12)
    return base,names

correction_specs={
 "ridge_01":Ridge(alpha=.1),"ridge_1":Ridge(alpha=1.0),"ridge_10":Ridge(alpha=10.0),
 "rf_d4_l5":RandomForestRegressor(n_estimators=400,max_depth=4,min_samples_leaf=5,n_jobs=-1,random_state=42),
 "rf_d8_l8":RandomForestRegressor(n_estimators=400,max_depth=8,min_samples_leaf=8,n_jobs=-1,random_state=42),
 "hist_poisson":HistGradientBoostingRegressor(loss="poisson",max_iter=250,max_leaf_nodes=15,l2_regularization=1.0,random_state=42),
 "xgb_tweedie":XGBRegressor(objective="reg:tweedie",tweedie_variance_power=1.35,n_estimators=400,learning_rate=.03,max_depth=2,min_child_weight=8,reg_lambda=10,n_jobs=-1,random_state=42),
}
correction_choices=[]; correction_training={}
for cohort,ids in cohorts.items():
    candidate_ids=shortlists[cohort][:5]
    train_wide,names=wide_candidate_frame(candidate_ids,tuning_s,tuning_c,ids)
    valid_wide,_=wide_candidate_frame(candidate_ids,validation_s,validation_c,ids)
    xcols=names+["month_sin","month_cos"]
    rows=[]; frames={}
    for name,model in correction_specs.items():
        fitted=clone(model); fitted.fit(train_wide[xcols],train_wide.target)
        f=valid_wide[["sku_id","block_start","block_number","target","block_naive_scale","origin_id"]].copy()
        f["forecast"]=np.maximum(0,fitted.predict(valid_wide[xcols])); f["strategy"]="correction_regression"; frames[name]=f
        _,s=opt.score_forecast(f); rows.append({"candidate_id":name,**s})
    ranked=opt.rank_summary(pd.DataFrame(rows)); winner=ranked.iloc[0].candidate_id
    correction_choices.append({"cohort":cohort,"correction_model":winner,"input_candidates":" | ".join(candidate_ids),**ranked.iloc[0].to_dict()})
    strategy_validation[(cohort,"correction_regression")]=frames[winner]
    _,s=opt.score_forecast(frames[winner]); strategy_rows.append({"cohort":cohort,"strategy":"correction_regression","detail":winner,**s})
    correction_training[cohort]=(candidate_ids,xcols,winner)
correction_choices=pd.DataFrame(correction_choices); correction_choices.to_csv(DOC/"recurring_optimisation_correction_choices.csv",index=False)
strategy_summary=opt.rank_summary(pd.DataFrame(strategy_rows).assign(candidate_id=lambda x:x.cohort+"__"+x.strategy))
strategy_summary.to_csv(DOC/"recurring_optimisation_validation_strategy_summary.csv",index=False)
display(strategy_summary[["cohort","strategy","detail","under_70","under_50","under_100","median_wmape","bias_pct"]])

locked=strategy_summary.sort_values(["cohort","under_70","under_50","under_100","median_wmape"],ascending=[True,False,False,False,True]).groupby("cohort",as_index=False).head(1)
locked.to_csv(DOC/"recurring_optimisation_locked_strategy_per_cohort.csv",index=False); display(locked[["cohort","strategy","detail","under_70","under_50","median_wmape"]])

fspath=CKPT/"final_structural.pkl"; fcpath=CKPT/"final_classical.pkl"
if RESUME and fspath.exists() and fcpath.exists(): final_s=pd.read_pickle(fspath); final_c=pd.read_pickle(fcpath)
else:
    final_test=data.loc[data.month.between(final_meta.test_start,final_meta.test_end)].copy(); sf=[];cf=[]
    for window in opt.WINDOW_OPTIONS:
        wstart=max(pd.Timestamp(data.month.min()),final_cutoff-pd.DateOffset(months=window-1)); train=data.loc[data.month.between(wstart,final_cutoff)]
        prepared=bh.prepare_fold(train,final_test,lf.SKU_COLUMN,lf.MONTH_COLUMN,lf.TARGET_COLUMN,3,18)
        c=opt.classical_component_rows(train,prepared,window); c["origin_id"]=99;cf.append(c)
        for _,trial in trial_table.loc[trial_table.history_window.eq(window)].iterrows():
            f=opt.fit_structural_components(prepared,spec_lookup[trial.key],trial.trial_id,CONFIG.random_state);f["origin_id"]=99;sf.append(f)
        print(f"Final window {window} complete",flush=True)
    final_s=pd.concat(sf,ignore_index=True);final_c=pd.concat(cf,ignore_index=True);final_s.to_pickle(fspath);final_c.to_pickle(fcpath)

final_strategy_frames=[]
subchoice=pd.DataFrame(subgroup_choices); enchoice=pd.DataFrame(ensemble_choices)
for cohort,ids in cohorts.items():
    strategy=locked.loc[locked.cohort.eq(cohort)].iloc[0].strategy
    if strategy=="global":
        cid=shortlists[cohort][0]; f=opt.candidate_forecast(final_s,final_c,cid); f=f.loc[f.sku_id.isin(ids)]
    elif strategy=="subgroup_router":
        parts=[]
        for _,choice in subchoice.loc[subchoice.cohort.eq(cohort)].iterrows():
            segment_ids=set(focus.loc[focus.optimization_segment.eq(choice.optimization_segment),"sku_id"])
            p=opt.candidate_forecast(final_s,final_c,choice.candidate_id);parts.append(p.loc[p.sku_id.isin(segment_ids)])
        f=pd.concat(parts,ignore_index=True)
    elif strategy=="ensemble":
        choice=enchoice.loc[enchoice.cohort.eq(cohort)].iloc[0]; left=opt.candidate_forecast(final_s,final_c,choice.left); right=opt.candidate_forecast(final_s,final_c,choice.right)
        left=left.loc[left.sku_id.isin(ids)].sort_values(["sku_id","block_start"]);right=right.loc[right.sku_id.isin(ids)].sort_values(["sku_id","block_start"])
        f=left.copy();f["forecast"]=choice.left_weight*left.forecast.to_numpy()+(1-choice.left_weight)*right.forecast.to_numpy()
    else:
        candidate_ids,xcols,winner=correction_training[cohort]
        all_wide,names=wide_candidate_frame(candidate_ids,structural,classical,ids)
        final_wide,_=wide_candidate_frame(candidate_ids,final_s,final_c,ids)
        model=clone(correction_specs[winner]);model.fit(all_wide[xcols],all_wide.target)
        f=final_wide[["sku_id","block_start","block_number","target","block_naive_scale","origin_id"]].copy();f["forecast"]=np.maximum(0,model.predict(final_wide[xcols]))
    f["cohort"]=cohort;f["locked_strategy"]=strategy;final_strategy_frames.append(f)
final_challenger=pd.concat(final_strategy_frames,ignore_index=True)

n14=pd.read_csv(project_root/"results/lumpy_02_forecasting/stages/recurring_ab/recurring_ab_recommended_retained_forecasts.csv",parse_dates=["block_start"])
retained=[];decision_rows=[]
for cohort,ids in cohorts.items():
    incumbent=n14.loc[n14.sku_id.isin(ids)]; challenger=final_challenger.loc[final_challenger.cohort.eq(cohort)]
    _,si=opt.score_forecast(incumbent);_,sc=opt.score_forecast(challenger)
    accept=(sc["under_70"],sc["under_50"],sc["under_100"],-sc["median_wmape"])>(si["under_70"],si["under_50"],si["under_100"],-si["median_wmape"])
    chosen=challenger if accept else incumbent; chosen=chosen.copy();chosen["cohort"]=cohort;chosen["retained_source"]="15_challenger" if accept else "14_incumbent";retained.append(chosen)
    decision_rows.append({"cohort":cohort,"decision":"accept_15" if accept else "retain_14","incumbent_under_70":si["under_70"],"challenger_under_70":sc["under_70"],"incumbent_under_50":si["under_50"],"challenger_under_50":sc["under_50"],"incumbent_median":si["median_wmape"],"challenger_median":sc["median_wmape"]})
recommended=pd.concat(retained,ignore_index=True); per,combined=opt.score_forecast(recommended)
rows=[]
for cohort,g in recommended.groupby("cohort"):
    _,s=opt.score_forecast(g);rows.append({"cohort":cohort,**s})
rows.append({"cohort":"combined",**combined});recommended_summary=pd.DataFrame(rows)
pd.DataFrame(decision_rows).to_csv(DOC/"recurring_optimisation_retention_decisions.csv",index=False);recommended_summary.to_csv(DOC/"recurring_optimisation_recommended_summary.csv",index=False);per.to_csv(DOC/"recurring_optimisation_recommended_per_sku.csv",index=False);recommended.to_csv(OUT/"recurring_optimisation_recommended_forecasts.csv",index=False)
display(pd.DataFrame(decision_rows));display(recommended_summary)

fig,axes=plt.subplots(2,2,figsize=(15,9));examples=[]
for cohort,g in recommended.groupby("cohort"):
    examples.extend([(cohort,sku) for sku in g.groupby("sku_id").target.sum().nlargest(2).index])
for ax,(cohort,sku) in zip(axes.flat,examples):
    g=recommended.loc[recommended.sku_id.eq(sku)].sort_values("block_start");ax.plot(g.block_start,g.target,marker="o",label="Actual");ax.plot(g.block_start,g.forecast,marker="o",label="Forecast");ax.set_title(f"{cohort}: {sku}");ax.grid(alpha=.25);ax.legend()
fig.suptitle("Recurring A/B candidate optimisation retained A/B forecasts: actual versus forecast");fig.tight_layout();plt.show()

for label,path in {
"Tuning candidates":DOC/"recurring_optimisation_tuning_all_candidates.csv","Validation trials":DOC/"recurring_optimisation_validation_tuned_trials.csv","Subgroup choices":DOC/"recurring_optimisation_subgroup_choices.csv","Ensembles":DOC/"recurring_optimisation_ensemble_choices.csv","Correction models":DOC/"recurring_optimisation_correction_choices.csv","Locked strategies":DOC/"recurring_optimisation_locked_strategy_per_cohort.csv","Retention":DOC/"recurring_optimisation_retention_decisions.csv","Recommended result":DOC/"recurring_optimisation_recommended_summary.csv"}.items(): print(label,path)


## 6. Recurring layered ensemble

Combines the strongest recurring candidates. It tests routing, stacking and correction layers, accepting a challenger only when it improves the SKU-level thresholds.


In [ ]:
opt = lumpy_ab_optimization
layer = lumpy_layered_ensemble

roots=[Path.cwd(),Path.cwd().parent,Path(r"D:/Lumpy_Fellas-/Inchscape Repo")]
project_root=next(r.resolve() for r in roots if (r/"src"/"lumpy_forecasting.py").exists())
sys.path.insert(0,str(project_root/"src")) if str(project_root/"src") not in sys.path else None
importlib.reload(opt); importlib.reload(layer)

OUT=project_root/"results"/"lumpy_02_forecasting/stages/recurring_ensemble"
DOC=project_root/"docs"/"lumpy_02_forecasting/stages/recurring_ensemble"
CKPT=project_root/"results"/"lumpy_02_forecasting/stages/recurring_optimisation"/"checkpoints_v1"
for path in (OUT,DOC): path.mkdir(parents=True,exist_ok=True)
RANDOM_STATE=42
print("Recurring layered ensemble reuses Recurring A/B candidate optimisation checkpoints; it does not refit the 455 base structures.")

focus=pd.read_csv(project_root/"docs/lumpy_02_forecasting/stages/recurring_optimisation/recurring_optimisation_focus_segments.csv")
cohorts={name:set(group.sku_id) for name,group in focus.groupby("tournament_cohort")}
splits=pd.read_csv(project_root/"docs/lumpy_02_forecasting/stages/recurring_ab/recurring_ab_rolling_origin_inventory.csv",parse_dates=["train_start","train_end","test_start","test_end"])
tune_ids=splits.loc[splits.role.eq("tuning"),"origin_id"].astype(int).tolist()
valid_ids=splits.loc[splits.role.eq("validation"),"origin_id"].astype(int).tolist()
tuning=pd.read_csv(project_root/"docs/lumpy_02_forecasting/stages/recurring_optimisation/recurring_optimisation_tuning_all_candidates.csv")

structural=pd.concat([pd.read_pickle(CKPT/f"origin_{oid:02d}__structural.pkl") for oid in splits.origin_id.astype(int)],ignore_index=True)
classical=pd.concat([pd.read_pickle(CKPT/f"origin_{oid:02d}__classical.pkl") for oid in splits.origin_id.astype(int)],ignore_index=True)
final_structural=pd.read_pickle(CKPT/"final_structural.pkl")
final_classical=pd.read_pickle(CKPT/"final_classical.pkl")

incumbent_candidates={
    "recurring_a":"direct_d2_base_p135__w48__direct__p1.00__t0.00__s0.75__none__b0.00",
    "recurring_b":"hurdle_d2_sqrt_log__w48__soft__p0.75__t0.00__s1.00__none__b0.00",
}
candidate_ids={cohort:layer.select_diverse_candidates(tuning,cohort,[incumbent_candidates[cohort]],per_architecture=4) for cohort in cohorts}
for cohort,values in candidate_ids.items(): print(cohort,"experts:",len(values))

wide={}; catalogues=[]
for cohort,ids in cohorts.items():
    pool=candidate_ids[cohort]
    all_frame,columns,catalogue=layer.build_wide_candidate_frame(pool,structural,classical,ids)
    final_frame,final_columns,_=layer.build_wide_candidate_frame(pool,final_structural,final_classical,ids)
    assert columns==final_columns
    wide[cohort]={
        "development":all_frame.loc[all_frame.origin_id.isin(tune_ids)].copy(),
        "validation":all_frame.loc[all_frame.origin_id.isin(valid_ids)].copy(),
        "all_known":all_frame.copy(),"final":final_frame.copy(),"columns":columns,
    }
    catalogue["cohort"]=cohort; catalogues.append(catalogue)
catalogue=pd.concat(catalogues,ignore_index=True)
catalogue.to_csv(DOC/"recurring_ensemble_expert_catalogue.csv",index=False)
display(catalogue)

oof_predictions={}; oof_tables=[]
for cohort,parts in wide.items():
    print("Developing layered strategies for",cohort,flush=True)
    cohort_predictions={}
    for number,strategy in enumerate(layer.DEFAULT_STRATEGIES,1):
        print(f"  [{number}/{len(layer.DEFAULT_STRATEGIES)}] {strategy}",flush=True)
        cohort_predictions[strategy]=layer.leave_one_origin_out(parts["development"],parts["columns"],strategy,tune_ids,RANDOM_STATE)
    oof_predictions[cohort]=cohort_predictions
    table=layer.score_strategies(cohort_predictions); table["cohort"]=cohort; oof_tables.append(table)
oof_summary=pd.concat(oof_tables,ignore_index=True)
oof_summary.to_csv(DOC/"recurring_ensemble_agent_development_oof.csv",index=False)
display(oof_summary.groupby("cohort").head(12)[["cohort","strategy","under_70","under_50","under_100","median_wmape","portfolio_wmape","bias_pct"]])

validation_predictions={}; validation_tables=[]
for cohort,parts in wide.items():
    cohort_predictions={}
    for strategy in layer.DEFAULT_STRATEGIES:
        predicted=parts["validation"].copy()
        predicted["forecast"]=layer.fit_predict_strategy(strategy,parts["development"],predicted,parts["columns"],RANDOM_STATE).forecast
        predicted["strategy"]=strategy
        cohort_predictions[strategy]=predicted
    validation_predictions[cohort]=cohort_predictions
    table=layer.score_strategies(cohort_predictions); table["cohort"]=cohort; validation_tables.append(table)
validation_summary=pd.concat(validation_tables,ignore_index=True)

audit=oof_summary[["cohort","strategy","positive_skus","under_70","under_50","median_wmape"]].merge(
    validation_summary[["cohort","strategy","positive_skus","under_70","under_50","median_wmape"]],on=["cohort","strategy"],suffixes=("_development","_validation"))
audit["below_70_rate_development"]=audit.under_70_development/audit.positive_skus_development
audit["below_70_rate_validation"]=audit.under_70_validation/audit.positive_skus_validation
audit["below_70_rate_drop"]=audit.below_70_rate_development-audit.below_70_rate_validation
audit["overfit_flag"]=audit.below_70_rate_drop.gt(0.15)
audit.to_csv(DOC/"recurring_ensemble_overfit_agent_audit.csv",index=False)
validation_summary.to_csv(DOC/"recurring_ensemble_agent_validation.csv",index=False)
display(audit.sort_values(["cohort","overfit_flag","below_70_rate_validation"],ascending=[True,True,False]))

locks=[]
for cohort in cohorts:
    eligible=audit.loc[audit.cohort.eq(cohort)&~audit.overfit_flag,"strategy"]
    ranked=validation_summary.loc[validation_summary.cohort.eq(cohort)&validation_summary.strategy.isin(eligible)].copy()
    if ranked.empty:
        chosen="median"
    else:
        chosen=opt.rank_summary(ranked.assign(candidate_id=ranked.strategy)).iloc[0].strategy
    locks.append({"cohort":cohort,"locked_strategy":chosen,"selection_basis":"validation_with_overfit_guard"})
locks=pd.DataFrame(locks)
locks.to_csv(DOC/"recurring_ensemble_locked_strategies.csv",index=False)
display(locks.merge(validation_summary,on="cohort").query("locked_strategy == strategy")[["cohort","locked_strategy","under_70","under_50","median_wmape","portfolio_wmape","bias_pct"]])

challenger_frames=[]; confidence_frames=[]
for row in locks.itertuples(index=False):
    parts=wide[row.cohort]
    final=parts["final"].copy()
    final["forecast"]=layer.fit_predict_strategy(row.locked_strategy,parts["all_known"],final,parts["columns"],RANDOM_STATE).forecast
    final["cohort"]=row.cohort; final["strategy"]=row.locked_strategy; final["source"]="layered_challenger"
    challenger_frames.append(final)
    # Confidence is trained from honest OOF predictions for the now-locked strategy.
    confidence=layer.confidence_agent(oof_predictions[row.cohort][row.locked_strategy],final,parts["columns"],RANDOM_STATE)
    confidence["cohort"]=row.cohort; confidence_frames.append(confidence)
challenger=pd.concat(challenger_frames,ignore_index=True)
confidence=pd.concat(confidence_frames,ignore_index=True)

incumbent=pd.read_csv(project_root/"results/lumpy_02_forecasting/stages/recurring_ab/recurring_ab_recommended_retained_forecasts.csv",parse_dates=["block_start"])
incumbent=incumbent.loc[incumbent.cohort.isin(cohorts)].copy()
comparison=[]; recommended=[]
for cohort in cohorts:
    c=challenger.loc[challenger.cohort.eq(cohort)].copy(); i=incumbent.loc[incumbent.cohort.eq(cohort)].copy()
    _,cs=opt.score_forecast(c); _,is_=opt.score_forecast(i)
    accepted,source=layer.retention_decision(cs,is_)
    comparison.extend([{"cohort":cohort,"source":"layered_challenger","accepted":accepted,**cs},{"cohort":cohort,"source":"recurring_ab_incumbent","accepted":not accepted,**is_}])
    chosen=c if accepted else i
    chosen=chosen.copy(); chosen["recommended_source"]=source; recommended.append(chosen)
comparison=pd.DataFrame(comparison)
recommended=pd.concat(recommended,ignore_index=True)
comparison.to_csv(DOC/"recurring_ensemble_final_champion_comparison.csv",index=False)
challenger.to_csv(OUT/"recurring_ensemble_layered_challenger_forecasts.csv",index=False)
recommended.to_csv(OUT/"recurring_ensemble_recommended_forecasts.csv",index=False)
confidence.to_csv(DOC/"recurring_ensemble_confidence_agent.csv",index=False)
display(comparison[["cohort","source","accepted","positive_skus","under_70","under_50","under_100","median_wmape","portfolio_wmape","bias_pct"]])

sku_tables=[]
for cohort,group in recommended.groupby("cohort"):
    sku,summary=opt.score_forecast(group)
    sku["cohort"]=cohort; sku_tables.append(sku)
sku_results=pd.concat(sku_tables,ignore_index=True).merge(confidence,on=["sku_id","cohort"],how="left")
sku_results.to_csv(DOC/"recurring_ensemble_individual_sku_results.csv",index=False)

coverage=sku_results.loc[sku_results.wmape.notna()].groupby("cohort").agg(
    positive_skus=("sku_id","nunique"),below_50=("wmape",lambda s:s.lt(50).sum()),
    below_70=("wmape",lambda s:s.lt(70).sum()),below_100=("wmape",lambda s:s.lt(100).sum()),
    median_wmape=("wmape","median"),median_mase=("mase","median"),
).reset_index()
for threshold in (50,70,100): coverage[f"pct_below_{threshold}"]=100*coverage[f"below_{threshold}"]/coverage.positive_skus
coverage.to_csv(DOC/"recurring_ensemble_sku_coverage.csv",index=False)
display(coverage)
display(sku_results.groupby(["cohort","confidence_band"],observed=True).agg(skus=("sku_id","nunique"),actual_below_70=("wmape",lambda s:s.lt(70).mean())).reset_index())

plot_data=recommended.groupby(["cohort","block_start"],as_index=False).agg(actual=("target","sum"),forecast=("forecast","sum"))
fig,axes=plt.subplots(1,len(cohorts),figsize=(14,4),sharey=False)
if len(cohorts)==1: axes=[axes]
for ax,(cohort,group) in zip(axes,plot_data.groupby("cohort")):
    ax.plot(group.block_start,group.actual,marker="o",linewidth=2,label="Actual")
    ax.plot(group.block_start,group.forecast,marker="o",linewidth=2,label="Forecast")
    ax.set_title(cohort.replace("_"," ").title()); ax.set_xlabel("3-month block start"); ax.set_ylabel("Demand")
    ax.grid(alpha=.25); ax.legend(); ax.tick_params(axis="x",rotation=35)
fig.suptitle("Required 18-Month Forecast: Actual vs Recommended")
fig.tight_layout(); fig.savefig(DOC/"recurring_ensemble_actual_vs_forecast.png",dpi=160,bbox_inches="tight"); plt.show()

# A six-SKU diagnostic gallery spanning the final WMAPE distribution.
positive=sku_results.loc[sku_results.wmape.notna()].sort_values("wmape")
positions=np.linspace(0,len(positive)-1,6).astype(int)
sample_ids=positive.iloc[positions].sku_id.tolist()
sample=recommended.loc[recommended.sku_id.isin(sample_ids)].copy()
fig,axes=plt.subplots(3,2,figsize=(13,10),sharex=False)
for ax,(sku,group) in zip(axes.ravel(),sample.groupby("sku_id",sort=False)):
    group=group.sort_values("block_start")
    score=float(positive.loc[positive.sku_id.eq(sku),"wmape"].iloc[0])
    ax.plot(group.block_start,group.target,marker="o",label="Actual")
    ax.plot(group.block_start,group.forecast,marker="o",label="Forecast")
    ax.set_title(f"SKU {sku} | WMAPE {score:.1f}%"); ax.grid(alpha=.2); ax.legend(); ax.tick_params(axis="x",rotation=30)
fig.tight_layout(); fig.savefig(DOC/"recurring_ensemble_sku_actual_vs_forecast_gallery.png",dpi=160,bbox_inches="tight"); plt.show()

decisions=locks.merge(comparison.loc[comparison.source.eq("layered_challenger"),["cohort","accepted","under_70","under_50","median_wmape"]],on="cohort")
decisions["routing_agent"]="tested hard and probability-weighted expert routing"
decisions["stacking_agent"]="tested robust, constrained linear, tree, and Tweedie stacks"
decisions["correction_agent"]="tested a half-strength residual correction"
decisions["confidence_agent"]="reported reliability bands; excluded no SKUs"
decisions["overfit_agent"]="blocked strategies losing >15 percentage points below-70 coverage"
decisions.to_csv(DOC/"recurring_ensemble_agent_decision_record.csv",index=False)
display(decisions)
print("Complete. Use final_champion_comparison, sku_coverage, individual_sku_results, and the actual-vs-forecast visuals as the main evidence.")


## 7. Recurring personalised model memory

Learns which forecasting experts have worked best for each SKU in earlier periods. Short histories are stabilised using cohort and similar-SKU evidence.


In [ ]:
opt = lumpy_ab_optimization
layer = lumpy_layered_ensemble
online = lumpy_online_experts

roots=[Path.cwd(),Path.cwd().parent,Path(r"D:/Lumpy_Fellas-/Inchscape Repo")]
project_root=next(r.resolve() for r in roots if (r/"src"/"lumpy_forecasting.py").exists())
sys.path.insert(0,str(project_root/"src")) if str(project_root/"src") not in sys.path else None
for module in (opt,layer,online): importlib.reload(module)

OUT=project_root/"results"/"lumpy_02_forecasting/stages/recurring_memory"
DOC=project_root/"docs"/"lumpy_02_forecasting/stages/recurring_memory"
CKPT=project_root/"results"/"lumpy_02_forecasting/stages/recurring_optimisation"/"checkpoints_v1"
for path in (OUT,DOC): path.mkdir(parents=True,exist_ok=True)
print("Configurations:",len(online.config_grid()),"| no base models will be refitted")

focus=pd.read_csv(project_root/"docs/lumpy_02_forecasting/stages/recurring_optimisation/recurring_optimisation_focus_segments.csv")
cohorts={name:set(group.sku_id) for name,group in focus.groupby("tournament_cohort")}
splits=pd.read_csv(project_root/"docs/lumpy_02_forecasting/stages/recurring_ab/recurring_ab_rolling_origin_inventory.csv")
catalogue=pd.read_csv(project_root/"docs/lumpy_02_forecasting/stages/recurring_ensemble/recurring_ensemble_expert_catalogue.csv")
structural=pd.concat([pd.read_pickle(CKPT/f"origin_{oid:02d}__structural.pkl") for oid in splits.origin_id.astype(int)],ignore_index=True)
classical=pd.concat([pd.read_pickle(CKPT/f"origin_{oid:02d}__classical.pkl") for oid in splits.origin_id.astype(int)],ignore_index=True)
final_structural=pd.read_pickle(CKPT/"final_structural.pkl"); final_classical=pd.read_pickle(CKPT/"final_classical.pkl")

incumbent_id="direct_d2_base_p135__w48__direct__p1.00__t0.00__s0.75__none__b0.00"
wide={}
for cohort,ids in cohorts.items():
    ids_for_pool=catalogue.loc[catalogue.cohort.eq(cohort)].sort_values("candidate_column").candidate_id.tolist()
    known,columns,map_table=layer.build_wide_candidate_frame(ids_for_pool,structural,classical,ids)
    final,final_columns,_=layer.build_wide_candidate_frame(ids_for_pool,final_structural,final_classical,ids)
    assert columns==final_columns
    for frame in (known,final):
        frame["synthetic_mean"]=frame[columns].mean(axis=1)
        frame["synthetic_median"]=frame[columns].median(axis=1)
    experts=columns+["synthetic_mean","synthetic_median"]
    if cohort=="recurring_a":
        incumbent_column=map_table.loc[map_table.candidate_id.eq(incumbent_id),"candidate_column"].iloc[0]
        known["current_champion"]=known[incumbent_column]; final["current_champion"]=final[incumbent_column]
    else:
        known["current_champion"]=known["synthetic_mean"]; final["current_champion"]=final["synthetic_mean"]
    wide[cohort]={"known":known,"final":final,"experts":experts}
    print(cohort,"SKUs",known.sku_id.nunique(),"experts",len(experts))

grid=online.config_grid(); lookup={config.config_id:config for config in grid}
tuning_rows=[]; started=time.perf_counter()
for cohort,parts in wide.items():
    print("Tuning",cohort,flush=True)
    for number,config in enumerate(grid,1):
        predicted=online.prequential_forecast(parts["known"],parts["experts"],config,[3,4,5])
        row=online.score_config(predicted,config); row["cohort"]=cohort; tuning_rows.append(row)
        if number%25==0: print(f"  [{number}/{len(grid)}] elapsed {(time.perf_counter()-started)/60:.1f}m",flush=True)
tuning=pd.DataFrame(tuning_rows)
tuning.to_csv(DOC/"recurring_memory_prequential_tuning.csv",index=False)
shortlists={cohort:online.rank_configs(tuning.loc[tuning.cohort.eq(cohort)]).head(20).config_id.tolist() for cohort in cohorts}
display(pd.concat([online.rank_configs(tuning.loc[tuning.cohort.eq(cohort)]).head(10) for cohort in cohorts])[["cohort","config_id","under_70","under_50","under_100","median_wmape","portfolio_wmape","bias_pct"]])

validation_rows=[]; validation_predictions={}
for cohort,parts in wide.items():
    for config_id in shortlists[cohort]:
        config=lookup[config_id]
        predicted=online.prequential_forecast(parts["known"],parts["experts"],config,[6,7])
        row=online.score_config(predicted,config); row["cohort"]=cohort; validation_rows.append(row)
        validation_predictions[(cohort,config_id)]=predicted
validation=pd.DataFrame(validation_rows)

tune_rates=tuning[["cohort","config_id","positive_skus","under_70"]].copy()
tune_rates["development_rate"]=tune_rates.under_70/tune_rates.positive_skus
audit=validation.merge(tune_rates[["cohort","config_id","development_rate"]],on=["cohort","config_id"])
audit["validation_rate"]=audit.under_70/audit.positive_skus
audit["rate_drop"]=audit.development_rate-audit.validation_rate
audit["overfit_flag"]=audit.rate_drop.gt(.20)
audit.to_csv(DOC/"recurring_memory_validation_and_overfit_audit.csv",index=False)

locks=[]
for cohort in cohorts:
    eligible=audit.loc[audit.cohort.eq(cohort)&~audit.overfit_flag]
    if eligible.empty: eligible=audit.loc[audit.cohort.eq(cohort)]
    chosen=online.rank_configs(eligible).iloc[0]
    locks.append({"cohort":cohort,"config_id":chosen.config_id,"validation_under_70":chosen.under_70,"validation_under_50":chosen.under_50,"validation_median_wmape":chosen.median_wmape,"overfit_flag":chosen.overfit_flag})
locks=pd.DataFrame(locks); locks.to_csv(DOC/"recurring_memory_locked_memory_configs.csv",index=False)
display(locks)

challengers=[]; weight_rows=[]
for row in locks.itertuples(index=False):
    parts=wide[row.cohort]; config=lookup[row.config_id]
    predicted=online.forecast_with_memory(parts["known"],parts["final"],parts["experts"],config)
    predicted["cohort"]=row.cohort; predicted["source"]="personalised_memory"
    challengers.append(predicted)
    weights=online.expert_weights(parts["known"],parts["experts"],config)
    diagnostic=pd.DataFrame({"sku_id":weights.index,"top_expert":weights.idxmax(axis=1).to_numpy(),"top_expert_weight":weights.max(axis=1).to_numpy(),"effective_experts":1/np.square(weights).sum(axis=1).to_numpy()})
    diagnostic["cohort"]=row.cohort; weight_rows.append(diagnostic)
challenger=pd.concat(challengers,ignore_index=True)
weights=pd.concat(weight_rows,ignore_index=True)

incumbent=pd.read_csv(project_root/"results/lumpy_02_forecasting/stages/recurring_ensemble/recurring_ensemble_recommended_forecasts.csv",parse_dates=["block_start"])
comparison=[]; recommended=[]
for cohort in cohorts:
    c=challenger.loc[challenger.cohort.eq(cohort)].copy(); i=incumbent.loc[incumbent.cohort.eq(cohort)].copy()
    _,cs=opt.score_forecast(c); _,is_=opt.score_forecast(i)
    accepted,source=layer.retention_decision(cs,is_)
    comparison.extend([{"cohort":cohort,"source":"personalised_memory","accepted":accepted,**cs},{"cohort":cohort,"source":"recurring_ensemble_champion","accepted":not accepted,**is_}])
    chosen=c if accepted else i; chosen=chosen.copy(); chosen["recommended_source"]=source; recommended.append(chosen)
comparison=pd.DataFrame(comparison); recommended=pd.concat(recommended,ignore_index=True)
comparison.to_csv(DOC/"recurring_memory_final_champion_comparison.csv",index=False)
weights.to_csv(DOC/"recurring_memory_sku_expert_memory.csv",index=False)
challenger.to_csv(OUT/"recurring_memory_memory_challenger_forecasts.csv",index=False)
recommended.to_csv(OUT/"recurring_memory_recommended_forecasts.csv",index=False)
display(comparison[["cohort","source","accepted","positive_skus","under_50","under_70","under_100","median_wmape","portfolio_wmape","bias_pct"]])

sku_tables=[]
for cohort,group in recommended.groupby("cohort"):
    sku,summary=opt.score_forecast(group); sku["cohort"]=cohort; sku_tables.append(sku)
sku_results=pd.concat(sku_tables,ignore_index=True).merge(weights,on=["sku_id","cohort"],how="left")
sku_results.to_csv(DOC/"recurring_memory_individual_sku_results.csv",index=False)
coverage=sku_results.loc[sku_results.wmape.notna()].groupby("cohort").agg(positive_skus=("sku_id","nunique"),below_50=("wmape",lambda s:s.lt(50).sum()),below_70=("wmape",lambda s:s.lt(70).sum()),below_100=("wmape",lambda s:s.lt(100).sum()),median_wmape=("wmape","median"),median_mase=("mase","median")).reset_index()
for threshold in (50,70,100): coverage[f"pct_below_{threshold}"]=100*coverage[f"below_{threshold}"]/coverage.positive_skus
coverage.to_csv(DOC/"recurring_memory_sku_coverage.csv",index=False)
display(coverage)
display(weights.groupby("cohort").agg(mean_top_weight=("top_expert_weight","mean"),mean_effective_experts=("effective_experts","mean")))

plot_data=recommended.groupby(["cohort","block_start"],as_index=False).agg(actual=("target","sum"),forecast=("forecast","sum"))
fig,axes=plt.subplots(1,len(cohorts),figsize=(14,4))
if len(cohorts)==1: axes=[axes]
for ax,(cohort,group) in zip(axes,plot_data.groupby("cohort")):
    ax.plot(group.block_start,group.actual,marker="o",linewidth=2,label="Actual")
    ax.plot(group.block_start,group.forecast,marker="o",linewidth=2,label="Forecast")
    ax.set_title(cohort.replace("_"," ").title()); ax.set_xlabel("3-month block start"); ax.set_ylabel("Demand"); ax.grid(alpha=.25); ax.legend(); ax.tick_params(axis="x",rotation=35)
fig.suptitle("Personalised Expert Memory: Actual vs Recommended")
fig.tight_layout(); fig.savefig(DOC/"recurring_memory_actual_vs_forecast.png",dpi=160,bbox_inches="tight"); plt.show()

positive=sku_results.loc[sku_results.wmape.notna()].sort_values("wmape"); positions=np.linspace(0,len(positive)-1,6).astype(int); sample_ids=positive.iloc[positions].sku_id.tolist()
sample=recommended.loc[recommended.sku_id.isin(sample_ids)]
fig,axes=plt.subplots(3,2,figsize=(13,10))
for ax,(sku,group) in zip(axes.ravel(),sample.groupby("sku_id",sort=False)):
    group=group.sort_values("block_start"); score=float(positive.loc[positive.sku_id.eq(sku),"wmape"].iloc[0])
    ax.plot(group.block_start,group.target,marker="o",label="Actual"); ax.plot(group.block_start,group.forecast,marker="o",label="Forecast")
    ax.set_title(f"SKU {sku} | WMAPE {score:.1f}%"); ax.grid(alpha=.2); ax.legend(); ax.tick_params(axis="x",rotation=30)
fig.tight_layout(); fig.savefig(DOC/"recurring_memory_sku_actual_vs_forecast_gallery.png",dpi=160,bbox_inches="tight"); plt.show()

record=locks.merge(comparison.loc[comparison.source.eq("personalised_memory"),["cohort","accepted","under_70","under_50","median_wmape"]],on="cohort")
record["memory_agent"]="per-SKU exponentially weighted expert performance"
record["peer_agent"]="nearest-neighbour loss borrowing controlled by peer weight"
record["shift_agent"]="shortens memory after a detected demand-level change"
record["concentration_agent"]="tests top-2, top-4, and all-expert portfolios"
record["retention_agent"]="keeps Recurring layered ensemble unless the final SKU ordering improves"
record.to_csv(DOC/"recurring_memory_decision_record.csv",index=False)
display(record)
print("Complete. This is the final A/B modelling challenge before lifecycle expansion.")


## 8. Recurring C strict-cutoff route

Handles lower-volume recurring C SKUs separately. The model is selected using a complete historical six-block horizon without using information from the final period.


In [ ]:
lf = lumpy_forecasting
bh = lumpy_block_hybrid
opt = lumpy_ab_optimization

roots=[Path.cwd(),Path.cwd().parent,Path(r"D:/Lumpy_Fellas-/Inchscape Repo")]
project_root=next(r.resolve() for r in roots if (r/"src"/"lumpy_forecasting.py").exists())
sys.path.insert(0,str(project_root/"src")) if str(project_root/"src") not in sys.path else None
for module in (lf,bh,opt): importlib.reload(module)

OUT=project_root/"results"/"lumpy_02_forecasting/stages/recurring_c"; CKPT=OUT/"checkpoints_v2_h18"; DOC=project_root/"docs"/"lumpy_02_forecasting/stages/recurring_c"
for path in (OUT,CKPT,DOC): path.mkdir(parents=True,exist_ok=True)
CONFIG=lf.LumpyConfig(variant="all_sku_history",train_months=48,gap_months=3,test_months=18,step_months=3,min_train_months=18,max_folds=6,external_mode="off")
SCOPE_SPECS={"pooled_transfer":("direct_d2_fast_p135","direct_d2_base_p135","hurdle_d1_sqrt_log","hurdle_d2_sqrt_log","hurdle_d2_full_log"),"c_only":("direct_d2_base_p135","hurdle_d1_sqrt_log","hurdle_d2_sqrt_log")}
spec_lookup={spec.key:spec for spec in opt.STRUCTURAL_SPECS}; RESUME=True

sales,external=lf.load_lumpy_inputs(project_root,CONFIG); model_data,_=lf.build_lumpy_model_frame(sales,external,CONFIG)
assignments=pd.read_csv(project_root/"docs/lumpy_02_forecasting/stages/chronology_strategy/chronology_strategy_strategy_assignment_all_690.csv")
recurring=assignments.loc[assignments.lifecycle_tier.eq("established")&assignments.frequency_tier.eq("recurring_4_6")&assignments.abc_units_tier.isin(["A","B","C"])]
pool_ids=set(recurring.sku_id); c_ids=set(recurring.loc[recurring.abc_units_tier.eq("C"),"sku_id"]); assert len(c_ids)==35
pool_data=model_data.loc[model_data.sku_id.isin(pool_ids)].copy(); c_data=model_data.loc[model_data.sku_id.isin(c_ids)].copy()
strict_job={"train_start":pd.Timestamp("2021-01-01"),"train_end":pd.Timestamp("2022-10-01"),"test_start":pd.Timestamp("2023-02-01"),"test_end":pd.Timestamp("2024-07-01")}
n11=pd.read_csv(project_root/"results/lumpy_02_forecasting/stages/core_block/core_block_block_backtest_forecasts.csv",parse_dates=["train_start","train_end","test_start","test_end"])
final_meta=n11.loc[n11.policy.eq("required_18m_gap3"),["train_start","train_end","test_start","test_end"]].drop_duplicates().sort_values("test_end").iloc[-1]
final_job=final_meta.to_dict()
pd.DataFrame([{"role":"strict_validation",**strict_job},{"role":"final_test",**final_job}]).to_csv(DOC/"recurring_c_strict_horizon.csv",index=False)
print(strict_job,"| final",final_job,"| C SKUs",len(c_ids))

structural_frames=[]; classical_frames=[]; errors=[]; started=time.perf_counter(); counter=0; total=2*sum(map(len,SCOPE_SPECS.values()))
for job_id,job in (("strict",strict_job),("final",final_job)):
  for scope,spec_keys in SCOPE_SPECS.items():
    spath=CKPT/f"{job_id}__{scope}__structural.pkl"; cpath=CKPT/f"{job_id}__{scope}__classical.pkl"
    if RESUME and spath.exists() and cpath.exists(): structural_frames.append(pd.read_pickle(spath)); classical_frames.append(pd.read_pickle(cpath)); print("loaded",job_id,scope); continue
    source=pool_data if scope=="pooled_transfer" else c_data
    train=source.loc[source.month.between(job["train_start"],job["train_end"])].copy(); test=source.loc[source.month.between(job["test_start"],job["test_end"])].copy()
    try:
        prepared=bh.prepare_fold(train,test,lf.SKU_COLUMN,lf.MONTH_COLUMN,lf.TARGET_COLUMN,3,18)
        classical=opt.classical_component_rows(train,prepared,24); classical["trial_id"]=scope+"__"+classical.trial_id; classical["training_scope"]=scope; classical["job_id"]=job_id
        structural=[]
        for spec_key in spec_keys:
            counter+=1; trial_id=f"{scope}__{spec_key}__w24"; frame=opt.fit_structural_components(prepared,spec_lookup[spec_key],trial_id,CONFIG.random_state); frame["training_scope"]=scope; frame["job_id"]=job_id; structural.append(frame); print(f"[{counter}/{total}] {job_id} {trial_id}",flush=True)
        structural=pd.concat(structural,ignore_index=True); structural.to_pickle(spath); classical.to_pickle(cpath); structural_frames.append(structural); classical_frames.append(classical)
    except Exception as exc: errors.append({"scope":f"{job_id}__{scope}","error":repr(exc)})
structural=pd.concat(structural_frames,ignore_index=True); classical=pd.concat(classical_frames,ignore_index=True)
pd.DataFrame(errors,columns=["scope","error"]).to_csv(OUT/"recurring_c_component_errors.csv",index=False)
print("complete",round((time.perf_counter()-started)/60,1),"minutes | errors",len(errors))

def materialise_all(s_part,c_part):
    frames={}; metadata=[]
    for trial_id,g in s_part.groupby("trial_id",sort=False):
        for recipe in opt.recipe_grid(g.architecture.iloc[0]):
            f=opt.compose_components(g,recipe); f=f.loc[f.sku_id.isin(c_ids)].copy(); cid=f.candidate_id.iloc[0]; frames[cid]=f; metadata.append({"candidate_id":cid,"architecture":g.architecture.iloc[0],"training_scope":g.training_scope.iloc[0]})
    for trial_id,g in c_part.groupby("trial_id",sort=False):
        for scale in opt.CALIBRATION_SCALES:
            f=opt.compose_classical(g,scale); f=f.loc[f.sku_id.isin(c_ids)].copy(); cid=f.candidate_id.iloc[0]; frames[cid]=f; metadata.append({"candidate_id":cid,"architecture":"classical","training_scope":g.training_scope.iloc[0]})
    return frames,pd.DataFrame(metadata)

frames,metadata=materialise_all(structural.loc[structural.job_id.eq("strict")],classical.loc[classical.job_id.eq("strict")])
first_rows=[]
for cid,frame in frames.items():
    first=frame.loc[frame.block_number.le(3)]; _,summary=opt.score_forecast(first); meta=metadata.loc[metadata.candidate_id.eq(cid)].iloc[0]; first_rows.append({"candidate_id":cid,"architecture":meta.architecture,"training_scope":meta.training_scope,**summary})
first_rank=opt.rank_summary(pd.DataFrame(first_rows)); first_rank.to_csv(DOC/"recurring_c_first_half_candidate_ranking.csv",index=False)
pool=first_rank.head(10).candidate_id.tolist(); global_ids=pool[:5]
display(first_rank.head(15)[["training_scope","architecture","candidate_id","under_50","under_70","under_100","median_wmape","bias_pct"]])

def align(candidate_ids,blocks):
    aligned=[frames[cid].loc[frames[cid].block_number.isin(blocks)].sort_values(["sku_id","block_start"]).reset_index(drop=True) for cid in candidate_ids]
    return aligned,np.column_stack([frame.forecast for frame in aligned])

first_aligned,first_matrix=align(pool,[1,2,3]); second_aligned,second_matrix=align(pool,[4,5,6])
global_first_aligned,global_first_matrix=align(global_ids,[1,2,3]); global_second_aligned,global_second_matrix=align(global_ids,[4,5,6])
global_first=global_first_aligned[0].copy(); global_first["forecast"]=global_first_matrix.mean(axis=1)
global_second=global_second_aligned[0].copy(); global_second["forecast"]=global_second_matrix.mean(axis=1)

def sku_wmape_map(frame):
    scored,_=opt.score_forecast(frame); return scored.set_index("sku_id").wmape

global_first_score=sku_wmape_map(global_first)
candidate_first_scores={cid:sku_wmape_map(frame) for cid,frame in zip(pool,first_aligned)}
policy_rows=[]; policy_frames={}; route_tables={}
for pool_size in (3,5,10):
    available=pool[:pool_size]
    for margin in (0,5,10,20,30,40):
        for specialist_weight in (.50,.75,1.00):
            routes=[]
            for sku_id in sorted(c_ids):
                candidate_errors={cid:candidate_first_scores[cid].get(sku_id,np.nan) for cid in available}
                valid={cid:value for cid,value in candidate_errors.items() if pd.notna(value)}
                best=min(valid,key=valid.get) if valid else None; baseline=global_first_score.get(sku_id,np.nan); improvement=(baseline-valid[best]) if best is not None and pd.notna(baseline) else -np.inf
                use=best is not None and improvement>=margin
                routes.append({"sku_id":sku_id,"selected_candidate":best if use else "global_mean_top5","historical_improvement":improvement,"used_specialist":use})
            route=pd.DataFrame(routes); out=global_second.copy(); out=out.merge(route[["sku_id","selected_candidate","used_specialist"]],on="sku_id",how="left")
            for cid in available:
                specialist=second_aligned[pool.index(cid)].forecast.to_numpy(); mask=out.selected_candidate.eq(cid).to_numpy(); out.loc[mask,"forecast"]=specialist_weight*specialist[mask]+(1-specialist_weight)*out.loc[mask,"forecast"].to_numpy()
            key=f"pool{pool_size}__margin{margin}__weight{specialist_weight:.2f}"; _,summary=opt.score_forecast(out); policy_rows.append({"candidate_id":key,"policy_id":key,"pool_size":pool_size,"margin":margin,"specialist_weight":specialist_weight,"routed_skus":int(route.used_specialist.sum()),**summary}); policy_frames[key]=out; route_tables[key]=route
_,global_summary=opt.score_forecast(global_second); policy_rows.append({"candidate_id":"global_mean_top5","policy_id":"global_mean_top5","pool_size":0,"margin":np.nan,"specialist_weight":0,"routed_skus":0,**global_summary}); policy_frames["global_mean_top5"]=global_second
policy_rank=opt.rank_summary(pd.DataFrame(policy_rows)); policy_rank.to_csv(DOC/"recurring_c_router_policy_validation.csv",index=False); locked=policy_rank.iloc[0].to_dict(); pd.DataFrame([locked]).to_json(DOC/"recurring_c_locked_router_policy.json",orient="records",indent=2)
display(policy_rank.head(15)[["policy_id","routed_skus","under_50","under_70","under_100","median_wmape","portfolio_wmape","bias_pct"]])

full_global=first_aligned[0].copy(); full_global=pd.concat([global_first,global_second],ignore_index=True); full_global_score=sku_wmape_map(full_global)
full_candidate_scores={}
for cid in pool:
    full=pd.concat([frames[cid].loc[frames[cid].block_number.le(3)],frames[cid].loc[frames[cid].block_number.ge(4)]],ignore_index=True); full_candidate_scores[cid]=sku_wmape_map(full)
if locked["policy_id"]=="global_mean_top5": final_routes=pd.DataFrame({"sku_id":sorted(c_ids),"selected_candidate":"global_mean_top5","historical_improvement":0.0,"used_specialist":False})
else:
    available=pool[:int(locked["pool_size"])]; routes=[]
    for sku_id in sorted(c_ids):
        valid={cid:full_candidate_scores[cid].get(sku_id,np.nan) for cid in available}; valid={cid:v for cid,v in valid.items() if pd.notna(v)}; best=min(valid,key=valid.get) if valid else None; baseline=full_global_score.get(sku_id,np.nan); improvement=(baseline-valid[best]) if best is not None and pd.notna(baseline) else -np.inf; use=best is not None and improvement>=locked["margin"]; routes.append({"sku_id":sku_id,"selected_candidate":best if use else "global_mean_top5","historical_improvement":improvement,"used_specialist":use})
    final_routes=pd.DataFrame(routes)
final_routes.to_csv(DOC/"recurring_c_final_sku_routes.csv",index=False); display(final_routes.groupby("used_specialist").sku_id.nunique().to_frame("skus"))

final_s=structural.loc[structural.job_id.eq("final")]
final_c=classical.loc[classical.job_id.eq("final")]
final_frames={cid:opt.candidate_forecast(final_s,final_c,cid).loc[lambda d:d.sku_id.isin(c_ids)].sort_values(["sku_id","block_start"]).reset_index(drop=True) for cid in pool}
global_parts=[final_frames[cid] for cid in global_ids]; global_matrix=np.column_stack([frame.forecast for frame in global_parts]); final=global_parts[0].copy(); final["forecast"]=global_matrix.mean(axis=1); final=final.merge(final_routes[["sku_id","selected_candidate","used_specialist"]],on="sku_id",how="left")
if locked["policy_id"]!="global_mean_top5":
    weight=float(locked["specialist_weight"])
    for cid in pool[:int(locked["pool_size"])]:
        specialist=final_frames[cid].forecast.to_numpy(); mask=final.selected_candidate.eq(cid).to_numpy(); final.loc[mask,"forecast"]=weight*specialist[mask]+(1-weight)*final.loc[mask,"forecast"].to_numpy()
final["locked_before_final"]=True
sku,summary=opt.score_forecast(final); summary_table=pd.DataFrame([{"cohort":"established_recurring_c","router_policy":locked["policy_id"],"locked_before_final":True,**summary}])
final.to_csv(OUT/"recurring_c_c_recommended_forecasts.csv",index=False); sku.to_csv(DOC/"recurring_c_c_individual_sku_results.csv",index=False); summary_table.to_csv(DOC/"recurring_c_c_final_summary.csv",index=False)
display(summary_table)

global_final=global_parts[0].copy(); global_final["forecast"]=global_matrix.mean(axis=1)
comparison_rows=[]
for name,frame in (("locked_router",final),("global_mean_top5",global_final)):
    _,result=opt.score_forecast(frame); comparison_rows.append({"source":name,"selection_eligible":True,**result})
for cid,frame in final_frames.items():
    _,result=opt.score_forecast(frame); comparison_rows.append({"source":cid,"selection_eligible":False,**result})
comparison=pd.DataFrame(comparison_rows)
eligible=comparison.loc[comparison.selection_eligible].copy(); diagnostic=comparison.loc[~comparison.selection_eligible].copy()
eligible.to_csv(DOC/"recurring_c_final_router_vs_global.csv",index=False)
opt.rank_summary(diagnostic.assign(candidate_id=diagnostic.source)).to_csv(DOC/"recurring_c_diagnostic_final_candidates.csv",index=False)

oracle=[]
for sku_id in sorted(c_ids):
    actual=final_frames[pool[0]].loc[lambda d:d.sku_id.eq(sku_id),"target"].sum()
    if actual<=0: continue
    best=np.inf
    for frame in final_frames.values():
        group=frame.loc[frame.sku_id.eq(sku_id)]; best=min(best,100*(group.target-group.forecast).abs().sum()/actual)
    oracle.append(best)
oracle=np.asarray(oracle)
oracle_summary=pd.DataFrame([{"positive_skus":len(oracle),"below_50":int((oracle<50).sum()),"below_70":int((oracle<70).sum()),"below_100":int((oracle<100).sum()),"median_wmape":float(np.median(oracle)),"selection_eligible":False}])
oracle_summary.to_csv(DOC/"recurring_c_diagnostic_oracle_ceiling.csv",index=False)
display(eligible[["source","under_50","under_70","under_100","median_wmape","portfolio_wmape","bias_pct"]]); display(oracle_summary)

aggregate=final.groupby("block_start",as_index=False).agg(actual=("target","sum"),forecast=("forecast","sum")); fig,ax=plt.subplots(figsize=(9,4)); ax.plot(aggregate.block_start,aggregate.actual,marker="o",linewidth=2,label="Actual"); ax.plot(aggregate.block_start,aggregate.forecast,marker="o",linewidth=2,label="Forecast"); ax.set_title("C Strict-Cutoff SKU Router: Actual vs Forecast"); ax.set_xlabel("3-month block start"); ax.set_ylabel("Demand"); ax.grid(alpha=.25); ax.legend(); ax.tick_params(axis="x",rotation=35); fig.tight_layout(); fig.savefig(DOC/"recurring_c_c_actual_vs_forecast.png",dpi=160,bbox_inches="tight"); plt.show()
positive=sku.loc[sku.wmape.notna()].sort_values("wmape"); positions=np.linspace(0,len(positive)-1,min(6,len(positive))).astype(int); sample_ids=positive.iloc[positions].sku_id.tolist(); sample=final.loc[final.sku_id.isin(sample_ids)]; fig,axes=plt.subplots(3,2,figsize=(13,10))
for ax,(sku_id,group) in zip(axes.ravel(),sample.groupby("sku_id",sort=False)):
    group=group.sort_values("block_start"); score=float(positive.loc[positive.sku_id.eq(sku_id),"wmape"].iloc[0]); ax.plot(group.block_start,group.target,marker="o",label="Actual"); ax.plot(group.block_start,group.forecast,marker="o",label="Forecast"); ax.set_title(f"SKU {sku_id} | WMAPE {score:.1f}%"); ax.grid(alpha=.2); ax.legend(); ax.tick_params(axis="x",rotation=30)
fig.tight_layout(); fig.savefig(DOC/"recurring_c_c_sku_gallery.png",dpi=160,bbox_inches="tight"); plt.show(); print("Complete. No gap-period or final outcomes were used for routing.")


## 9. Final recurring A/B/C result

Combines the locked A, B and C processes and scores all recurring SKUs under one identical evaluation contract.


In [ ]:
opt = lumpy_ab_optimization

roots=[Path.cwd(),Path.cwd().parent,Path(r"D:/Lumpy_Fellas-/Inchscape Repo")]
project_root=next(r.resolve() for r in roots if (r/"src"/"lumpy_forecasting.py").exists())
sys.path.insert(0,str(project_root/"src")) if str(project_root/"src") not in sys.path else None

DOC=project_root/"docs"/"lumpy_02_forecasting/stages/recurring_scorecard"; OUT=project_root/"results"/"lumpy_02_forecasting/stages/recurring_scorecard"
for path in (DOC,OUT): path.mkdir(parents=True,exist_ok=True)

ab=pd.read_csv(project_root/"results/lumpy_02_forecasting/stages/recurring_memory/recurring_memory_memory_challenger_forecasts.csv",parse_dates=["block_start"])
c=pd.read_csv(project_root/"results/lumpy_02_forecasting/stages/recurring_c/recurring_c_c_recommended_forecasts.csv",parse_dates=["block_start"])
ab["official_source_notebook"]="17_validation_locked"
c["cohort"]="recurring_c"; c["official_source_notebook"]="20_validation_locked"
official=pd.concat([ab,c],ignore_index=True,sort=False)
expected_starts=pd.to_datetime(["2024-11-01","2025-02-01","2025-05-01","2025-08-01","2025-11-01","2026-02-01"])
assert set(official.block_start)==set(expected_starts)
assert official.groupby("sku_id").block_start.nunique().eq(6).all()
assert official.sku_id.nunique()==360
official.to_csv(OUT/"recurring_scorecard_official_abc_forecasts.csv",index=False)
print("Verified 360 SKUs, six blocks each, common dates, common metric inputs, and champions locked before final evaluation.")

sku_tables=[]; summary_rows=[]
for cohort,group in official.groupby("cohort",sort=True):
    sku,summary=opt.score_forecast(group); sku["cohort"]=cohort; sku["official_source_notebook"]=group.official_source_notebook.iloc[0]; sku_tables.append(sku); summary_rows.append({"cohort":cohort,**summary})
combined_sku,combined_summary=opt.score_forecast(official); combined_sku["cohort"]="all_abc"; combined_sku["official_source_notebook"]="17_and_20"
sku_results=pd.concat(sku_tables,ignore_index=True)
summary=pd.DataFrame(summary_rows+[{"cohort":"all_abc",**combined_summary}])
for threshold in (50,70,100): summary[f"pct_below_{threshold}"]=100*summary[f"under_{threshold}"]/summary.positive_skus
sku_results.to_csv(DOC/"recurring_scorecard_official_individual_sku_results.csv",index=False)
summary.to_csv(DOC/"recurring_scorecard_official_abc_summary.csv",index=False)
display(summary[["cohort","all_skus","positive_skus","under_50","pct_below_50","under_70","pct_below_70","under_100","pct_below_100","median_wmape","portfolio_wmape","bias_pct"]])

rows=[]
for cohort,group in list(official.groupby("cohort"))+[("all_abc",official)]:
    frame=group.copy(); frame["absolute_error"]=(frame.target-frame.forecast).abs()
    block=frame.groupby("block_start",as_index=False).agg(actual=("target","sum"),forecast=("forecast","sum"),absolute_error=("absolute_error","sum"))
    block["block_portfolio_wmape"]=100*block.absolute_error/block.actual
    block["cohort"]=cohort; rows.append(block)
block_results=pd.concat(rows,ignore_index=True)
block_results.to_csv(DOC/"recurring_scorecard_official_three_month_blocks.csv",index=False)
block_average=block_results.groupby("cohort",as_index=False).block_portfolio_wmape.mean().rename(columns={"block_portfolio_wmape":"unweighted_mean_of_six_block_portfolio_wmapes"})
block_average.to_csv(DOC/"recurring_scorecard_secondary_block_average.csv",index=False)
display(block_results.loc[block_results.cohort.eq("all_abc")])

registry=pd.DataFrame([
    {"result_lane":"required","horizon_months":18,"gap_months":3,"block_months":3,"status":"official","comparison_rule":"A/B/C reported together"},
    {"result_lane":"recommended_challenger","horizon_months":9,"gap_months":1,"block_months":3,"status":"not_yet_run_for_all_abc","comparison_rule":"must be run for A/B/C together before recommendation"},
])
registry.to_csv(DOC/"recurring_scorecard_configuration_registry.csv",index=False); display(registry)

plot=official.groupby(["cohort","block_start"],as_index=False).agg(actual=("target","sum"),forecast=("forecast","sum"))
fig,axes=plt.subplots(2,2,figsize=(14,9)); panels=list(plot.groupby("cohort"))+[("all_abc",official.groupby("block_start",as_index=False).agg(actual=("target","sum"),forecast=("forecast","sum")))]
for ax,(cohort,group) in zip(axes.ravel(),panels):
    ax.plot(group.block_start,group.actual,marker="o",linewidth=2,label="Actual"); ax.plot(group.block_start,group.forecast,marker="o",linewidth=2,label="Forecast"); ax.set_title(cohort.replace("_"," ").title()); ax.set_xlabel("3-month block start"); ax.set_ylabel("Demand"); ax.grid(alpha=.25); ax.legend(); ax.tick_params(axis="x",rotation=30)
fig.suptitle("Official 18-Month / 3-Month-Gap A/B/C Scorecard"); fig.tight_layout(); fig.savefig(DOC/"recurring_scorecard_official_actual_vs_forecast.png",dpi=160,bbox_inches="tight"); plt.show()


## 10. Occasional-demand baseline

Begins the focused work on established occasional SKUs. It compares hurdle models, intermittent-demand methods and pooled approaches.


In [ ]:
lf = lumpy_forecasting
bh = lumpy_block_hybrid
opt = lumpy_ab_optimization

roots=[Path.cwd(),Path.cwd().parent,Path(r"D:/Lumpy_Fellas-/Inchscape Repo")]
project_root=next(r.resolve() for r in roots if (r/"src"/"lumpy_forecasting.py").exists())
sys.path.insert(0,str(project_root/"src")) if str(project_root/"src") not in sys.path else None
for module in (lf,bh,opt): importlib.reload(module)

OUT=project_root/"results"/"lumpy_02_forecasting/stages/occasional_baseline"; CKPT=OUT/"checkpoints_v1_h18"; DOC=project_root/"docs"/"lumpy_02_forecasting/stages/occasional_baseline"
for path in (OUT,CKPT,DOC): path.mkdir(parents=True,exist_ok=True)
CONFIG=lf.LumpyConfig(variant="all_sku_history",train_months=48,gap_months=3,test_months=18,step_months=3,min_train_months=18,max_folds=6,external_mode="off")
SPECS=("direct_d2_fast_p135","direct_d2_base_p135","hurdle_d1_sqrt_log","hurdle_d2_sqrt_log","hurdle_d2_full_log")
spec_lookup={spec.key:spec for spec in opt.STRUCTURAL_SPECS}; RESUME=True

sales,external=lf.load_lumpy_inputs(project_root,CONFIG); model_data,_=lf.build_lumpy_model_frame(sales,external,CONFIG)
assignments=pd.read_csv(project_root/"docs/lumpy_02_forecasting/stages/chronology_strategy/chronology_strategy_strategy_assignment_all_690.csv")
focus=assignments.loc[assignments.lifecycle_tier.eq("established")&assignments.frequency_tier.eq("occasional_2_3")&assignments.abc_units_tier.isin(["A","B","C"])].copy()
focus_ids=set(focus.sku_id); established_ids=set(assignments.loc[assignments.lifecycle_tier.eq("established"),"sku_id"])
assert len(focus_ids)==116
occasional_data=model_data.loc[model_data.sku_id.isin(focus_ids)].copy(); established_data=model_data.loc[model_data.sku_id.isin(established_ids)].copy()
cohorts={f"occasional_{tier.lower()}":set(group.sku_id) for tier,group in focus.groupby("abc_units_tier")}
strict_job={"train_start":pd.Timestamp("2021-01-01"),"train_end":pd.Timestamp("2022-10-01"),"test_start":pd.Timestamp("2023-02-01"),"test_end":pd.Timestamp("2024-07-01")}
n11=pd.read_csv(project_root/"results/lumpy_02_forecasting/stages/core_block/core_block_block_backtest_forecasts.csv",parse_dates=["train_start","train_end","test_start","test_end"])
final_job=n11.loc[n11.policy.eq("required_18m_gap3"),["train_start","train_end","test_start","test_end"]].drop_duplicates().sort_values("test_end").iloc[-1].to_dict()
jobs={"validation":strict_job,"final":final_job}
focus[["sku_id","abc_units_tier"]].to_csv(DOC/"occasional_baseline_population.csv",index=False); display(focus.groupby("abc_units_tier").sku_id.nunique().to_frame("skus")); display(pd.DataFrame([{"role":key,**value} for key,value in jobs.items()]))

structural_frames=[]; classical_frames=[]; errors=[]; total=len(jobs)*2*len(SPECS); counter=0; started=time.perf_counter()
for job_id,job in jobs.items():
  for scope,source in (("occasional_pool",occasional_data),("established_transfer",established_data)):
    spath=CKPT/f"{job_id}__{scope}__structural.pkl"; cpath=CKPT/f"{job_id}__{scope}__classical.pkl"
    if RESUME and spath.exists() and cpath.exists(): structural_frames.append(pd.read_pickle(spath)); classical_frames.append(pd.read_pickle(cpath)); print("loaded",job_id,scope); continue
    train_start=max(pd.Timestamp(source.month.min()),pd.Timestamp(job["train_end"])-pd.DateOffset(months=23))
    train=source.loc[source.month.between(train_start,job["train_end"])].copy(); test=source.loc[source.month.between(job["test_start"],job["test_end"])].copy()
    try:
        prepared=bh.prepare_fold(train,test,lf.SKU_COLUMN,lf.MONTH_COLUMN,lf.TARGET_COLUMN,3,18)
        assert prepared["test_samples"].groupby("sku_id").block_number.nunique().eq(6).all()
        classical=opt.classical_component_rows(train,prepared,24); classical["trial_id"]=scope+"__"+classical.trial_id; classical["training_scope"]=scope; classical["job_id"]=job_id
        fitted=[]
        for spec_key in SPECS:
            counter+=1; trial_id=f"{scope}__{spec_key}__w24"; frame=opt.fit_structural_components(prepared,spec_lookup[spec_key],trial_id,CONFIG.random_state); frame["training_scope"]=scope; frame["job_id"]=job_id; fitted.append(frame); print(f"[{counter}/{total}] {job_id} {trial_id}",flush=True)
        structural=pd.concat(fitted,ignore_index=True); structural.to_pickle(spath); classical.to_pickle(cpath); structural_frames.append(structural); classical_frames.append(classical)
    except Exception as exc: errors.append({"job_id":job_id,"scope":scope,"error":repr(exc)})
structural=pd.concat(structural_frames,ignore_index=True); classical=pd.concat(classical_frames,ignore_index=True)
pd.DataFrame(errors,columns=["job_id","scope","error"]).to_csv(OUT/"occasional_baseline_component_errors.csv",index=False)
print("complete",round((time.perf_counter()-started)/60,1),"minutes | errors",len(errors))

def materialise(s_part,c_part):
    frames={}; metadata=[]
    for trial_id,g in s_part.groupby("trial_id",sort=False):
        for recipe in opt.recipe_grid(g.architecture.iloc[0]):
            frame=opt.compose_components(g,recipe); frame=frame.loc[frame.sku_id.isin(focus_ids)].copy(); cid=frame.candidate_id.iloc[0]; frames[cid]=frame; metadata.append({"candidate_id":cid,"architecture":g.architecture.iloc[0],"training_scope":g.training_scope.iloc[0]})
    # Classical forecasts are identical for focus SKUs across training scopes; keep the occasional copy once.
    for trial_id,g in c_part.loc[c_part.training_scope.eq("occasional_pool")].groupby("trial_id",sort=False):
        for scale in opt.CALIBRATION_SCALES:
            frame=opt.compose_classical(g,scale); frame=frame.loc[frame.sku_id.isin(focus_ids)].copy(); cid=frame.candidate_id.iloc[0]; frames[cid]=frame; metadata.append({"candidate_id":cid,"architecture":"classical","training_scope":"occasional_pool"})
    return frames,pd.DataFrame(metadata).drop_duplicates("candidate_id")

validation_frames,metadata=materialise(structural.loc[structural.job_id.eq("validation")],classical.loc[classical.job_id.eq("validation")])
final_frames,_=materialise(structural.loc[structural.job_id.eq("final")],classical.loc[classical.job_id.eq("final")])
assert set(validation_frames)==set(final_frames)
metadata.to_csv(DOC/"occasional_baseline_candidate_catalogue.csv",index=False); print("Candidates",len(validation_frames))

locks=[]; validation_rows=[]; strategy_specs={}
for cohort,ids in cohorts.items():
    single_rows=[]
    for cid,frame in validation_frames.items():
        scored=frame.loc[frame.sku_id.isin(ids)]; _,summary=opt.score_forecast(scored); meta=metadata.loc[metadata.candidate_id.eq(cid)].iloc[0]; single_rows.append({"candidate_id":cid,"architecture":meta.architecture,"training_scope":meta.training_scope,**summary})
    single_rank=opt.rank_summary(pd.DataFrame(single_rows)); single_rank["cohort"]=cohort; validation_rows.append(single_rank)
    top_ids=single_rank.head(5).candidate_id.tolist(); strategy_frames={"best_single":validation_frames[top_ids[0]].loc[lambda d:d.sku_id.isin(ids)].copy()}; strategy_specs[(cohort,"best_single")]={"method":"single","candidate_ids":[top_ids[0]]}
    for count in (3,5):
        chosen=top_ids[:count]; aligned=[validation_frames[cid].loc[lambda d:d.sku_id.isin(ids)].sort_values(["sku_id","block_start"]).reset_index(drop=True) for cid in chosen]; matrix=np.column_stack([frame.forecast for frame in aligned])
        for method,values in (("mean",matrix.mean(axis=1)),("median",np.median(matrix,axis=1))):
            name=f"{method}_top{count}"; frame=aligned[0].copy(); frame["forecast"]=values; strategy_frames[name]=frame; strategy_specs[(cohort,name)]={"method":method,"candidate_ids":chosen}
    for left,right in itertools.combinations(top_ids[:3],2):
        l=validation_frames[left].loc[lambda d:d.sku_id.isin(ids)].sort_values(["sku_id","block_start"]).reset_index(drop=True); rr=validation_frames[right].loc[lambda d:d.sku_id.isin(ids)].sort_values(["sku_id","block_start"]).reset_index(drop=True)
        name=f"pair_{len(strategy_frames)}"; frame=l.copy(); frame["forecast"]=.5*l.forecast+.5*rr.forecast; strategy_frames[name]=frame; strategy_specs[(cohort,name)]={"method":"pair","candidate_ids":[left,right],"left_weight":.5}
    rows=[]
    for name,frame in strategy_frames.items(): _,summary=opt.score_forecast(frame); rows.append({"candidate_id":name,"strategy":name,**summary})
    strategy_rank=opt.rank_summary(pd.DataFrame(rows)); winner=strategy_rank.iloc[0]; spec=strategy_specs[(cohort,winner.strategy)]; locks.append({"cohort":cohort,"locked_strategy":winner.strategy,**spec,"validation_under_70":winner.under_70,"validation_under_50":winner.under_50,"validation_median_wmape":winner.median_wmape})
locks=pd.DataFrame(locks); all_validation=pd.concat(validation_rows,ignore_index=True); locks.to_json(DOC/"occasional_baseline_locked_strategies.json",orient="records",indent=2); all_validation.to_csv(DOC/"occasional_baseline_validation_single_candidates.csv",index=False); display(locks)

recommended=[]
for lock in locks.to_dict("records"):
    ids=cohorts[lock["cohort"]]; candidate_ids=lock["candidate_ids"]; aligned=[final_frames[cid].loc[lambda d:d.sku_id.isin(ids)].sort_values(["sku_id","block_start"]).reset_index(drop=True) for cid in candidate_ids]; matrix=np.column_stack([frame.forecast for frame in aligned]); frame=aligned[0].copy()
    if lock["method"]=="single": values=matrix[:,0]
    elif lock["method"]=="mean": values=matrix.mean(axis=1)
    elif lock["method"]=="median": values=np.median(matrix,axis=1)
    else: values=lock["left_weight"]*matrix[:,0]+(1-lock["left_weight"])*matrix[:,1]
    frame["forecast"]=values; frame["cohort"]=lock["cohort"]; frame["locked_strategy"]=lock["locked_strategy"]; frame["locked_before_final"]=True; recommended.append(frame)
recommended=pd.concat(recommended,ignore_index=True); assert recommended.groupby("sku_id").block_number.nunique().eq(6).all()
recommended.to_csv(OUT/"occasional_baseline_official_occasional_forecasts.csv",index=False)

sku_tables=[]; summaries=[]
for cohort,group in recommended.groupby("cohort"):
    sku,summary=opt.score_forecast(group); sku["cohort"]=cohort; sku_tables.append(sku); summaries.append({"cohort":cohort,**summary})
sku_all,summary_all=opt.score_forecast(recommended); sku_results=pd.concat(sku_tables,ignore_index=True); summary=pd.DataFrame(summaries+[{"cohort":"all_occasional",**summary_all}])
for threshold in (50,70,100): summary[f"pct_below_{threshold}"]=100*summary[f"under_{threshold}"]/summary.positive_skus
sku_results.to_csv(DOC/"occasional_baseline_individual_sku_results.csv",index=False); summary.to_csv(DOC/"occasional_baseline_official_summary.csv",index=False)
display(summary[["cohort","all_skus","positive_skus","under_50","pct_below_50","under_70","pct_below_70","under_100","pct_below_100","median_wmape","portfolio_wmape","bias_pct"]])

oracle_rows=[]
for cohort,ids in cohorts.items():
    values=[]
    for sku_id in ids:
        actual=next(iter(final_frames.values())).loc[lambda d:d.sku_id.eq(sku_id),"target"].sum()
        if actual<=0: continue
        best=np.inf
        for frame in final_frames.values():
            group=frame.loc[frame.sku_id.eq(sku_id)]; best=min(best,100*(group.target-group.forecast).abs().sum()/actual)
        values.append(best)
    values=np.asarray(values); oracle_rows.append({"cohort":cohort,"positive_skus":len(values),"below_50":int((values<50).sum()),"below_70":int((values<70).sum()),"below_100":int((values<100).sum()),"median_wmape":float(np.median(values)),"selection_eligible":False})
oracle=pd.DataFrame(oracle_rows); oracle.to_csv(DOC/"occasional_baseline_diagnostic_oracle_ceiling.csv",index=False); display(oracle)

plot=recommended.groupby(["cohort","block_start"],as_index=False).agg(actual=("target","sum"),forecast=("forecast","sum")); fig,axes=plt.subplots(2,2,figsize=(14,9)); panels=list(plot.groupby("cohort"))+[("all_occasional",recommended.groupby("block_start",as_index=False).agg(actual=("target","sum"),forecast=("forecast","sum")))]
for ax,(cohort,group) in zip(axes.ravel(),panels): ax.plot(group.block_start,group.actual,marker="o",linewidth=2,label="Actual"); ax.plot(group.block_start,group.forecast,marker="o",linewidth=2,label="Forecast"); ax.set_title(cohort.replace("_"," ").title()); ax.set_xlabel("3-month block start"); ax.set_ylabel("Demand"); ax.grid(alpha=.25); ax.legend(); ax.tick_params(axis="x",rotation=30)
fig.suptitle("Established Occasional: Official 18-Month / 3-Month-Gap Result"); fig.tight_layout(); fig.savefig(DOC/"occasional_baseline_actual_vs_forecast.png",dpi=160,bbox_inches="tight"); plt.show(); print("Complete. All results contain six blocks and were locked before final evaluation.")


## 11. Occasional calibrated hurdle

Separates forecasting into:

- Probability that demand occurs
- Expected size when demand occurs
- Number and placement of forecast events

The occurrence probabilities and event budgets are calibrated using historical validation.


In [ ]:
lf = lumpy_forecasting
bh = lumpy_block_hybrid
opt = lumpy_ab_optimization
ob = lumpy_occurrence_budget

roots=[Path.cwd(),Path.cwd().parent,Path(r"D:/Lumpy_Fellas-/Inchscape Repo")]
project_root=next(r.resolve() for r in roots if (r/"src"/"lumpy_forecasting.py").exists())
sys.path.insert(0,str(project_root/"src")) if str(project_root/"src") not in sys.path else None
for module in (lf,bh,opt,ob): importlib.reload(module)

OUT=project_root/"results"/"lumpy_02_forecasting/stages/occasional_hurdle"; CKPT=OUT/"checkpoints"; DOC=project_root/"docs"/"lumpy_02_forecasting/stages/occasional_hurdle"
for path in (OUT,CKPT,DOC): path.mkdir(parents=True,exist_ok=True)
CONFIG=lf.LumpyConfig(variant="all_sku_history",train_months=48,gap_months=3,test_months=18,step_months=3,min_train_months=18,max_folds=6,external_mode="off")
BASE_SPECS=("hurdle_d1_sqrt_log","hurdle_d2_sqrt_log","hurdle_d2_full_log")
NEW_SPECS=("hurdle_d2_sqrt_tweedie","hurdle_d2_long_sqrt_log")
SPECS=BASE_SPECS+NEW_SPECS
spec_lookup={spec.key:spec for spec in opt.STRUCTURAL_SPECS}
CALIBRATIONS=("raw","sigmoid","isotonic")
SIZE_SOURCES=("ml","median","mean","blend50_median","blend50_mean")
RECIPES=ob.event_recipe_grid()

sales,external=lf.load_lumpy_inputs(project_root,CONFIG); model_data,_=lf.build_lumpy_model_frame(sales,external,CONFIG)
assignments=pd.read_csv(project_root/"docs/lumpy_02_forecasting/stages/chronology_strategy/chronology_strategy_strategy_assignment_all_690.csv")
focus=assignments.loc[assignments.lifecycle_tier.eq("established")&assignments.frequency_tier.eq("occasional_2_3")&assignments.abc_units_tier.isin(["A","B","C"])].copy()
focus_ids=set(focus.sku_id); established_ids=set(assignments.loc[assignments.lifecycle_tier.eq("established"),"sku_id"])
assert len(focus_ids)==116
occasional_data=model_data.loc[model_data.sku_id.isin(focus_ids)].copy(); established_data=model_data.loc[model_data.sku_id.isin(established_ids)].copy()
cohorts={f"occasional_{tier.lower()}":set(group.sku_id) for tier,group in focus.groupby("abc_units_tier")}
strict_job={"train_start":pd.Timestamp("2021-01-01"),"train_end":pd.Timestamp("2022-10-01"),"test_start":pd.Timestamp("2023-02-01"),"test_end":pd.Timestamp("2024-07-01")}
n11=pd.read_csv(project_root/"results/lumpy_02_forecasting/stages/core_block/core_block_block_backtest_forecasts.csv",parse_dates=["train_start","train_end","test_start","test_end"])
final_job=n11.loc[n11.policy.eq("required_18m_gap3"),["train_start","train_end","test_start","test_end"]].drop_duplicates().sort_values("test_end").iloc[-1].to_dict()
jobs={"validation":strict_job,"final":final_job}
display(focus.groupby("abc_units_tier").sku_id.nunique().to_frame("skus")); display(pd.DataFrame([{"role":key,**value} for key,value in jobs.items()]))

component_frames=[]; total_jobs=len(jobs)*2; counter=0; started=time.perf_counter()
for job_id,job in jobs.items():
  for scope,source in (("occasional_pool",occasional_data),("established_transfer",established_data)):
    counter+=1; frames=[]
    old_path=project_root/"results/lumpy_02_forecasting/stages/occasional_baseline/checkpoints_v1_h18"/f"{job_id}__{scope}__structural.pkl"
    if old_path.exists():
      old=pd.read_pickle(old_path); frames.append(old.loc[old.architecture.eq("hurdle")&old.trial_id.str.contains("|".join(BASE_SPECS),regex=True)].copy())
    available=set(pd.concat(frames).trial_id.str.extract(r"__(hurdle_.+)__w24")[0].dropna()) if frames else set()
    missing=[key for key in SPECS if key not in available]
    cache=CKPT/f"{job_id}__{scope}__new.pkl"
    cached=pd.read_pickle(cache) if cache.exists() else pd.DataFrame()
    if len(cached): frames.append(cached); available.update(cached.trial_id.str.extract(r"__(hurdle_.+)__w24")[0].dropna())
    missing=[key for key in SPECS if key not in available]
    if missing:
      train_start=max(pd.Timestamp(source.month.min()),pd.Timestamp(job["train_end"])-pd.DateOffset(months=23))
      train=source.loc[source.month.between(train_start,job["train_end"])].copy(); test=source.loc[source.month.between(job["test_start"],job["test_end"])].copy()
      prepared=bh.prepare_fold(train,test,lf.SKU_COLUMN,lf.MONTH_COLUMN,lf.TARGET_COLUMN,3,18)
      fitted=[]
      for spec_key in missing:
        trial_id=f"{scope}__{spec_key}__w24"; fitted.append(opt.fit_structural_components(prepared,spec_lookup[spec_key],trial_id,CONFIG.random_state)); print("fitted",job_id,trial_id,flush=True)
      new=pd.concat(fitted,ignore_index=True); new["training_scope"]=scope; new["job_id"]=job_id
      combined_new=pd.concat([cached,new],ignore_index=True) if len(cached) else new; combined_new.to_pickle(cache); frames.append(new)
    part=pd.concat(frames,ignore_index=True).drop_duplicates(["trial_id","sku_id","block_number"])
    part["training_scope"]=scope; part["job_id"]=job_id; component_frames.append(part)
    print(f"[{counter}/{total_jobs}] {job_id} {scope}: {part.trial_id.nunique()} hurdle structures",flush=True)
structural=pd.concat(component_frames,ignore_index=True)
assert structural.groupby(["job_id","trial_id","sku_id"]).block_number.nunique().eq(6).all()
print("Component stage complete in",round((time.perf_counter()-started)/60,1),"minutes")

for job_id,job in jobs.items():
    train=occasional_data.loc[occasional_data.month.le(job["train_end"])].copy()
    size=ob.historical_size_table(train,focus_ids,24)
    mask=structural.job_id.eq(job_id)
    structural.loc[mask,["historical_positive_median","historical_positive_mean","event_budget"]]=structural.loc[mask,["sku_id"]].merge(size,on="sku_id",how="left")[["historical_positive_median","historical_positive_mean","event_budget"]].to_numpy()
structural[["historical_positive_median","historical_positive_mean"]]=structural[["historical_positive_median","historical_positive_mean"]].fillna(0.0)
structural["event_budget"]=structural.event_budget.fillna(2).astype(int)
display(structural.loc[structural.job_id.eq("final"),["sku_id","event_budget"]].drop_duplicates().event_budget.value_counts().sort_index().to_frame("skus"))

n22_locks=pd.read_json(project_root/"docs/lumpy_02_forecasting/stages/occasional_baseline/occasional_baseline_locked_strategies.json")
def materialise_standard(part):
    frames={}
    for trial_id,g in part.groupby("trial_id",sort=False):
        for recipe in opt.recipe_grid("hurdle"):
            frame=opt.compose_components(g,recipe); frames[frame.candidate_id.iloc[0]]=frame
    return frames
def reconstruct_incumbent(job_id):
    frames=materialise_standard(structural.loc[structural.job_id.eq(job_id)&structural.trial_id.str.contains("|".join(BASE_SPECS),regex=True)])
    output=[]
    for lock in n22_locks.to_dict("records"):
        ids=cohorts[lock["cohort"]]; aligned=[frames[cid].loc[lambda d:d.sku_id.isin(ids)].sort_values(["sku_id","block_start"]).reset_index(drop=True) for cid in lock["candidate_ids"]]
        matrix=np.column_stack([frame.forecast for frame in aligned]); frame=aligned[0].copy()
        if lock["method"]=="single": values=matrix[:,0]
        elif lock["method"]=="mean": values=matrix.mean(axis=1)
        elif lock["method"]=="median": values=np.median(matrix,axis=1)
        else: values=lock["left_weight"]*matrix[:,0]+(1-lock["left_weight"])*matrix[:,1]
        frame["forecast"]=values; frame["cohort"]=lock["cohort"]; output.append(frame)
    return pd.concat(output,ignore_index=True)
incumbent_validation=reconstruct_incumbent("validation"); incumbent_final=reconstruct_incumbent("final")
incumbent_scores={cohort:opt.score_forecast(group)[1] for cohort,group in incumbent_validation.groupby("cohort")}
display(pd.DataFrame([{"cohort":key,**value} for key,value in incumbent_scores.items()]))

validation_parts={trial:g.sort_values(["sku_id","block_start"]).reset_index(drop=True) for trial,g in structural.loc[structural.job_id.eq("validation")].groupby("trial_id",sort=False)}
final_parts={trial:g.sort_values(["sku_id","block_start"]).reset_index(drop=True) for trial,g in structural.loc[structural.job_id.eq("final")].groupby("trial_id",sort=False)}
calibration_cache={}; candidate_specs={}; validation_rows=[]; pure_frames={}; started=time.perf_counter()
trials=sorted(validation_parts)
for trial_number,trial_id in enumerate(trials,1):
    validation=validation_parts[trial_id]; final=final_parts[trial_id]
    for calibration in CALIBRATIONS:
        calibration_cache[(trial_id,calibration)]=ob.cross_fitted_calibration(validation,final,calibration)
    for calibration,size_source,recipe in itertools.product(CALIBRATIONS,SIZE_SOURCES,RECIPES.to_dict("records")):
        probability=calibration_cache[(trial_id,calibration)][0]; size=ob.positive_size(validation,size_source)
        forecast=ob.compose_event_forecast(validation,probability,size,recipe["mode"],recipe["event_count"],recipe["scale"])
        cid=f"{trial_id}__cal_{calibration}__size_{size_source}__{recipe['recipe_id']}"
        candidate_specs[cid]={"trial_id":trial_id,"calibration":calibration,"size_source":size_source,**recipe,"blend_weight":1.0}
        frame=validation[["sku_id","block_start","block_number","target","block_naive_scale"]].copy(); frame["forecast"]=forecast
        pure_frames[cid]=frame
        for cohort,ids in cohorts.items():
            _,summary=opt.score_forecast(frame.loc[frame.sku_id.isin(ids)]); validation_rows.append({"cohort":cohort,"candidate_id":cid,"stage":"pure",**summary})
    print(f"[{trial_number}/{len(trials)}] {trial_id} | {len(validation_rows):,} cohort scores",flush=True)
pure_summary=pd.DataFrame(validation_rows)
blend_rows=[]; blend_specs={}
for cohort,ids in cohorts.items():
    ranked=opt.rank_summary(pure_summary.loc[pure_summary.cohort.eq(cohort)]).head(12)
    incumbent=incumbent_validation.loc[incumbent_validation.sku_id.isin(ids)].sort_values(["sku_id","block_start"]).reset_index(drop=True)
    for cid in ranked.candidate_id:
        candidate=pure_frames[cid].loc[lambda d:d.sku_id.isin(ids)].sort_values(["sku_id","block_start"]).reset_index(drop=True)
        for weight in (0.25,0.50,0.75):
            frame=candidate.copy(); frame["forecast"]=weight*candidate.forecast+(1-weight)*incumbent.forecast
            blend_id=f"{cid}__incumbent_blend_{weight:.2f}"; _,summary=opt.score_forecast(frame)
            blend_rows.append({"cohort":cohort,"candidate_id":blend_id,"stage":"incumbent_blend",**summary})
            blend_specs[blend_id]={**candidate_specs[cid],"pure_candidate_id":cid,"blend_weight":weight}
validation_summary=pd.concat([pure_summary,pd.DataFrame(blend_rows)],ignore_index=True)
candidate_specs.update(blend_specs); validation_summary.to_csv(DOC/"occasional_hurdle_validation_candidates.csv",index=False)
print("Tournament complete in",round((time.perf_counter()-started)/60,1),"minutes | candidates",validation_summary.candidate_id.nunique())
display(pd.concat([opt.rank_summary(group).head(5) for _,group in validation_summary.groupby("cohort")])[['cohort','stage','candidate_id','under_50','under_70','under_100','median_wmape','portfolio_wmape','bias_pct']])

locks=[]
for cohort in cohorts:
    challenger=opt.rank_summary(validation_summary.loc[validation_summary.cohort.eq(cohort)]).iloc[0]
    incumbent=incumbent_scores[cohort]
    accepted=(challenger.under_70>incumbent["under_70"]) or (challenger.under_70==incumbent["under_70"] and challenger.under_50>incumbent["under_50"])
    locks.append({"cohort":cohort,"locked_source":"occasional_hurdle_challenger" if accepted else "occasional_baseline_incumbent","candidate_id":challenger.candidate_id,"candidate_stage":challenger.stage,"challenger_under_50":int(challenger.under_50),"challenger_under_70":int(challenger.under_70),"challenger_under_100":int(challenger.under_100),"challenger_median_wmape":float(challenger.median_wmape),"incumbent_under_50":int(incumbent["under_50"]),"incumbent_under_70":int(incumbent["under_70"]),"incumbent_under_100":int(incumbent["under_100"]),"incumbent_median_wmape":float(incumbent["median_wmape"])})
locks=pd.DataFrame(locks); locks.to_json(DOC/"occasional_hurdle_locked_strategies.json",orient="records",indent=2); display(locks)

def final_candidate(candidate_id):
    spec=candidate_specs[candidate_id]; trial_id=spec["trial_id"]; final=final_parts[trial_id]
    probability=calibration_cache[(trial_id,spec["calibration"])][1]; size=ob.positive_size(final,spec["size_source"])
    forecast=ob.compose_event_forecast(final,probability,size,spec["mode"],spec["event_count"],spec["scale"])
    frame=final[["sku_id","block_start","block_number","target","block_naive_scale"]].copy(); frame["forecast"]=forecast
    if spec["blend_weight"]<1.0:
        incumbent=incumbent_final.sort_values(["sku_id","block_start"]).reset_index(drop=True); aligned=frame.sort_values(["sku_id","block_start"]).reset_index(drop=True)
        aligned["forecast"]=spec["blend_weight"]*aligned.forecast+(1-spec["blend_weight"])*incumbent.forecast; frame=aligned
    return frame
recommended=[]
for lock in locks.to_dict("records"):
    ids=cohorts[lock["cohort"]]
    if lock["locked_source"]=="occasional_hurdle_challenger": frame=final_candidate(lock["candidate_id"])
    else: frame=incumbent_final.copy()
    frame=frame.loc[frame.sku_id.isin(ids)].copy(); frame["cohort"]=lock["cohort"]; frame["locked_source"]=lock["locked_source"]; frame["locked_before_final"]=True; recommended.append(frame)
recommended=pd.concat(recommended,ignore_index=True); assert recommended.groupby("sku_id").block_number.nunique().eq(6).all()
recommended.to_csv(OUT/"occasional_hurdle_official_occasional_forecasts.csv",index=False)
sku_tables=[]; summaries=[]
for cohort,group in recommended.groupby("cohort"):
    sku,summary=opt.score_forecast(group); sku["cohort"]=cohort; sku_tables.append(sku); summaries.append({"cohort":cohort,**summary})
sku_all,summary_all=opt.score_forecast(recommended); sku_results=pd.concat(sku_tables,ignore_index=True); summary=pd.DataFrame(summaries+[{"cohort":"all_occasional",**summary_all}])
for threshold in (50,70,100): summary[f"pct_below_{threshold}"]=100*summary[f"under_{threshold}"]/summary.positive_skus
sku_results.to_csv(DOC/"occasional_hurdle_individual_sku_results.csv",index=False); summary.to_csv(DOC/"occasional_hurdle_official_summary.csv",index=False)
display(summary[["cohort","all_skus","positive_skus","under_50","pct_below_50","under_70","pct_below_70","under_100","pct_below_100","median_wmape","portfolio_wmape","bias_pct"]])

stage_summary=[]
for (cohort,stage),group in validation_summary.groupby(["cohort","stage"]):
    best=opt.rank_summary(group).iloc[0]; stage_summary.append({"cohort":cohort,"stage":stage,"best_under_50":best.under_50,"best_under_70":best.under_70,"best_under_100":best.under_100,"best_median_wmape":best.median_wmape,"best_candidate_id":best.candidate_id})
stage_summary=pd.DataFrame(stage_summary); stage_summary.to_csv(DOC/"occasional_hurdle_stage_attribution.csv",index=False); display(stage_summary)

oracle=[]
top_ids=set(pd.concat([opt.rank_summary(group).head(15) for _,group in validation_summary.groupby("cohort")]).candidate_id)
final_pool={cid:final_candidate(cid) for cid in top_ids}
for cohort,ids in cohorts.items():
    values=[]
    for sku_id in ids:
        actual=incumbent_final.loc[incumbent_final.sku_id.eq(sku_id),"target"].sum()
        if actual<=0: continue
        scores=[]
        for frame in final_pool.values():
            rows=frame.loc[frame.sku_id.eq(sku_id)]; scores.append(100*(rows.target-rows.forecast).abs().sum()/actual)
        values.append(min(scores))
    values=np.asarray(values); oracle.append({"cohort":cohort,"positive_skus":len(values),"below_50":int((values<50).sum()),"below_70":int((values<70).sum()),"below_100":int((values<100).sum()),"median_wmape":float(np.median(values)),"selection_eligible":False})
oracle=pd.DataFrame(oracle); oracle.to_csv(DOC/"occasional_hurdle_diagnostic_oracle_ceiling.csv",index=False); display(oracle)

plot=recommended.groupby(["cohort","block_start"],as_index=False).agg(actual=("target","sum"),forecast=("forecast","sum")); fig,axes=plt.subplots(2,2,figsize=(14,9)); panels=list(plot.groupby("cohort"))+[("all_occasional",recommended.groupby("block_start",as_index=False).agg(actual=("target","sum"),forecast=("forecast","sum")))]
for ax,(cohort,group) in zip(axes.ravel(),panels): ax.plot(group.block_start,group.actual,marker="o",linewidth=2,label="Actual"); ax.plot(group.block_start,group.forecast,marker="o",linewidth=2,label="Forecast"); ax.set_title(cohort.replace("_"," ").title()); ax.set_xlabel("3-month block start"); ax.set_ylabel("Demand"); ax.grid(alpha=.25); ax.legend(); ax.tick_params(axis="x",rotation=30)
fig.suptitle("Established Occasional: Calibrated Hurdle And Event Budget"); fig.tight_layout(); fig.savefig(DOC/"occasional_hurdle_actual_vs_forecast.png",dpi=160,bbox_inches="tight"); plt.show(); print("Complete. Strategies were locked on historical validation before final evaluation.")


## 12. Occasional SKU router

Tests whether SKU characteristics or similar SKUs can choose a better occasional-demand strategy. The existing process is retained when routing does not improve the main below-70% measure.


In [ ]:
lf = lumpy_forecasting
opt = lumpy_ab_optimization
ob = lumpy_occurrence_budget
cr = lumpy_candidate_router
sr = lumpy_sku_router

roots=[Path.cwd(),Path.cwd().parent,Path(r"D:/Lumpy_Fellas-/Inchscape Repo")]
project_root=next(r.resolve() for r in roots if (r/"src"/"lumpy_forecasting.py").exists())
sys.path.insert(0,str(project_root/"src")) if str(project_root/"src") not in sys.path else None
for module in (lf,opt,ob,cr,sr): importlib.reload(module)

OUT=project_root/"results"/"lumpy_02_forecasting/stages/occasional_router"; DOC=project_root/"docs"/"lumpy_02_forecasting/stages/occasional_router"
for path in (OUT,DOC): path.mkdir(parents=True,exist_ok=True)
CONFIG=lf.LumpyConfig(variant="all_sku_history",train_months=48,gap_months=3,test_months=18,step_months=3,min_train_months=18,max_folds=6,external_mode="off")

sales,external=lf.load_lumpy_inputs(project_root,CONFIG); model_data,_=lf.build_lumpy_model_frame(sales,external,CONFIG)
assignments=pd.read_csv(project_root/"docs/lumpy_02_forecasting/stages/chronology_strategy/chronology_strategy_strategy_assignment_all_690.csv")
focus=assignments.loc[assignments.lifecycle_tier.eq("established")&assignments.frequency_tier.eq("occasional_2_3")&assignments.abc_units_tier.isin(["A","B","C"])].copy()
focus_ids=set(focus.sku_id); assert len(focus_ids)==116
occasional_data=model_data.loc[model_data.sku_id.isin(focus_ids)].copy()
cohorts={f"occasional_{tier.lower()}":set(group.sku_id) for tier,group in focus.groupby("abc_units_tier")}
strict_job={"train_start":pd.Timestamp("2021-01-01"),"train_end":pd.Timestamp("2022-10-01"),"test_start":pd.Timestamp("2023-02-01"),"test_end":pd.Timestamp("2024-07-01")}
n11=pd.read_csv(project_root/"results/lumpy_02_forecasting/stages/core_block/core_block_block_backtest_forecasts.csv",parse_dates=["train_start","train_end","test_start","test_end"])
final_job=n11.loc[n11.policy.eq("required_18m_gap3"),["train_start","train_end","test_start","test_end"]].drop_duplicates().sort_values("test_end").iloc[-1].to_dict()
jobs={"validation":strict_job,"final":final_job}
display(focus.groupby("abc_units_tier").sku_id.nunique().to_frame("skus"))

n25_validation=pd.read_csv(project_root/"docs/lumpy_02_forecasting/stages/occasional_hurdle/occasional_hurdle_validation_candidates.csv")
top=[]
for cohort,group in n25_validation.loc[n25_validation.stage.eq("pure")].groupby("cohort"):
    top.append(opt.rank_summary(group).head(10))
library=sorted(pd.concat(top).candidate_id.unique()); pd.DataFrame({"candidate_id":library}).to_csv(DOC/"occasional_router_candidate_library.csv",index=False)
print("Frozen candidate library:",len(library),"strategies")

parts=[]
for job_id in jobs:
  for scope in ("occasional_pool","established_transfer"):
    old_path=project_root/"results/lumpy_02_forecasting/stages/occasional_baseline/checkpoints_v1_h18"/f"{job_id}__{scope}__structural.pkl"
    new_path=project_root/"results/lumpy_02_forecasting/stages/occasional_hurdle/checkpoints"/f"{job_id}__{scope}__new.pkl"
    old=pd.read_pickle(old_path); old=old.loc[old.architecture.eq("hurdle")].copy(); new=pd.read_pickle(new_path)
    part=pd.concat([old,new],ignore_index=True).drop_duplicates(["trial_id","sku_id","block_number"]); part["job_id"]=job_id; parts.append(part)
structural=pd.concat(parts,ignore_index=True)
for job_id,job in jobs.items():
    size=ob.historical_size_table(occasional_data.loc[occasional_data.month.le(job["train_end"])],focus_ids,24)
    mask=structural.job_id.eq(job_id)
    structural.loc[mask,["historical_positive_median","historical_positive_mean","event_budget"]]=structural.loc[mask,["sku_id"]].merge(size,on="sku_id",how="left")[["historical_positive_median","historical_positive_mean","event_budget"]].to_numpy()
structural[["historical_positive_median","historical_positive_mean"]]=structural[["historical_positive_median","historical_positive_mean"]].fillna(0.0); structural["event_budget"]=structural.event_budget.fillna(2).astype(int)
trial_ids=sorted(structural.trial_id.unique()); specs=[cr.parse_candidate_id(cid,trial_ids) for cid in library]
validation_parts={trial:g.sort_values(["sku_id","block_start"]).reset_index(drop=True) for trial,g in structural.loc[structural.job_id.eq("validation")].groupby("trial_id")}
final_parts={trial:g.sort_values(["sku_id","block_start"]).reset_index(drop=True) for trial,g in structural.loc[structural.job_id.eq("final")].groupby("trial_id")}
calibrations={}
for trial_id,calibration in sorted({(spec["trial_id"],spec["calibration"]) for spec in specs}):
    calibrations[(trial_id,calibration)]=ob.cross_fitted_calibration(validation_parts[trial_id],final_parts[trial_id],calibration)
def build_candidates(job_id):
    output=[]; index=0 if job_id=="validation" else 1; source=validation_parts if job_id=="validation" else final_parts
    for spec in specs:
        frame=source[spec["trial_id"]]; probability=calibrations[(spec["trial_id"],spec["calibration"])][index]; size=ob.positive_size(frame,spec["size_source"])
        forecast=ob.compose_event_forecast(frame,probability,size,spec["mode"],spec["event_count"],spec["scale"])
        candidate=frame[["sku_id","block_start","block_number","target","block_naive_scale"]].copy(); candidate["forecast"]=forecast; candidate["candidate_id"]=spec["candidate_id"]; output.append(candidate)
    return pd.concat(output,ignore_index=True)
validation_forecasts=build_candidates("validation"); final_forecasts=build_candidates("final")
validation_errors=cr.sku_candidate_errors(validation_forecasts); print("Candidate forecast rows:",len(validation_forecasts))

metadata=sr.extract_static_metadata(model_data)
def router_features(job):
    train=occasional_data.loc[occasional_data.month.le(job["train_end"])].copy()
    features=sr.history_feature_table(train,focus_ids,metadata)
    return features.merge(focus[["sku_id","abc_units_tier"]].rename(columns={"abc_units_tier":"frozen_abc"}),on="sku_id",how="left")
validation_features=router_features(strict_job); final_features=router_features(final_job)
numeric=[column for column in sr.NUMERIC_ROUTER_FEATURES if column in validation_features and validation_features[column].notna().any()]
categorical=[column for column in ["frozen_abc","recency_tier","size_tier","SUBFAMILY_DESCRIPTION"] if column in validation_features]
print("Router features:",len(numeric),"numeric and",len(categorical),"categorical")

pair=validation_errors.merge(validation_features,on="sku_id",how="left"); pair["target_error"]=pair.wmape.clip(upper=500.0)
candidate_columns=["candidate_id",*categorical]; feature_columns=[*numeric,*candidate_columns]
preprocessor=ColumnTransformer([("numeric",SimpleImputer(strategy="median"),numeric),("categorical",Pipeline([("impute",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore"))]),candidate_columns)])
rf_specs=[("rf_d5_l5",5,5,"raw"),("rf_d8_l5",8,5,"raw"),("rf_d8_l15",8,15,"raw"),("rf_full_l10",None,10,"raw"),("rf_d8_l5_log",8,5,"log")]
router_selections={}; router_rows=[]; splitter=GroupKFold(n_splits=5); started=time.perf_counter()
for name,depth,leaf,target_mode in rf_specs:
    oof=np.zeros(len(pair))
    for train_idx,holdout_idx in splitter.split(pair,groups=pair.sku_id):
        model=Pipeline([("prep",preprocessor),("rf",RandomForestRegressor(n_estimators=250,max_depth=depth,min_samples_leaf=leaf,max_features=.75,n_jobs=-1,random_state=42))])
        y=pair.target_error.iloc[train_idx].to_numpy(); y=np.log1p(y) if target_mode=="log" else y
        model.fit(pair.iloc[train_idx][feature_columns],y); prediction=model.predict(pair.iloc[holdout_idx][feature_columns]); oof[holdout_idx]=np.expm1(prediction) if target_mode=="log" else prediction
    scored=pair[["sku_id","candidate_id"]].copy(); scored["predicted_error"]=oof
    selection=scored.sort_values(["sku_id","predicted_error","candidate_id"]).groupby("sku_id",as_index=False).head(1)[["sku_id","candidate_id"]].rename(columns={"candidate_id":"selected_candidate_id"}); router_selections[name]=selection
    routed=cr.selected_forecasts(validation_forecasts,selection)
    for cohort,ids in cohorts.items(): _,summary=opt.score_forecast(routed.loc[routed.sku_id.isin(ids)]); router_rows.append({"cohort":cohort,"router_id":name,"router_family":"grouped_random_forest","eligible":True,**summary})
    print("complete",name,flush=True)
print("RF routing complete in",round((time.perf_counter()-started)/60,1),"minutes")

for neighbours,aggregation in itertools.product((5,10,20),("median","mean")):
    name=f"knn_{neighbours}_{aggregation}"; selection=cr.neighbour_selections(validation_errors,validation_features,validation_features,numeric,neighbours,aggregation,exclude_same_sku=True); router_selections[name]=selection
    routed=cr.selected_forecasts(validation_forecasts,selection)
    for cohort,ids in cohorts.items(): _,summary=opt.score_forecast(routed.loc[routed.sku_id.isin(ids)]); router_rows.append({"cohort":cohort,"router_id":name,"router_family":"peer_neighbour","eligible":True,**summary})
own=cr.own_history_selections(validation_errors); own_routed=cr.selected_forecasts(validation_forecasts,own)
for cohort,ids in cohorts.items(): _,summary=opt.score_forecast(own_routed.loc[own_routed.sku_id.isin(ids)]); router_rows.append({"cohort":cohort,"router_id":"same_sku_oracle","router_family":"same_sku_history","eligible":False,**summary})
router_summary=pd.DataFrame(router_rows); router_summary["candidate_id"]=router_summary.router_id; router_summary.to_csv(DOC/"occasional_router_router_validation.csv",index=False)
display(pd.concat([opt.rank_summary(group).head(6) for _,group in router_summary.groupby("cohort")])[["cohort","router_id","router_family","eligible","under_50","under_70","under_100","median_wmape","portfolio_wmape"]])

final_router_selections={}
final_pair=pd.DataFrame({"sku_id":np.repeat(final_features.sku_id.to_numpy(),len(library)),"candidate_id":library*len(final_features)}).merge(final_features,on="sku_id",how="left")
for name,depth,leaf,target_mode in rf_specs:
    model=Pipeline([("prep",preprocessor),("rf",RandomForestRegressor(n_estimators=250,max_depth=depth,min_samples_leaf=leaf,max_features=.75,n_jobs=-1,random_state=42))])
    y=pair.target_error.to_numpy(); y=np.log1p(y) if target_mode=="log" else y; model.fit(pair[feature_columns],y)
    prediction=model.predict(final_pair[feature_columns]); final_pair["predicted_error"]=np.expm1(prediction) if target_mode=="log" else prediction
    final_router_selections[name]=final_pair.sort_values(["sku_id","predicted_error","candidate_id"]).groupby("sku_id",as_index=False).head(1)[["sku_id","candidate_id"]].rename(columns={"candidate_id":"selected_candidate_id"})
for neighbours,aggregation in itertools.product((5,10,20),("median","mean")):
    name=f"knn_{neighbours}_{aggregation}"; final_router_selections[name]=cr.neighbour_selections(validation_errors,validation_features,final_features,numeric,neighbours,aggregation,exclude_same_sku=True)

n25_locks=pd.read_json(project_root/"docs/lumpy_02_forecasting/stages/occasional_hurdle/occasional_hurdle_locked_strategies.json")
def n25_baseline(forecasts):
    output=[]
    for lock in n25_locks.to_dict("records"):
        ids=cohorts[lock["cohort"]]; frame=forecasts.loc[forecasts.sku_id.isin(ids)&forecasts.candidate_id.eq(lock["candidate_id"])].copy(); frame["cohort"]=lock["cohort"]; output.append(frame)
    return pd.concat(output,ignore_index=True)
baseline_validation=n25_baseline(validation_forecasts); baseline_final=n25_baseline(final_forecasts)
baseline_scores={cohort:opt.score_forecast(group)[1] for cohort,group in baseline_validation.groupby("cohort")}
locks=[]
for cohort in cohorts:
    eligible=opt.rank_summary(router_summary.loc[router_summary.cohort.eq(cohort)&router_summary.eligible]).iloc[0]; incumbent=baseline_scores[cohort]
    accepted=(eligible.under_70>incumbent["under_70"]) or (eligible.under_70==incumbent["under_70"] and eligible.under_50>incumbent["under_50"])
    locks.append({"cohort":cohort,"locked_source":"occasional_router_router" if accepted else "occasional_hurdle_incumbent","router_id":eligible.router_id,"router_family":eligible.router_family,"router_under_50":int(eligible.under_50),"router_under_70":int(eligible.under_70),"router_under_100":int(eligible.under_100),"router_median_wmape":float(eligible.median_wmape),"incumbent_under_50":int(incumbent["under_50"]),"incumbent_under_70":int(incumbent["under_70"]),"incumbent_under_100":int(incumbent["under_100"]),"incumbent_median_wmape":float(incumbent["median_wmape"])})
locks=pd.DataFrame(locks); locks.to_json(DOC/"occasional_router_locked_routers.json",orient="records",indent=2); display(locks)

recommended=[]
for lock in locks.to_dict("records"):
    ids=cohorts[lock["cohort"]]
    if lock["locked_source"]=="occasional_router_router": frame=cr.selected_forecasts(final_forecasts,final_router_selections[lock["router_id"]])
    else: frame=baseline_final.copy()
    frame=frame.loc[frame.sku_id.isin(ids)].copy(); frame["cohort"]=lock["cohort"]; frame["locked_source"]=lock["locked_source"]; frame["router_id"]=lock["router_id"]; frame["locked_before_final"]=True; recommended.append(frame)
recommended=pd.concat(recommended,ignore_index=True); assert recommended.groupby("sku_id").block_number.nunique().eq(6).all(); recommended.to_csv(OUT/"occasional_router_official_occasional_forecasts.csv",index=False)
sku_tables=[]; summaries=[]
for cohort,group in recommended.groupby("cohort"):
    sku,summary=opt.score_forecast(group); sku["cohort"]=cohort; sku_tables.append(sku); summaries.append({"cohort":cohort,**summary})
sku_all,summary_all=opt.score_forecast(recommended); sku_results=pd.concat(sku_tables,ignore_index=True); summary=pd.DataFrame(summaries+[{"cohort":"all_occasional",**summary_all}])
for threshold in (50,70,100): summary[f"pct_below_{threshold}"]=100*summary[f"under_{threshold}"]/summary.positive_skus
sku_results.to_csv(DOC/"occasional_router_individual_sku_results.csv",index=False); summary.to_csv(DOC/"occasional_router_official_summary.csv",index=False)
display(summary[["cohort","all_skus","positive_skus","under_50","pct_below_50","under_70","pct_below_70","under_100","pct_below_100","median_wmape","portfolio_wmape","bias_pct"]])

plot=recommended.groupby(["cohort","block_start"],as_index=False).agg(actual=("target","sum"),forecast=("forecast","sum")); fig,axes=plt.subplots(2,2,figsize=(14,9)); panels=list(plot.groupby("cohort"))+[("all_occasional",recommended.groupby("block_start",as_index=False).agg(actual=("target","sum"),forecast=("forecast","sum")))]
for ax,(cohort,group) in zip(axes.ravel(),panels): ax.plot(group.block_start,group.actual,marker="o",linewidth=2,label="Actual"); ax.plot(group.block_start,group.forecast,marker="o",linewidth=2,label="Forecast"); ax.set_title(cohort.replace("_"," ").title()); ax.set_xlabel("3-month block start"); ax.set_ylabel("Demand"); ax.grid(alpha=.25); ax.legend(); ax.tick_params(axis="x",rotation=30)
fig.suptitle("Established Occasional: SKU-Level Router"); fig.tight_layout(); fig.savefig(DOC/"occasional_router_actual_vs_forecast.png",dpi=160,bbox_inches="tight"); plt.show(); print("Complete. Eligible routers used grouped or peer-excluded validation and were locked before final evaluation.")


## 13. Occasional temporal memory

Checks whether a strategy that worked for a particular SKU in one historical horizon remains useful in the next horizon. This becomes the final occasional route, although the improvement is small.


In [ ]:
lf = lumpy_forecasting
bh = lumpy_block_hybrid
opt = lumpy_ab_optimization
ob = lumpy_occurrence_budget
cr = lumpy_candidate_router
tm = lumpy_temporal_memory

roots=[Path.cwd(),Path.cwd().parent,Path(r"D:/Lumpy_Fellas-/Inchscape Repo")]
project_root=next(r.resolve() for r in roots if (r/"src"/"lumpy_forecasting.py").exists())
sys.path.insert(0,str(project_root/"src")) if str(project_root/"src") not in sys.path else None
for module in (lf,bh,opt,ob,cr,tm): importlib.reload(module)

OUT=project_root/"results"/"lumpy_02_forecasting/stages/occasional_memory"; CKPT=OUT/"checkpoints"; DOC=project_root/"docs"/"lumpy_02_forecasting/stages/occasional_memory"
for path in (OUT,CKPT,DOC): path.mkdir(parents=True,exist_ok=True)
CONFIG=lf.LumpyConfig(variant="all_sku_history",train_months=48,gap_months=3,test_months=18,step_months=3,min_train_months=18,max_folds=6,external_mode="off")

sales,external=lf.load_lumpy_inputs(project_root,CONFIG); model_data,_=lf.build_lumpy_model_frame(sales,external,CONFIG)
assignments=pd.read_csv(project_root/"docs/lumpy_02_forecasting/stages/chronology_strategy/chronology_strategy_strategy_assignment_all_690.csv")
focus=assignments.loc[assignments.lifecycle_tier.eq("established")&assignments.frequency_tier.eq("occasional_2_3")&assignments.abc_units_tier.isin(["A","B","C"])].copy(); focus_ids=set(focus.sku_id); assert len(focus_ids)==116
established_ids=set(assignments.loc[assignments.lifecycle_tier.eq("established"),"sku_id"])
occasional_data=model_data.loc[model_data.sku_id.isin(focus_ids)].copy(); established_data=model_data.loc[model_data.sku_id.isin(established_ids)].copy()
cohorts={f"occasional_{tier.lower()}":set(group.sku_id) for tier,group in focus.groupby("abc_units_tier")}; cohort_map=pd.concat([pd.DataFrame({"sku_id":list(ids),"cohort":cohort}) for cohort,ids in cohorts.items()],ignore_index=True)
n11=pd.read_csv(project_root/"results/lumpy_02_forecasting/stages/core_block/core_block_block_backtest_forecasts.csv",parse_dates=["train_start","train_end","test_start","test_end"]); final_job=n11.loc[n11.policy.eq("required_18m_gap3"),["train_start","train_end","test_start","test_end"]].drop_duplicates().sort_values("test_end").iloc[-1].to_dict()
jobs={
 "h1":{"train_start":pd.Timestamp("2021-01-01"),"train_end":pd.Timestamp("2022-06-01"),"test_start":pd.Timestamp("2022-10-01"),"test_end":pd.Timestamp("2024-03-01")},
 "h2":{"train_start":pd.Timestamp("2021-01-01"),"train_end":pd.Timestamp("2022-08-01"),"test_start":pd.Timestamp("2022-12-01"),"test_end":pd.Timestamp("2024-05-01")},
 "h3":{"train_start":pd.Timestamp("2021-01-01"),"train_end":pd.Timestamp("2022-10-01"),"test_start":pd.Timestamp("2023-02-01"),"test_end":pd.Timestamp("2024-07-01")},
 "final":final_job,
}
display(pd.DataFrame([{"horizon_id":key,**value} for key,value in jobs.items()]))

library=pd.read_csv(project_root/"docs/lumpy_02_forecasting/stages/occasional_router/occasional_router_candidate_library.csv").candidate_id.tolist()
trial_ids=sorted({cid.split("__cal_")[0] for cid in library}); specs=[cr.parse_candidate_id(cid,trial_ids) for cid in library]
spec_lookup={spec.key:spec for spec in opt.STRUCTURAL_SPECS}
def trial_parts(trial_id):
    scope="occasional_pool" if trial_id.startswith("occasional_pool__") else "established_transfer"
    spec_key=trial_id[len(scope)+2:].rsplit("__w24",1)[0]
    return scope,spec_key
components=[]; started=time.perf_counter()
for horizon_id,job in jobs.items():
  if horizon_id in ("h3","final"):
    old_id="validation" if horizon_id=="h3" else "final"
    for scope in ("occasional_pool","established_transfer"):
      old=pd.read_pickle(project_root/"results/lumpy_02_forecasting/stages/occasional_baseline/checkpoints_v1_h18"/f"{old_id}__{scope}__structural.pkl")
      new=pd.read_pickle(project_root/"results/lumpy_02_forecasting/stages/occasional_hurdle/checkpoints"/f"{old_id}__{scope}__new.pkl")
      part=pd.concat([old.loc[old.architecture.eq("hurdle")],new],ignore_index=True).drop_duplicates(["trial_id","sku_id","block_number"]); part=part.loc[part.trial_id.isin(trial_ids)].copy(); part["horizon_id"]=horizon_id; components.append(part)
    continue
  for scope,source in (("occasional_pool",occasional_data),("established_transfer",established_data)):
    required=[trial for trial in trial_ids if trial_parts(trial)[0]==scope]; cache=CKPT/f"{horizon_id}__{scope}.pkl"
    if cache.exists(): part=pd.read_pickle(cache)
    else:
      train_start=max(pd.Timestamp(source.month.min()),pd.Timestamp(job["train_end"])-pd.DateOffset(months=23)); train=source.loc[source.month.between(train_start,job["train_end"])].copy(); test=source.loc[source.month.between(job["test_start"],job["test_end"])].copy(); prepared=bh.prepare_fold(train,test,lf.SKU_COLUMN,lf.MONTH_COLUMN,lf.TARGET_COLUMN,3,18)
      fitted=[]
      for trial_id in required:
        _,spec_key=trial_parts(trial_id); fitted.append(opt.fit_structural_components(prepared,spec_lookup[spec_key],trial_id,CONFIG.random_state)); print("fitted",horizon_id,trial_id,flush=True)
      part=pd.concat(fitted,ignore_index=True); part.to_pickle(cache)
    part["horizon_id"]=horizon_id; components.append(part)
structural=pd.concat(components,ignore_index=True)
for horizon_id,job in jobs.items():
    size=ob.historical_size_table(occasional_data.loc[occasional_data.month.le(job["train_end"])],focus_ids,24); mask=structural.horizon_id.eq(horizon_id)
    structural.loc[mask,["historical_positive_median","historical_positive_mean","event_budget"]]=structural.loc[mask,["sku_id"]].merge(size,on="sku_id",how="left")[["historical_positive_median","historical_positive_mean","event_budget"]].to_numpy()
structural[["historical_positive_median","historical_positive_mean"]]=structural[["historical_positive_median","historical_positive_mean"]].fillna(0.0); structural["event_budget"]=structural.event_budget.fillna(2).astype(int)
assert structural.groupby(["horizon_id","trial_id","sku_id"]).block_number.nunique().eq(6).all(); print("Components complete in",round((time.perf_counter()-started)/60,1),"minutes")

parts_by_horizon={h:{trial:g.sort_values(["sku_id","block_start"]).reset_index(drop=True) for trial,g in structural.loc[structural.horizon_id.eq(h)].groupby("trial_id")} for h in jobs}
probability_cache={}
for horizon_id in ("h1","h2","h3"):
  for trial_id,calibration in sorted({(spec["trial_id"],spec["calibration"]) for spec in specs}): probability_cache[(horizon_id,trial_id,calibration)]=ob.cross_fitted_calibration(parts_by_horizon[horizon_id][trial_id],parts_by_horizon[horizon_id][trial_id],calibration)[0]
for trial_id,calibration in sorted({(spec["trial_id"],spec["calibration"]) for spec in specs}): probability_cache[("final",trial_id,calibration)]=ob.cross_fitted_calibration(parts_by_horizon["h3"][trial_id],parts_by_horizon["final"][trial_id],calibration)[1]
def build_horizon(horizon_id):
  output=[]
  for spec in specs:
    frame=parts_by_horizon[horizon_id][spec["trial_id"]]; probability=probability_cache[(horizon_id,spec["trial_id"],spec["calibration"])]; size=ob.positive_size(frame,spec["size_source"]); forecast=ob.compose_event_forecast(frame,probability,size,spec["mode"],spec["event_count"],spec["scale"])
    candidate=frame[["sku_id","block_start","block_number","target","block_naive_scale"]].copy(); candidate["forecast"]=forecast; candidate["candidate_id"]=spec["candidate_id"]; candidate["horizon_id"]=horizon_id; candidate=candidate.loc[candidate.sku_id.isin(focus_ids)].copy(); output.append(candidate)
  return pd.concat(output,ignore_index=True)
forecasts={h:build_horizon(h) for h in jobs}; historical_forecasts=pd.concat([forecasts[h] for h in ("h1","h2","h3")],ignore_index=True)
errors=pd.concat([cr.sku_candidate_errors(forecasts[h]).assign(horizon_id=h) for h in ("h1","h2","h3")],ignore_index=True).merge(cohort_map,on="sku_id",how="left")
print("Historical candidate rows:",len(historical_forecasts),"| SKU-candidate errors:",len(errors))

rules=[]
for score_mode,decay,individual_weight in itertools.product(("wmape","rank"),(.50,.75,1.0),(0.0,.25,.50,.75,1.0)):
    rules.append({"rule_id":f"{score_mode}__d{decay:.2f}__i{individual_weight:.2f}","score_mode":score_mode,"decay":decay,"individual_weight":individual_weight,"eligible":individual_weight>0})
walk_rows=[]; selections={}
for evaluation_horizon,history_ids in (("h2",["h1"]),("h3",["h1","h2"])):
  history=errors.loc[errors.horizon_id.isin(history_ids)]
  for rule in rules:
    selected=tm.temporal_selections(history,cohort_map,history_ids,rule["score_mode"],rule["decay"],rule["individual_weight"]); selections[(evaluation_horizon,rule["rule_id"])]=selected
    routed=cr.selected_forecasts(forecasts[evaluation_horizon],selected)
    for cohort,ids in cohorts.items(): _,summary=opt.score_forecast(routed.loc[routed.sku_id.isin(ids)]); walk_rows.append({"evaluation_horizon":evaluation_horizon,"cohort":cohort,**rule,**summary})
walk=pd.DataFrame(walk_rows); walk.to_csv(DOC/"occasional_memory_walk_forward_validation.csv",index=False)
aggregate=walk.groupby(["cohort","rule_id","score_mode","decay","individual_weight","eligible"],as_index=False).agg(total_under_50=("under_50","sum"),total_under_70=("under_70","sum"),total_under_100=("under_100","sum"),mean_median_wmape=("median_wmape","mean"),mean_portfolio_wmape=("portfolio_wmape","mean"),mean_abs_bias=("bias_pct",lambda x:x.abs().mean()))
aggregate=aggregate.sort_values(["cohort","total_under_70","total_under_50","total_under_100","mean_median_wmape","mean_portfolio_wmape","mean_abs_bias","rule_id"],ascending=[True,False,False,False,True,True,True,True]); aggregate.to_csv(DOC/"occasional_memory_memory_rule_summary.csv",index=False)
display(aggregate.groupby("cohort").head(8))

persistence=tm.persistence_table(errors); persistence.to_csv(DOC/"occasional_memory_candidate_persistence.csv",index=False); display(persistence)
locks=[]
for cohort in cohorts:
    table=aggregate.loc[aggregate.cohort.eq(cohort)]; challenger=table.loc[table.eligible].iloc[0]; baseline=table.loc[~table.eligible].iloc[0]
    accepted=(challenger.total_under_70>baseline.total_under_70) or (challenger.total_under_70==baseline.total_under_70 and challenger.total_under_50>baseline.total_under_50)
    locks.append({"cohort":cohort,"locked_source":"temporal_sku_memory" if accepted else "cohort_memory","rule_id":challenger.rule_id if accepted else baseline.rule_id,"score_mode":challenger.score_mode if accepted else baseline.score_mode,"decay":float(challenger.decay if accepted else baseline.decay),"individual_weight":float(challenger.individual_weight if accepted else baseline.individual_weight),"challenger_total_under_50":int(challenger.total_under_50),"challenger_total_under_70":int(challenger.total_under_70),"baseline_total_under_50":int(baseline.total_under_50),"baseline_total_under_70":int(baseline.total_under_70)})
locks=pd.DataFrame(locks); locks.to_json(DOC/"occasional_memory_locked_memory_rules.json",orient="records",indent=2); display(locks)

recommended=[]
for lock in locks.to_dict("records"):
    selected=tm.temporal_selections(errors,cohort_map,["h1","h2","h3"],lock["score_mode"],lock["decay"],lock["individual_weight"]); frame=cr.selected_forecasts(forecasts["final"],selected); frame=frame.loc[frame.sku_id.isin(cohorts[lock["cohort"]])].copy(); frame["cohort"]=lock["cohort"]; frame["memory_rule"]=lock["rule_id"]; frame["locked_before_final"]=True; recommended.append(frame)
recommended=pd.concat(recommended,ignore_index=True); assert recommended.groupby("sku_id").block_number.nunique().eq(6).all(); recommended.to_csv(OUT/"occasional_memory_temporal_memory_forecasts.csv",index=False)
sku_tables=[]; summaries=[]
for cohort,group in recommended.groupby("cohort"):
    sku,summary=opt.score_forecast(group); sku["cohort"]=cohort; sku_tables.append(sku); summaries.append({"cohort":cohort,**summary})
sku_all,summary_all=opt.score_forecast(recommended); sku_results=pd.concat(sku_tables,ignore_index=True); summary=pd.DataFrame(summaries+[{"cohort":"all_occasional",**summary_all}])
for threshold in (50,70,100): summary[f"pct_below_{threshold}"]=100*summary[f"under_{threshold}"]/summary.positive_skus
sku_results.to_csv(DOC/"occasional_memory_individual_sku_results.csv",index=False); summary.to_csv(DOC/"occasional_memory_official_summary.csv",index=False)
display(summary[["cohort","all_skus","positive_skus","under_50","pct_below_50","under_70","pct_below_70","under_100","pct_below_100","median_wmape","portfolio_wmape","bias_pct"]])

plot=recommended.groupby(["cohort","block_start"],as_index=False).agg(actual=("target","sum"),forecast=("forecast","sum")); fig,axes=plt.subplots(2,2,figsize=(14,9)); panels=list(plot.groupby("cohort"))+[("all_occasional",recommended.groupby("block_start",as_index=False).agg(actual=("target","sum"),forecast=("forecast","sum")))]
for ax,(cohort,group) in zip(axes.ravel(),panels): ax.plot(group.block_start,group.actual,marker="o",linewidth=2,label="Actual"); ax.plot(group.block_start,group.forecast,marker="o",linewidth=2,label="Forecast"); ax.set_title(cohort.replace("_"," ").title()); ax.set_xlabel("3-month block start"); ax.set_ylabel("Demand"); ax.grid(alpha=.25); ax.legend(); ax.tick_params(axis="x",rotation=30)
fig.suptitle("Established Occasional: Temporal Model Memory"); fig.tight_layout(); fig.savefig(DOC/"occasional_memory_actual_vs_forecast.png",dpi=160,bbox_inches="tight"); plt.show(); print("Complete. Memory rules were selected through two historical walk-forward transfers before final evaluation.")


## 14. Rare-demand event model

Treats rare demand mainly as an event-prediction problem. It estimates whether an event will occur, where it will occur and how large it may be.


In [ ]:
lf = lumpy_forecasting
bh = lumpy_block_hybrid
opt = lumpy_ab_optimization
ob = lumpy_occurrence_budget
rare = lumpy_rare_event

roots=[Path.cwd(),Path.cwd().parent,Path(r"D:/Lumpy_Fellas-/Inchscape Repo")]
project_root=next(r.resolve() for r in roots if (r/"src"/"lumpy_forecasting.py").exists())
sys.path.insert(0,str(project_root/"src")) if str(project_root/"src") not in sys.path else None
for module in (lf,bh,opt,ob,rare): importlib.reload(module)

OUT=project_root/"results"/"lumpy_02_forecasting/stages/rare_baseline"; CKPT=OUT/"checkpoints"; DOC=project_root/"docs"/"lumpy_02_forecasting/stages/rare_baseline"
for path in (OUT,CKPT,DOC): path.mkdir(parents=True,exist_ok=True)
CONFIG=lf.LumpyConfig(variant="all_sku_history",train_months=48,gap_months=3,test_months=18,step_months=3,min_train_months=18,max_folds=6,external_mode="off")
SPECS=("hurdle_d1_sqrt_log","hurdle_d2_sqrt_log","hurdle_d2_full_log","hurdle_d2_sqrt_tweedie")
spec_lookup={spec.key:spec for spec in opt.STRUCTURAL_SPECS}
CALIBRATIONS=("raw","sigmoid","isotonic"); SIZE_SOURCES=("ml","median","blend50_mean"); RECIPES=rare.rare_recipe_grid().loc[lambda d:~d['mode'].eq('zero')]

sales,external=lf.load_lumpy_inputs(project_root,CONFIG); model_data,_=lf.build_lumpy_model_frame(sales,external,CONFIG)
assignments=pd.read_csv(project_root/"docs/lumpy_02_forecasting/stages/chronology_strategy/chronology_strategy_strategy_assignment_all_690.csv")
focus=assignments.loc[assignments.lifecycle_tier.eq("established")&assignments.frequency_tier.eq("rare_0_1")&assignments.abc_units_tier.isin(["A","B","C"])].copy(); focus_ids=set(focus.sku_id); assert len(focus_ids)==56
established_ids=set(assignments.loc[assignments.lifecycle_tier.eq("established"),"sku_id"])
rare_data=model_data.loc[model_data.sku_id.isin(focus_ids)].copy(); established_data=model_data.loc[model_data.sku_id.isin(established_ids)].copy()
cohorts={f"rare_{tier.lower()}":set(group.sku_id) for tier,group in focus.groupby("abc_units_tier")}
strict_job={"train_start":pd.Timestamp("2021-01-01"),"train_end":pd.Timestamp("2022-10-01"),"test_start":pd.Timestamp("2023-02-01"),"test_end":pd.Timestamp("2024-07-01")}
n11=pd.read_csv(project_root/"results/lumpy_02_forecasting/stages/core_block/core_block_block_backtest_forecasts.csv",parse_dates=["train_start","train_end","test_start","test_end"]); final_job=n11.loc[n11.policy.eq("required_18m_gap3"),["train_start","train_end","test_start","test_end"]].drop_duplicates().sort_values("test_end").iloc[-1].to_dict(); jobs={"validation":strict_job,"final":final_job}
display(focus.groupby("abc_units_tier").sku_id.nunique().to_frame("skus")); display(pd.DataFrame([{"role":key,**value} for key,value in jobs.items()]))

components=[]; classical={}; prepared_rare={}; started=time.perf_counter()
for job_id,job in jobs.items():
  train_start=max(pd.Timestamp(rare_data.month.min()),pd.Timestamp(job["train_end"])-pd.DateOffset(months=23)); train=rare_data.loc[rare_data.month.between(train_start,job["train_end"])].copy(); test=rare_data.loc[rare_data.month.between(job["test_start"],job["test_end"])].copy(); prepared=bh.prepare_fold(train,test,lf.SKU_COLUMN,lf.MONTH_COLUMN,lf.TARGET_COLUMN,3,18); prepared_rare[job_id]=prepared
  cache=CKPT/f"{job_id}__rare_pool.pkl"
  if cache.exists(): rare_part=pd.read_pickle(cache)
  else:
    fitted=[]
    for spec_key in SPECS:
      trial_id=f"rare_pool__{spec_key}__w24"; fitted.append(opt.fit_structural_components(prepared,spec_lookup[spec_key],trial_id,CONFIG.random_state)); print("fitted",job_id,trial_id,flush=True)
    rare_part=pd.concat(fitted,ignore_index=True); rare_part.to_pickle(cache)
  rare_part["job_id"]=job_id; components.append(rare_part)
  old=pd.read_pickle(project_root/"results/lumpy_02_forecasting/stages/occasional_baseline/checkpoints_v1_h18"/f"{job_id}__established_transfer__structural.pkl"); new=pd.read_pickle(project_root/"results/lumpy_02_forecasting/stages/occasional_hurdle/checkpoints"/f"{job_id}__established_transfer__new.pkl")
  transfer=pd.concat([old.loc[old.architecture.eq("hurdle")],new],ignore_index=True); transfer=transfer.loc[transfer.trial_id.str.contains("|".join(SPECS),regex=True)&transfer.sku_id.isin(focus_ids)].drop_duplicates(["trial_id","sku_id","block_number"]).copy(); transfer["job_id"]=job_id; components.append(transfer)
  classical[job_id]=opt.classical_component_rows(train,prepared,24)
structural=pd.concat(components,ignore_index=True)
for job_id,job in jobs.items():
  size=ob.historical_size_table(rare_data.loc[rare_data.month.le(job["train_end"])],focus_ids,24); mask=structural.job_id.eq(job_id)
  structural.loc[mask,["historical_positive_median","historical_positive_mean","event_budget"]]=structural.loc[mask,["sku_id"]].merge(size,on="sku_id",how="left")[["historical_positive_median","historical_positive_mean","event_budget"]].to_numpy()
structural[["historical_positive_median","historical_positive_mean"]]=structural[["historical_positive_median","historical_positive_mean"]].fillna(0.0); structural["event_budget"]=structural.event_budget.fillna(1).astype(int)
assert structural.groupby(["job_id","trial_id","sku_id"]).block_number.nunique().eq(6).all(); print("Component stage complete in",round((time.perf_counter()-started)/60,1),"minutes")

validation_parts={trial:g.sort_values(["sku_id","block_start"]).reset_index(drop=True) for trial,g in structural.loc[structural.job_id.eq("validation")].groupby("trial_id")}; final_parts={trial:g.sort_values(["sku_id","block_start"]).reset_index(drop=True) for trial,g in structural.loc[structural.job_id.eq("final")].groupby("trial_id")}
calibration_cache={}
for trial_id in validation_parts:
  for calibration in CALIBRATIONS: calibration_cache[(trial_id,calibration)]=ob.cross_fitted_calibration(validation_parts[trial_id],final_parts[trial_id],calibration)
candidate_specs={"no_event":{"kind":"zero"}}; validation_rows=[]
base=next(iter(validation_parts.values()))[["sku_id","block_start","block_number","target","block_naive_scale"]].copy(); base["forecast"]=0.0
for cohort,ids in cohorts.items(): _,summary=opt.score_forecast(base.loc[base.sku_id.isin(ids)]); validation_rows.append({"cohort":cohort,"candidate_id":"no_event","architecture":"zero",**summary})
started=time.perf_counter(); trials=sorted(validation_parts)
for number,trial_id in enumerate(trials,1):
  validation=validation_parts[trial_id]
  for calibration,size_source,recipe in itertools.product(CALIBRATIONS,SIZE_SOURCES,RECIPES.to_dict("records")):
    probability=calibration_cache[(trial_id,calibration)][0]; size=ob.positive_size(validation,size_source); forecast=rare.compose_rare_forecast(validation,probability,size,recipe["mode"],recipe["scale"],recipe["horizon_threshold"])
    cid=f"{trial_id}__cal_{calibration}__size_{size_source}__{recipe['recipe_id']}"; candidate_specs[cid]={"kind":"structural","trial_id":trial_id,"calibration":calibration,"size_source":size_source,**recipe}
    frame=validation[["sku_id","block_start","block_number","target","block_naive_scale"]].copy(); frame["forecast"]=forecast
    for cohort,ids in cohorts.items(): _,summary=opt.score_forecast(frame.loc[frame.sku_id.isin(ids)]); validation_rows.append({"cohort":cohort,"candidate_id":cid,"architecture":"single_event_hurdle",**summary})
  print(f"[{number}/{len(trials)}] {trial_id}",flush=True)
for trial_id,g in classical["validation"].groupby("trial_id"):
  for scale in (.5,.75,1.0,1.25):
    frame=opt.compose_classical(g,scale); cid=frame.candidate_id.iloc[0]; candidate_specs[cid]={"kind":"classical","trial_id":trial_id,"scale":scale}
    for cohort,ids in cohorts.items(): _,summary=opt.score_forecast(frame.loc[frame.sku_id.isin(ids)]); validation_rows.append({"cohort":cohort,"candidate_id":cid,"architecture":"classical",**summary})
validation_summary=pd.DataFrame(validation_rows); validation_summary.to_csv(DOC/"rare_baseline_validation_candidates.csv",index=False); print("Tournament complete in",round((time.perf_counter()-started)/60,1),"minutes")
display(pd.concat([opt.rank_summary(group).head(8) for _,group in validation_summary.groupby("cohort")])[["cohort","architecture","candidate_id","under_50","under_70","under_100","median_wmape","portfolio_wmape","bias_pct"]])

locks=[]
for cohort,group in validation_summary.groupby("cohort"):
  winner=opt.rank_summary(group).iloc[0]; locks.append({"cohort":cohort,"candidate_id":winner.candidate_id,"architecture":winner.architecture,"validation_under_50":int(winner.under_50),"validation_under_70":int(winner.under_70),"validation_under_100":int(winner.under_100),"validation_median_wmape":float(winner.median_wmape),"locked_before_final":True})
locks=pd.DataFrame(locks); locks.to_json(DOC/"rare_baseline_locked_strategies.json",orient="records",indent=2); display(locks)

def final_candidate(candidate_id):
  spec=candidate_specs[candidate_id]
  if spec["kind"]=="zero": frame=next(iter(final_parts.values()))[["sku_id","block_start","block_number","target","block_naive_scale"]].copy(); frame["forecast"]=0.0; return frame
  if spec["kind"]=="classical": return opt.compose_classical(classical["final"].loc[classical["final"].trial_id.eq(spec["trial_id"])],spec["scale"])
  frame=final_parts[spec["trial_id"]]; probability=calibration_cache[(spec["trial_id"],spec["calibration"])][1]; size=ob.positive_size(frame,spec["size_source"]); forecast=rare.compose_rare_forecast(frame,probability,size,spec["mode"],spec["scale"],spec["horizon_threshold"]); output=frame[["sku_id","block_start","block_number","target","block_naive_scale"]].copy(); output["forecast"]=forecast; return output
recommended=[]
for lock in locks.to_dict("records"):
  frame=final_candidate(lock["candidate_id"]); frame=frame.loc[frame.sku_id.isin(cohorts[lock["cohort"]])].copy(); frame["cohort"]=lock["cohort"]; frame["candidate_id"]=lock["candidate_id"]; frame["locked_before_final"]=True; recommended.append(frame)
recommended=pd.concat(recommended,ignore_index=True); assert recommended.groupby("sku_id").block_number.nunique().eq(6).all(); recommended.to_csv(OUT/"rare_baseline_official_rare_forecasts.csv",index=False)
sku_tables=[]; summaries=[]; diagnostics=[]
for cohort,group in recommended.groupby("cohort"):
  sku,summary=opt.score_forecast(group); sku["cohort"]=cohort; sku_tables.append(sku); summaries.append({"cohort":cohort,**summary}); diagnostics.append({"cohort":cohort,**rare.occurrence_diagnostics(group)})
sku_all,summary_all=opt.score_forecast(recommended); sku_results=pd.concat(sku_tables,ignore_index=True); summary=pd.DataFrame(summaries+[{"cohort":"all_rare",**summary_all}]); diagnostics=pd.DataFrame(diagnostics+[{"cohort":"all_rare",**rare.occurrence_diagnostics(recommended)}])
for threshold in (50,70,100): summary[f"pct_below_{threshold}"]=100*summary[f"under_{threshold}"]/summary.positive_skus
sku_results.to_csv(DOC/"rare_baseline_individual_sku_results.csv",index=False); summary.to_csv(DOC/"rare_baseline_official_summary.csv",index=False); diagnostics.to_csv(DOC/"rare_baseline_event_diagnostics.csv",index=False)
display(summary[["cohort","all_skus","positive_skus","under_50","pct_below_50","under_70","pct_below_70","under_100","pct_below_100","median_wmape","portfolio_wmape","bias_pct"]]); display(diagnostics)

plot=recommended.groupby(["cohort","block_start"],as_index=False).agg(actual=("target","sum"),forecast=("forecast","sum")); fig,axes=plt.subplots(2,2,figsize=(14,9)); panels=list(plot.groupby("cohort"))+[("all_rare",recommended.groupby("block_start",as_index=False).agg(actual=("target","sum"),forecast=("forecast","sum")))]
for ax,(cohort,group) in zip(axes.ravel(),panels): ax.plot(group.block_start,group.actual,marker="o",linewidth=2,label="Actual"); ax.plot(group.block_start,group.forecast,marker="o",linewidth=2,label="Forecast"); ax.set_title(cohort.replace("_"," ").title()); ax.set_xlabel("3-month block start"); ax.set_ylabel("Demand"); ax.grid(alpha=.25); ax.legend(); ax.tick_params(axis="x",rotation=30)
fig.suptitle("Established Rare: Single-Event Hurdle"); fig.tight_layout(); fig.savefig(DOC/"rare_baseline_actual_vs_forecast.png",dpi=160,bbox_inches="tight"); plt.show(); print("Complete. Rare-event strategies were locked on historical validation before final evaluation.")


## 15. Rare group occurrence priors

Adds family and subfamily demand timing to the rare-SKU process. This helps when an individual SKU does not have enough events to establish its own timing pattern.


In [ ]:
lf = lumpy_forecasting
opt = lumpy_ab_optimization
ob = lumpy_occurrence_budget
rare = lumpy_rare_event
group = lumpy_group_occurrence

roots=[Path.cwd(),Path.cwd().parent,Path(r"D:/Lumpy_Fellas-/Inchscape Repo")]
project_root=next(r.resolve() for r in roots if (r/"src"/"lumpy_forecasting.py").exists())
sys.path.insert(0,str(project_root/"src")) if str(project_root/"src") not in sys.path else None
for module in (lf,opt,ob,rare,group): importlib.reload(module)

OUT=project_root/"results"/"lumpy_02_forecasting/stages/rare_group_occurrence"; DOC=project_root/"docs"/"lumpy_02_forecasting/stages/rare_group_occurrence"
for path in (OUT,DOC): path.mkdir(parents=True,exist_ok=True)
CONFIG=lf.LumpyConfig(variant="all_sku_history",train_months=48,gap_months=3,test_months=18,step_months=3,min_train_months=18,max_folds=6,external_mode="off")
SPECS=("hurdle_d1_sqrt_log","hurdle_d2_sqrt_log","hurdle_d2_full_log","hurdle_d2_sqrt_tweedie")
CALIBRATIONS=("raw","sigmoid","isotonic"); SIZE_SOURCES=("ml","median","blend50_mean"); PRIOR_TYPES=("family","subfamily","blend"); POWERS=(.5,1.0,2.0); MODES=("top1_expected","top1_full","top2_expected","top2_full"); SCALES=(.5,.75,1.0)

sales,external=lf.load_lumpy_inputs(project_root,CONFIG); model_data,_=lf.build_lumpy_model_frame(sales,external,CONFIG)
assignments=pd.read_csv(project_root/"docs/lumpy_02_forecasting/stages/chronology_strategy/chronology_strategy_strategy_assignment_all_690.csv")
focus=assignments.loc[assignments.lifecycle_tier.eq("established")&assignments.frequency_tier.eq("rare_0_1")&assignments.abc_units_tier.isin(["A","B","C"])].copy(); focus_ids=set(focus.sku_id); assert len(focus_ids)==56
rare_data=model_data.loc[model_data.sku_id.isin(focus_ids)].copy(); cohorts={f"rare_{tier.lower()}":set(rows.sku_id) for tier,rows in focus.groupby("abc_units_tier")}
strict_job={"train_start":pd.Timestamp("2021-01-01"),"train_end":pd.Timestamp("2022-10-01"),"test_start":pd.Timestamp("2023-02-01"),"test_end":pd.Timestamp("2024-07-01")}
n11=pd.read_csv(project_root/"results/lumpy_02_forecasting/stages/core_block/core_block_block_backtest_forecasts.csv",parse_dates=["train_start","train_end","test_start","test_end"]); final_job=n11.loc[n11.policy.eq("required_18m_gap3"),["train_start","train_end","test_start","test_end"]].drop_duplicates().sort_values("test_end").iloc[-1].to_dict(); jobs={"validation":strict_job,"final":final_job}

parts=[]
for job_id,job in jobs.items():
  old=pd.read_pickle(project_root/"results/lumpy_02_forecasting/stages/occasional_baseline/checkpoints_v1_h18"/f"{job_id}__established_transfer__structural.pkl"); new=pd.read_pickle(project_root/"results/lumpy_02_forecasting/stages/occasional_hurdle/checkpoints"/f"{job_id}__established_transfer__new.pkl")
  part=pd.concat([old.loc[old.architecture.eq("hurdle")],new],ignore_index=True); part=part.loc[part.trial_id.str.contains("|".join(SPECS),regex=True)&part.sku_id.isin(focus_ids)].drop_duplicates(["trial_id","sku_id","block_number"]).copy(); part["job_id"]=job_id; parts.append(part)
structural=pd.concat(parts,ignore_index=True)
for job_id,job in jobs.items():
  size=ob.historical_size_table(rare_data.loc[rare_data.month.le(job["train_end"])],focus_ids,24); mask=structural.job_id.eq(job_id)
  structural.loc[mask,["historical_positive_median","historical_positive_mean","event_budget"]]=structural.loc[mask,["sku_id"]].merge(size,on="sku_id",how="left")[["historical_positive_median","historical_positive_mean","event_budget"]].to_numpy()
structural[["historical_positive_median","historical_positive_mean"]]=structural[["historical_positive_median","historical_positive_mean"]].fillna(0.0); structural["event_budget"]=structural.event_budget.fillna(1).astype(int)
validation_parts={trial:g.sort_values(["sku_id","block_start"]).reset_index(drop=True) for trial,g in structural.loc[structural.job_id.eq("validation")].groupby("trial_id")}; final_parts={trial:g.sort_values(["sku_id","block_start"]).reset_index(drop=True) for trial,g in structural.loc[structural.job_id.eq("final")].groupby("trial_id")}
calibration_cache={(trial,cal):ob.cross_fitted_calibration(validation_parts[trial],final_parts[trial],cal) for trial in validation_parts for cal in CALIBRATIONS}

prior_tables={}
for job_id,job in jobs.items():
  sample=next(iter(validation_parts.values())) if job_id=="validation" else next(iter(final_parts.values())); starts=sample.block_start.drop_duplicates().sort_values().tolist(); train=rare_data.loc[rare_data.month.le(job["train_end"])]
  family=group.group_block_priors(train,focus_ids,starts,"FAMILY_DESCRIPTION").rename(columns={"group_factor":"family_factor"}); subfamily=group.group_block_priors(train,focus_ids,starts,"SUBFAMILY_DESCRIPTION").rename(columns={"group_factor":"subfamily_factor"})
  priors=family.merge(subfamily,on=["sku_id","block_number","block_start"],how="outer"); priors["blend_factor"]=np.sqrt(priors.family_factor.fillna(1.0)*priors.subfamily_factor.fillna(1.0)); prior_tables[job_id]=priors
display(prior_tables["validation"][["family_factor","subfamily_factor","blend_factor"]].describe())

n28_locks=pd.read_json(project_root/"docs/lumpy_02_forecasting/stages/rare_baseline/rare_baseline_locked_strategies.json")
trial_ids=sorted(validation_parts)
def parse_n28(candidate_id):
  trial=next(trial for trial in trial_ids if candidate_id.startswith(trial+"__")); pieces=candidate_id[len(trial)+2:].split("__"); return {"trial_id":trial,"calibration":pieces[0].replace("cal_",""),"size_source":pieces[1].replace("size_",""),"mode":pieces[2],"horizon_threshold":float(pieces[3][1:]),"scale":float(pieces[4][1:])}
def base_candidate(job_id,candidate_id):
  spec=parse_n28(candidate_id); source=validation_parts if job_id=="validation" else final_parts; index=0 if job_id=="validation" else 1; frame=source[spec["trial_id"]]; probability=calibration_cache[(spec["trial_id"],spec["calibration"])][index]; size=ob.positive_size(frame,spec["size_source"]); forecast=rare.compose_rare_forecast(frame,probability,size,spec["mode"],spec["scale"],spec["horizon_threshold"]); output=frame[["sku_id","block_start","block_number","target","block_naive_scale"]].copy(); output["forecast"]=forecast; return output
def incumbent(job_id):
  output=[]
  for lock in n28_locks.to_dict("records"):
    frame=base_candidate(job_id,lock["candidate_id"]); frame=frame.loc[frame.sku_id.isin(cohorts[lock["cohort"]])].copy(); frame["cohort"]=lock["cohort"]; output.append(frame)
  return pd.concat(output,ignore_index=True)
incumbent_validation=incumbent("validation"); incumbent_final=incumbent("final"); incumbent_scores={cohort:opt.score_forecast(rows)[1] for cohort,rows in incumbent_validation.groupby("cohort")}

validation_rows=[]; candidate_specs={}; started=time.perf_counter(); trials=sorted(validation_parts)
for number,trial_id in enumerate(trials,1):
  validation=validation_parts[trial_id]; aligned=validation.merge(prior_tables["validation"],on=["sku_id","block_number","block_start"],how="left")
  for calibration,size_source,prior_type,power,mode,scale in itertools.product(CALIBRATIONS,SIZE_SOURCES,PRIOR_TYPES,POWERS,MODES,SCALES):
    probability=calibration_cache[(trial_id,calibration)][0]; factor=aligned[f"{prior_type}_factor"].fillna(1.0).to_numpy(float); adjusted=group.adjust_probabilities(validation,probability,factor,power); size=ob.positive_size(validation,size_source); forecast=rare.compose_rare_forecast(validation,adjusted,size,mode,scale,0.0)
    cid=f"{trial_id}__cal_{calibration}__size_{size_source}__prior_{prior_type}_p{power:.2f}__{mode}__s{scale:.2f}"; candidate_specs[cid]={"trial_id":trial_id,"calibration":calibration,"size_source":size_source,"prior_type":prior_type,"power":power,"mode":mode,"scale":scale}
    frame=validation[["sku_id","block_start","block_number","target","block_naive_scale"]].copy(); frame["forecast"]=forecast
    for cohort,ids in cohorts.items(): _,summary=opt.score_forecast(frame.loc[frame.sku_id.isin(ids)]); validation_rows.append({"cohort":cohort,"candidate_id":cid,**summary})
  print(f"[{number}/{len(trials)}] {trial_id}",flush=True)
validation_summary=pd.DataFrame(validation_rows); validation_summary.to_csv(DOC/"rare_group_occurrence_validation_candidates.csv",index=False); print("Tournament complete in",round((time.perf_counter()-started)/60,1),"minutes")
display(pd.concat([opt.rank_summary(rows).head(8) for _,rows in validation_summary.groupby("cohort")])[["cohort","candidate_id","under_50","under_70","under_100","median_wmape","portfolio_wmape","bias_pct"]])

locks=[]
for cohort in cohorts:
  challenger=opt.rank_summary(validation_summary.loc[validation_summary.cohort.eq(cohort)]).iloc[0]; base=incumbent_scores[cohort]; accepted=(challenger.under_70>base["under_70"]) or (challenger.under_70==base["under_70"] and challenger.under_50>base["under_50"])
  locks.append({"cohort":cohort,"locked_source":"group_occurrence_prior" if accepted else "rare_baseline_incumbent","candidate_id":challenger.candidate_id,"challenger_under_50":int(challenger.under_50),"challenger_under_70":int(challenger.under_70),"challenger_under_100":int(challenger.under_100),"incumbent_under_50":int(base["under_50"]),"incumbent_under_70":int(base["under_70"]),"incumbent_under_100":int(base["under_100"])})
locks=pd.DataFrame(locks); locks.to_json(DOC/"rare_group_occurrence_locked_strategies.json",orient="records",indent=2); display(locks)
def final_challenger(candidate_id):
  spec=candidate_specs[candidate_id]; frame=final_parts[spec["trial_id"]]; aligned=frame.merge(prior_tables["final"],on=["sku_id","block_number","block_start"],how="left"); probability=calibration_cache[(spec["trial_id"],spec["calibration"])][1]; adjusted=group.adjust_probabilities(frame,probability,aligned[f"{spec['prior_type']}_factor"].fillna(1.0).to_numpy(float),spec["power"]); size=ob.positive_size(frame,spec["size_source"]); forecast=rare.compose_rare_forecast(frame,adjusted,size,spec["mode"],spec["scale"],0.0); output=frame[["sku_id","block_start","block_number","target","block_naive_scale"]].copy(); output["forecast"]=forecast; return output
recommended=[]
for lock in locks.to_dict("records"):
  ids=cohorts[lock["cohort"]]
  if lock["locked_source"]=="group_occurrence_prior": frame=final_challenger(lock["candidate_id"])
  else: frame=incumbent_final.copy()
  frame=frame.loc[frame.sku_id.isin(ids)].copy(); frame["cohort"]=lock["cohort"]; frame["locked_source"]=lock["locked_source"]; frame["locked_before_final"]=True; recommended.append(frame)
recommended=pd.concat(recommended,ignore_index=True); assert recommended.groupby("sku_id").block_number.nunique().eq(6).all(); recommended.to_csv(OUT/"rare_group_occurrence_official_rare_forecasts.csv",index=False)
sku_tables=[]; summaries=[]; diagnostics=[]
for cohort,rows in recommended.groupby("cohort"):
  sku,summary=opt.score_forecast(rows); sku["cohort"]=cohort; sku_tables.append(sku); summaries.append({"cohort":cohort,**summary}); diagnostics.append({"cohort":cohort,**rare.occurrence_diagnostics(rows)})
sku_all,summary_all=opt.score_forecast(recommended); summary=pd.DataFrame(summaries+[{"cohort":"all_rare",**summary_all}]); sku_results=pd.concat(sku_tables,ignore_index=True); diagnostics=pd.DataFrame(diagnostics+[{"cohort":"all_rare",**rare.occurrence_diagnostics(recommended)}])
for threshold in (50,70,100): summary[f"pct_below_{threshold}"]=100*summary[f"under_{threshold}"]/summary.positive_skus
summary.to_csv(DOC/"rare_group_occurrence_official_summary.csv",index=False); sku_results.to_csv(DOC/"rare_group_occurrence_individual_sku_results.csv",index=False); diagnostics.to_csv(DOC/"rare_group_occurrence_event_diagnostics.csv",index=False); display(summary); display(diagnostics)

plot=recommended.groupby(["cohort","block_start"],as_index=False).agg(actual=("target","sum"),forecast=("forecast","sum")); fig,axes=plt.subplots(2,2,figsize=(14,9)); panels=list(plot.groupby("cohort"))+[("all_rare",recommended.groupby("block_start",as_index=False).agg(actual=("target","sum"),forecast=("forecast","sum")))]
for ax,(cohort,rows) in zip(axes.ravel(),panels): ax.plot(rows.block_start,rows.actual,marker="o",linewidth=2,label="Actual"); ax.plot(rows.block_start,rows.forecast,marker="o",linewidth=2,label="Forecast"); ax.set_title(cohort.replace("_"," ").title()); ax.grid(alpha=.25); ax.legend(); ax.tick_params(axis="x",rotation=30)
fig.suptitle("Established Rare: Group Occurrence Priors"); fig.tight_layout(); fig.savefig(DOC/"rare_group_occurrence_actual_vs_forecast.png",dpi=160,bbox_inches="tight"); plt.show(); print("Complete. Group-prior challengers were locked before final evaluation.")


## 16. Cold-start launch hurdle

Uses genuine historical product launches as training examples. It finds similar launches, estimates demand occurrence and estimates positive demand size separately.


In [ ]:
lf = lumpy_forecasting
sr = lumpy_sku_router
launch = lumpy_launch_hurdle
score = lumpy_cold_start_calibration

roots=[Path.cwd(),Path.cwd().parent,Path(r'D:/Lumpy_Fellas-/Inchscape Repo')]; project_root=next(r.resolve() for r in roots if (r/'src/lumpy_forecasting.py').exists()); sys.path.insert(0,str(project_root/'src')) if str(project_root/'src') not in sys.path else None
for module in (lf,sr,launch,score): importlib.reload(module)
OUT=project_root/'results/lumpy_02_forecasting/stages/cold_start'; DOC=project_root/'docs/lumpy_02_forecasting/stages/cold_start'; OUT.mkdir(parents=True,exist_ok=True); DOC.mkdir(parents=True,exist_ok=True)
CONFIG=lf.LumpyConfig(variant='all_sku_history',train_months=48,gap_months=3,test_months=18,step_months=3,min_train_months=18,max_folds=6,external_mode='off')

sales,_=lf.load_lumpy_inputs(project_root,CONFIG); metadata=sr.extract_static_metadata(sales); first=sales.groupby('sku_id').month.min(); historical_ids=first.loc[first.ge(pd.Timestamp('2022-01-01')) & (first+pd.DateOffset(months=20)).le(pd.Timestamp('2024-10-01'))].index.tolist()
assignments=pd.read_csv(project_root/'docs/lumpy_02_forecasting/stages/chronology_strategy/chronology_strategy_strategy_assignment_all_690.csv'); official_ids=assignments.loc[assignments.lifecycle_tier.eq('cold_start'),'sku_id'].tolist(); assert len(official_ids)==40
historical_targets=launch.launch_targets(sales,historical_ids); historical_meta=metadata.loc[metadata.sku_id.isin(historical_ids)].copy(); official_targets=launch.launch_targets(sales,official_ids,fixed_test_start=pd.Timestamp('2024-11-01')); official_meta=metadata.loc[metadata.sku_id.isin(official_ids)].copy()
print(f'Eligible genuine historical launches: {len(historical_ids)}'); print(f'Official cold starts: {len(official_ids)}')

development=launch.analogue_hurdle_candidates(historical_targets,historical_meta,historical_targets,historical_meta,neighbour_counts=(3,5,10,20),thresholds=(0,.25,.5,.75),scales=(.25,.5,.75,1,1.25),leave_self_out=True)
validation=score.rank_candidates(score.candidate_summary(development)); winner=validation.iloc[0]; validation.to_csv(DOC/'cold_start_validation_candidates.csv',index=False); pd.DataFrame([winner]).to_csv(DOC/'cold_start_locked_strategy.csv',index=False); print('Locked:',winner.candidate_id); display(validation.head(15))

official_candidates=launch.analogue_hurdle_candidates(historical_targets,historical_meta,official_targets,official_meta,neighbour_counts=(3,5,10,20),thresholds=(0,.25,.5,.75),scales=(.25,.5,.75,1,1.25)); challenger=official_candidates.loc[official_candidates.candidate_id.eq(winner.candidate_id)].copy()
incumbent=pd.read_csv(project_root/'results/lumpy_02_forecasting/stages/sku_routing/sku_routing_real_cold_start_selected_forecasts.csv',parse_dates=['block_start']); incumbent['candidate_id']='SKU classification and cold-start candidates incumbent'; challenger['candidate_id']='Cold-start launch hurdle locked challenger'
comparison=pd.concat([score.candidate_summary(incumbent),score.candidate_summary(challenger)],ignore_index=True); comparison['pct_below_50']=100*comparison.under_50/comparison.positive_skus; comparison['pct_below_70']=100*comparison.under_70/comparison.positive_skus; comparison['pct_below_100']=100*comparison.under_100/comparison.positive_skus; comparison.to_csv(DOC/'cold_start_official_comparison.csv',index=False)
sku=[]
for label,frame in [('SKU classification and cold-start candidates incumbent',incumbent),('Cold-start launch hurdle locked challenger',challenger)]:
 s=frame.groupby('sku_id',as_index=False).agg(actual_total=('target','sum'),forecast_total=('forecast','sum')); e=frame.assign(error=(frame.target-frame.forecast).abs()).groupby('sku_id').error.sum(); s['absolute_error']=s.sku_id.map(e); s['wmape']=100*s.absolute_error/s.actual_total.replace(0,np.nan); s['strategy']=label; sku.append(s)
pd.concat(sku,ignore_index=True).to_csv(DOC/'cold_start_individual_sku_results.csv',index=False); challenger.to_csv(OUT/'cold_start_official_cold_start_forecasts.csv',index=False); display(comparison)

plot=pd.concat([incumbent.assign(strategy='SKU classification and cold-start candidates incumbent'),challenger.assign(strategy='Cold-start launch hurdle challenger')]).groupby(['strategy','block_start'],as_index=False).agg(actual=('target','sum'),forecast=('forecast','sum')); fig,axes=plt.subplots(1,2,figsize=(13,4),sharey=True)
for ax,(name,g) in zip(axes,plot.groupby('strategy',sort=False)): ax.plot(g.block_start,g.actual,color='black',marker='o',label='Actual'); ax.plot(g.block_start,g.forecast,color='#2878B5',marker='o',label='Forecast'); ax.set_title(name); ax.grid(alpha=.25); ax.legend()
axes[0].set_ylabel('Demand units'); fig.suptitle('Genuine cold starts: actual versus forecast'); fig.tight_layout(); fig.savefig(DOC/'cold_start_actual_vs_forecast.png',dpi=160,bbox_inches='tight'); plt.show()

print('Complete:',DOC/'cold_start_official_comparison.csv')


## 17. New-SKU short-history blend

Handles new SKUs that have some history, but not enough to be treated as established. It blends the SKU’s short demand history with demand from similar products.


In [ ]:
lf = lumpy_forecasting
sr = lumpy_sku_router
new = lumpy_new_sku_tournament

roots=[Path.cwd(),Path.cwd().parent,Path(r'D:/Lumpy_Fellas-/Inchscape Repo')]; project_root=next(r.resolve() for r in roots if (r/'src/lumpy_forecasting.py').exists()); sys.path.insert(0,str(project_root/'src')) if str(project_root/'src') not in sys.path else None
for module in (lf,sr,new): importlib.reload(module)
OUT=project_root/'results/lumpy_02_forecasting/stages/new_sku_blend'; DOC=project_root/'docs/lumpy_02_forecasting/stages/new_sku_blend'; OUT.mkdir(parents=True,exist_ok=True); DOC.mkdir(parents=True,exist_ok=True)
CONFIG=lf.LumpyConfig(variant='all_sku_history',train_months=48,gap_months=3,test_months=18,step_months=3,min_train_months=18,max_folds=6,external_mode='off'); POLICY='required_18m_gap3'

sales,external=lf.load_lumpy_inputs(project_root,CONFIG); model_data,_=lf.build_lumpy_model_frame(sales,external,CONFIG); metadata=sr.extract_static_metadata(sales); universe=sorted(model_data.sku_id.unique()); forecasts=pd.read_csv(project_root/'results/lumpy_02_forecasting/stages/core_block/core_block_block_backtest_forecasts.csv',parse_dates=['block_start','train_start','train_end','test_start','test_end']); forecasts=forecasts.loc[forecasts.policy.eq(POLICY)].copy(); inventory=forecasts[['fold_id','train_start','train_end','test_start','test_end']].drop_duplicates().sort_values('test_end'); holdout_fold=int(inventory.iloc[-1].fold_id); development_folds=inventory.loc[inventory.fold_id.ne(holdout_fold),'fold_id'].astype(int).tolist(); print('Development folds:',development_folds,'| official fold:',holdout_fold)

candidate_frames=[]; audit=[]; started=time.perf_counter()
for position,row in enumerate(inventory.itertuples(),start=1):
 train=model_data.loc[model_data.month.between(row.train_start,row.train_end)].copy(); test=model_data.loc[model_data.month.between(row.test_start,row.test_end)].copy(); features=sr.history_feature_table(train,universe,metadata); ids=features.loc[features.lifecycle_tier.eq('new'),'sku_id'].tolist(); short=forecasts.loc[forecasts.fold_id.eq(row.fold_id)&forecasts.sku_id.isin(ids),['fold_id','sku_id','block_number','block_start','target','forecast','model_key']].copy(); analogue=sr.cold_start_candidate_forecasts(train,test,ids,metadata,pd.Timestamp(row.test_start),18); analogue['fold_id']=row.fold_id; analogue=analogue[['fold_id','sku_id','block_number','block_start','target','forecast','model_key']]; combined=new.combine_candidates(short,analogue,weights=(.25,.5,.75)); candidate_frames.append(combined); audit.append({'fold_id':row.fold_id,'role':'official' if row.fold_id==holdout_fold else 'development','train_end':row.train_end,'test_start':row.test_start,'new_skus':len(ids),'candidate_rows':len(combined)}); print(f'Fold {position}/{len(inventory)} complete | new SKUs: {len(ids)} | elapsed: {(time.perf_counter()-started)/60:.1f}m',flush=True)
all_candidates=pd.concat(candidate_frames,ignore_index=True); pd.DataFrame(audit).to_csv(DOC/'new_sku_blend_chronology_audit.csv',index=False); display(pd.DataFrame(audit))

development=all_candidates.loc[all_candidates.fold_id.isin(development_folds)].copy(); validation=new.rank_candidates(new.score_candidates(development)); winner=validation.iloc[0]; validation.to_csv(DOC/'new_sku_blend_validation_candidates.csv',index=False); pd.DataFrame([winner]).to_csv(DOC/'new_sku_blend_locked_strategy.csv',index=False); print('Locked:',winner.candidate_id); display(validation.head(20))

challenger=all_candidates.loc[all_candidates.fold_id.eq(holdout_fold)&all_candidates.candidate_id.eq(winner.candidate_id)].copy(); assignments=pd.read_csv(project_root/'docs/lumpy_02_forecasting/stages/chronology_strategy/chronology_strategy_strategy_assignment_all_690.csv'); official_ids=assignments.loc[assignments.lifecycle_tier.eq('new'),'sku_id'].tolist(); assert len(official_ids)==39 and challenger.sku_id.nunique()==39
incumbent=pd.read_csv(project_root/'results/lumpy_02_forecasting/stages/chronology_strategy/chronology_strategy_final_all_690_holdout_forecasts_with_ranges.csv',parse_dates=['block_start']); incumbent=incumbent.loc[incumbent.sku_id.isin(official_ids)].copy(); incumbent['candidate_id']='Chronology-safe baseline incumbent'; challenger['candidate_id']='New-SKU short-history blend locked challenger'; comparison=pd.concat([new.score_candidates(incumbent),new.score_candidates(challenger)],ignore_index=True); comparison['pct_below_50']=100*comparison.under_50/comparison.positive_cases; comparison['pct_below_70']=100*comparison.under_70/comparison.positive_cases; comparison['pct_below_100']=100*comparison.under_100/comparison.positive_cases; comparison['bias_pct']=100*(comparison.forecast_total-comparison.actual_total)/comparison.actual_total; comparison.to_csv(DOC/'new_sku_blend_official_comparison.csv',index=False)
sku=[]
for label,frame in [('Chronology-safe baseline incumbent',incumbent),('New-SKU short-history blend locked challenger',challenger)]:
 s=frame.groupby('sku_id',as_index=False).agg(actual_total=('target','sum'),forecast_total=('forecast','sum')); e=frame.assign(error=(frame.target-frame.forecast).abs()).groupby('sku_id').error.sum(); s['absolute_error']=s.sku_id.map(e); s['wmape']=100*s.absolute_error/s.actual_total.replace(0,np.nan); s['strategy']=label; sku.append(s)
pd.concat(sku,ignore_index=True).to_csv(DOC/'new_sku_blend_individual_sku_results.csv',index=False); challenger.to_csv(OUT/'new_sku_blend_official_new_sku_forecasts.csv',index=False); display(comparison)

plot=pd.concat([incumbent.assign(strategy='Chronology-safe baseline incumbent'),challenger.assign(strategy='New-SKU short-history blend challenger')]).groupby(['strategy','block_start'],as_index=False).agg(actual=('target','sum'),forecast=('forecast','sum')); fig,axes=plt.subplots(1,2,figsize=(13,4),sharey=True)
for ax,(name,g) in zip(axes,plot.groupby('strategy',sort=False)): ax.plot(g.block_start,g.actual,color='black',marker='o',label='Actual'); ax.plot(g.block_start,g.forecast,color='#2878B5',marker='o',label='Forecast'); ax.set_title(name); ax.grid(alpha=.25); ax.legend()
axes[0].set_ylabel('Demand units'); fig.suptitle('New SKUs: actual versus forecast'); fig.tight_layout(); fig.savefig(DOC/'new_sku_blend_actual_vs_forecast.png',dpi=160,bbox_inches='tight'); plt.show()

print('Complete:',DOC/'new_sku_blend_official_comparison.csv')


## 18. New-SKU level calibration

Applies a conservative scale correction to the new-SKU blend. The correction is selected on historical new-SKU examples before the final period is examined.


In [ ]:
lf = lumpy_forecasting
sr = lumpy_sku_router
score = lumpy_new_sku_tournament
calibration = lumpy_new_sku_calibration

roots=[Path.cwd(),Path.cwd().parent,Path(r'D:/Lumpy_Fellas-/Inchscape Repo')]; project_root=next(r.resolve() for r in roots if (r/'src/lumpy_forecasting.py').exists()); sys.path.insert(0,str(project_root/'src')) if str(project_root/'src') not in sys.path else None
for module in (lf,sr,score,calibration): importlib.reload(module)
OUT=project_root/'results/lumpy_02_forecasting/stages/new_sku_calibration'; DOC=project_root/'docs/lumpy_02_forecasting/stages/new_sku_calibration'; OUT.mkdir(parents=True,exist_ok=True); DOC.mkdir(parents=True,exist_ok=True)
CONFIG=lf.LumpyConfig(variant='all_sku_history',train_months=48,gap_months=3,test_months=18,step_months=3,min_train_months=18,max_folds=6,external_mode='off'); POLICY='required_18m_gap3'

sales,external=lf.load_lumpy_inputs(project_root,CONFIG); model_data,_=lf.build_lumpy_model_frame(sales,external,CONFIG); metadata=sr.extract_static_metadata(sales); universe=sorted(model_data.sku_id.unique()); frozen=pd.read_csv(project_root/'results/lumpy_02_forecasting/stages/core_block/core_block_block_backtest_forecasts.csv',parse_dates=['block_start','train_start','train_end','test_start','test_end']); frozen=frozen.loc[frozen.policy.eq(POLICY)]; inventory=frozen[['fold_id','train_start','train_end','test_start','test_end']].drop_duplicates().sort_values('test_end'); holdout=int(inventory.iloc[-1].fold_id); development=inventory.loc[inventory.fold_id.ne(holdout),'fold_id'].astype(int).tolist(); blend_rows=[]; feature_rows=[]
for row in inventory.itertuples():
 train=model_data.loc[model_data.month.between(row.train_start,row.train_end)].copy(); test=model_data.loc[model_data.month.between(row.test_start,row.test_end)].copy(); features=sr.history_feature_table(train,universe,metadata); features=features.loc[features.lifecycle_tier.eq('new')].copy(); features['fold_id']=row.fold_id; ids=features.sku_id.tolist(); short=frozen.loc[frozen.fold_id.eq(row.fold_id)&frozen.sku_id.isin(ids)&frozen.model_key.eq('block_tsb'),['fold_id','sku_id','block_number','block_start','target','forecast']].rename(columns={'forecast':'short_forecast'}); analogue=sr.cold_start_candidate_forecasts(train,test,ids,metadata,pd.Timestamp(row.test_start),18); analogue=analogue.loc[analogue.model_key.eq('cold_subfamily_mean'),['sku_id','block_number','block_start','forecast']].rename(columns={'forecast':'analogue_forecast'}); merged=short.merge(analogue,on=['sku_id','block_number','block_start'],how='inner'); merged['forecast']=.75*merged.short_forecast+.25*merged.analogue_forecast; blend_rows.append(merged[['fold_id','sku_id','block_number','block_start','target','forecast']]); feature_rows.append(features[['fold_id','sku_id','recent_6m_total']]); print(f'Fold {row.fold_id} | new SKUs: {len(ids)}',flush=True)
base=pd.concat(blend_rows,ignore_index=True); feature_table=pd.concat(feature_rows,ignore_index=True); assert base.groupby(['fold_id','sku_id']).block_number.nunique().eq(6).all()

candidates=calibration.calibrated_routes(base,feature_table,global_scales=np.round(np.arange(.75,1.76,.10),2),route_quantiles=(.5,.75),low_scales=(.75,1,1.25),high_scales=(1,1.25,1.5,1.75,2)); historical=candidates.loc[candidates.fold_id.isin(development)]; validation=score.rank_candidates(score.score_candidates(historical)); winner=validation.iloc[0]; validation.to_csv(DOC/'new_sku_calibration_validation_candidates.csv',index=False); pd.DataFrame([winner]).to_csv(DOC/'new_sku_calibration_locked_strategy.csv',index=False); print('Locked:',winner.candidate_id); display(validation.head(20))

challenger=candidates.loc[candidates.fold_id.eq(holdout)&candidates.candidate_id.eq(winner.candidate_id)].copy(); incumbent=pd.read_csv(project_root/'results/lumpy_02_forecasting/stages/new_sku_blend/new_sku_blend_official_new_sku_forecasts.csv',parse_dates=['block_start']); incumbent['candidate_id']='New-SKU short-history blend incumbent'; challenger['candidate_id']='New-SKU level calibration locked challenger'; comparison=pd.concat([score.score_candidates(incumbent),score.score_candidates(challenger)],ignore_index=True); comparison['pct_below_50']=100*comparison.under_50/comparison.positive_cases; comparison['pct_below_70']=100*comparison.under_70/comparison.positive_cases; comparison['pct_below_100']=100*comparison.under_100/comparison.positive_cases; comparison['bias_pct']=100*(comparison.forecast_total-comparison.actual_total)/comparison.actual_total; comparison.to_csv(DOC/'new_sku_calibration_official_comparison.csv',index=False)
sku=[]
for label,frame in [('New-SKU short-history blend incumbent',incumbent),('New-SKU level calibration locked challenger',challenger)]:
 s=frame.groupby('sku_id',as_index=False).agg(actual_total=('target','sum'),forecast_total=('forecast','sum')); e=frame.assign(error=(frame.target-frame.forecast).abs()).groupby('sku_id').error.sum(); s['absolute_error']=s.sku_id.map(e); s['wmape']=100*s.absolute_error/s.actual_total.replace(0,np.nan); s['strategy']=label; sku.append(s)
pd.concat(sku,ignore_index=True).to_csv(DOC/'new_sku_calibration_individual_sku_results.csv',index=False); challenger.to_csv(OUT/'new_sku_calibration_official_new_sku_forecasts.csv',index=False); display(comparison)

plot=pd.concat([incumbent.assign(strategy='New-SKU short-history blend incumbent'),challenger.assign(strategy='New-SKU level calibration challenger')]).groupby(['strategy','block_start'],as_index=False).agg(actual=('target','sum'),forecast=('forecast','sum')); fig,axes=plt.subplots(1,2,figsize=(13,4),sharey=True)
for ax,(name,g) in zip(axes,plot.groupby('strategy',sort=False)): ax.plot(g.block_start,g.actual,color='black',marker='o',label='Actual'); ax.plot(g.block_start,g.forecast,color='#2878B5',marker='o',label='Forecast'); ax.set_title(name); ax.grid(alpha=.25); ax.legend()
axes[0].set_ylabel('Demand units'); fig.suptitle('New-SKU calibration: actual versus forecast'); fig.tight_layout(); fig.savefig(DOC/'new_sku_calibration_actual_vs_forecast.png',dpi=160,bbox_inches='tight'); plt.show()

print('Complete:',DOC/'new_sku_calibration_official_comparison.csv')


## 19. Developing-SKU model route

Handles SKUs with roughly 12–23 months of usable history. It compares block models, metadata analogues, intermittent-demand models and blended forecasts.


In [ ]:
lf = lumpy_forecasting
sr = lumpy_sku_router
tournament = lumpy_new_sku_tournament

roots=[Path.cwd(),Path.cwd().parent,Path(r'D:/Lumpy_Fellas-/Inchscape Repo')]; project_root=next(r.resolve() for r in roots if (r/'src/lumpy_forecasting.py').exists()); sys.path.insert(0,str(project_root/'src')) if str(project_root/'src') not in sys.path else None
for module in (lf,sr,tournament): importlib.reload(module)
OUT=project_root/'results/lumpy_02_forecasting/stages/developing_tournament'; DOC=project_root/'docs/lumpy_02_forecasting/stages/developing_tournament'; OUT.mkdir(parents=True,exist_ok=True); DOC.mkdir(parents=True,exist_ok=True)
CONFIG=lf.LumpyConfig(variant='all_sku_history',train_months=48,gap_months=3,test_months=18,step_months=3,min_train_months=18,max_folds=6,external_mode='off'); POLICY='required_18m_gap3'

sales,external=lf.load_lumpy_inputs(project_root,CONFIG); model_data,_=lf.build_lumpy_model_frame(sales,external,CONFIG); metadata=sr.extract_static_metadata(sales); universe=sorted(model_data.sku_id.unique()); forecasts=pd.read_csv(project_root/'results/lumpy_02_forecasting/stages/core_block/core_block_block_backtest_forecasts.csv',parse_dates=['block_start','train_start','train_end','test_start','test_end']); forecasts=forecasts.loc[forecasts.policy.eq(POLICY)].copy(); inventory=forecasts[['fold_id','train_start','train_end','test_start','test_end']].drop_duplicates().sort_values('test_end'); holdout=int(inventory.iloc[-1].fold_id); development_folds=inventory.loc[inventory.fold_id.ne(holdout),'fold_id'].astype(int).tolist(); print('Development folds:',development_folds,'| official fold:',holdout)

frames=[]; audit=[]; started=time.perf_counter(); scales=np.round(np.arange(.8,1.41,.1),2)
for position,row in enumerate(inventory.itertuples(),start=1):
 train=model_data.loc[model_data.month.between(row.train_start,row.train_end)].copy(); test=model_data.loc[model_data.month.between(row.test_start,row.test_end)].copy(); features=sr.history_feature_table(train,universe,metadata); ids=features.loc[features.lifecycle_tier.eq('developing'),'sku_id'].tolist(); short=forecasts.loc[forecasts.fold_id.eq(row.fold_id)&forecasts.sku_id.isin(ids),['fold_id','sku_id','block_number','block_start','target','forecast','model_key']].copy(); analogue=sr.cold_start_candidate_forecasts(train,test,ids,metadata,pd.Timestamp(row.test_start),18); analogue['fold_id']=row.fold_id; analogue=analogue[['fold_id','sku_id','block_number','block_start','target','forecast','model_key']]; base=tournament.combine_candidates(short,analogue,weights=(.25,.5,.75)); pairs=tournament.pair_blends(short,'block_sba','block_tsb',left_weights=(.25,.5,.75)); candidates=tournament.scale_candidate_grid(pd.concat([base,pairs],ignore_index=True),scales); frames.append(candidates); audit.append({'fold_id':row.fold_id,'role':'official' if row.fold_id==holdout else 'development','train_end':row.train_end,'test_start':row.test_start,'developing_skus':len(ids),'candidate_rows':len(candidates)}); print(f'Fold {position}/{len(inventory)} complete | developing SKUs: {len(ids)} | elapsed: {(time.perf_counter()-started)/60:.1f}m',flush=True)
all_candidates=pd.concat(frames,ignore_index=True); pd.DataFrame(audit).to_csv(DOC/'developing_tournament_chronology_audit.csv',index=False); display(pd.DataFrame(audit))

historical=all_candidates.loc[all_candidates.fold_id.isin(development_folds)]; validation=tournament.rank_candidates(tournament.score_candidates(historical)); winner=validation.iloc[0]; validation.to_csv(DOC/'developing_tournament_validation_candidates.csv',index=False); pd.DataFrame([winner]).to_csv(DOC/'developing_tournament_locked_strategy.csv',index=False); print('Locked:',winner.candidate_id); display(validation.head(20))

challenger=all_candidates.loc[all_candidates.fold_id.eq(holdout)&all_candidates.candidate_id.eq(winner.candidate_id)].copy(); assignments=pd.read_csv(project_root/'docs/lumpy_02_forecasting/stages/chronology_strategy/chronology_strategy_strategy_assignment_all_690.csv'); official_ids=assignments.loc[assignments.lifecycle_tier.eq('developing'),'sku_id'].tolist(); assert len(official_ids)==26 and challenger.sku_id.nunique()==26; incumbent=pd.read_csv(project_root/'results/lumpy_02_forecasting/stages/chronology_strategy/chronology_strategy_final_all_690_holdout_forecasts_with_ranges.csv',parse_dates=['block_start']); incumbent=incumbent.loc[incumbent.sku_id.isin(official_ids)].copy(); incumbent['candidate_id']='Chronology-safe baseline incumbent'; challenger['candidate_id']='Developing-SKU model route locked challenger'; comparison=pd.concat([tournament.score_candidates(incumbent),tournament.score_candidates(challenger)],ignore_index=True); comparison['pct_below_50']=100*comparison.under_50/comparison.positive_cases; comparison['pct_below_70']=100*comparison.under_70/comparison.positive_cases; comparison['pct_below_100']=100*comparison.under_100/comparison.positive_cases; comparison['bias_pct']=100*(comparison.forecast_total-comparison.actual_total)/comparison.actual_total; comparison.to_csv(DOC/'developing_tournament_official_comparison.csv',index=False)
sku=[]
for label,frame in [('Chronology-safe baseline incumbent',incumbent),('Developing-SKU model route locked challenger',challenger)]:
 s=frame.groupby('sku_id',as_index=False).agg(actual_total=('target','sum'),forecast_total=('forecast','sum')); e=frame.assign(error=(frame.target-frame.forecast).abs()).groupby('sku_id').error.sum(); s['absolute_error']=s.sku_id.map(e); s['wmape']=100*s.absolute_error/s.actual_total.replace(0,np.nan); s['strategy']=label; sku.append(s)
pd.concat(sku,ignore_index=True).to_csv(DOC/'developing_tournament_individual_sku_results.csv',index=False); challenger.to_csv(OUT/'developing_tournament_official_developing_forecasts.csv',index=False); display(comparison)

plot=pd.concat([incumbent.assign(strategy='Chronology-safe baseline incumbent'),challenger.assign(strategy='Developing-SKU model route challenger')]).groupby(['strategy','block_start'],as_index=False).agg(actual=('target','sum'),forecast=('forecast','sum')); fig,axes=plt.subplots(1,2,figsize=(13,4),sharey=True)
for ax,(name,g) in zip(axes,plot.groupby('strategy',sort=False)): ax.plot(g.block_start,g.actual,color='black',marker='o',label='Actual'); ax.plot(g.block_start,g.forecast,color='#2878B5',marker='o',label='Forecast'); ax.set_title(name); ax.grid(alpha=.25); ax.legend()
axes[0].set_ylabel('Demand units'); fig.suptitle('Developing SKUs: actual versus forecast'); fig.tight_layout(); fig.savefig(DOC/'developing_tournament_actual_vs_forecast.png',dpi=160,bbox_inches='tight'); plt.show()

print('Complete:',DOC/'developing_tournament_official_comparison.csv')


## 20. Developing high-volume routing

Identifies developing SKUs expected to have relatively high demand. A targeted adjustment is applied only to the historically selected high-volume group.


In [ ]:
lf = lumpy_forecasting
sr = lumpy_sku_router
score = lumpy_new_sku_tournament
routing = lumpy_new_sku_calibration

roots=[Path.cwd(),Path.cwd().parent,Path(r'D:/Lumpy_Fellas-/Inchscape Repo')]; project_root=next(r.resolve() for r in roots if (r/'src/lumpy_forecasting.py').exists()); sys.path.insert(0,str(project_root/'src')) if str(project_root/'src') not in sys.path else None
for module in (lf,sr,score,routing): importlib.reload(module)
OUT=project_root/'results/lumpy_02_forecasting/stages/developing_routing'; DOC=project_root/'docs/lumpy_02_forecasting/stages/developing_routing'; OUT.mkdir(parents=True,exist_ok=True); DOC.mkdir(parents=True,exist_ok=True)
CONFIG=lf.LumpyConfig(variant='all_sku_history',train_months=48,gap_months=3,test_months=18,step_months=3,min_train_months=18,max_folds=6,external_mode='off'); POLICY='required_18m_gap3'

sales,external=lf.load_lumpy_inputs(project_root,CONFIG); model_data,_=lf.build_lumpy_model_frame(sales,external,CONFIG); metadata=sr.extract_static_metadata(sales); universe=sorted(model_data.sku_id.unique()); frozen=pd.read_csv(project_root/'results/lumpy_02_forecasting/stages/core_block/core_block_block_backtest_forecasts.csv',parse_dates=['block_start','train_start','train_end','test_start','test_end']); frozen=frozen.loc[frozen.policy.eq(POLICY)]; inventory=frozen[['fold_id','train_start','train_end','test_start','test_end']].drop_duplicates().sort_values('test_end'); holdout=int(inventory.iloc[-1].fold_id); development=inventory.loc[inventory.fold_id.ne(holdout),'fold_id'].astype(int).tolist(); blend_rows=[]; feature_rows=[]
for row in inventory.itertuples():
 train=model_data.loc[model_data.month.between(row.train_start,row.train_end)].copy(); test=model_data.loc[model_data.month.between(row.test_start,row.test_end)].copy(); features=sr.history_feature_table(train,universe,metadata); features=features.loc[features.lifecycle_tier.eq('developing')].copy(); features['fold_id']=row.fold_id; ids=features.sku_id.tolist(); short=frozen.loc[frozen.fold_id.eq(row.fold_id)&frozen.sku_id.isin(ids)&frozen.model_key.eq('block_hurdle_d2_full_log'),['fold_id','sku_id','block_number','block_start','target','forecast']].rename(columns={'forecast':'short_forecast'}); analogue=sr.cold_start_candidate_forecasts(train,test,ids,metadata,pd.Timestamp(row.test_start),18); analogue=analogue.loc[analogue.model_key.eq('cold_subfamily_mean'),['sku_id','block_number','block_start','forecast']].rename(columns={'forecast':'analogue_forecast'}); merged=short.merge(analogue,on=['sku_id','block_number','block_start'],how='inner'); merged['forecast']=.75*merged.short_forecast+.25*merged.analogue_forecast; base=merged[['fold_id','sku_id','block_number','block_start','target','forecast']]; predicted=base.groupby(['fold_id','sku_id'],as_index=False).forecast.sum().rename(columns={'forecast':'predicted_horizon_total'}); features=features.merge(predicted,on=['fold_id','sku_id'],how='left'); blend_rows.append(base); feature_rows.append(features[['fold_id','sku_id','recent_6m_total','positive_mean','recent_vs_prior_6m_ratio','predicted_horizon_total']]); print(f'Fold {row.fold_id} | developing SKUs: {len(ids)}',flush=True)
base=pd.concat(blend_rows,ignore_index=True); feature_table=pd.concat(feature_rows,ignore_index=True); assert base.groupby(['fold_id','sku_id']).block_number.nunique().eq(6).all()

route_features=('recent_6m_total','positive_mean','recent_vs_prior_6m_ratio','predicted_horizon_total'); candidates=routing.calibrated_routes(base,feature_table,global_scales=(1.0,),route_quantiles=(.5,.75),low_scales=(.8,1,1.1),high_scales=(1,1.25,1.5,1.75),route_features=route_features); historical=candidates.loc[candidates.fold_id.isin(development)]; validation=score.rank_candidates(score.score_candidates(historical)); winner=validation.iloc[0]; validation.to_csv(DOC/'developing_routing_validation_candidates.csv',index=False); pd.DataFrame([winner]).to_csv(DOC/'developing_routing_locked_strategy.csv',index=False); print('Locked:',winner.candidate_id); display(validation.head(20))

challenger=candidates.loc[candidates.fold_id.eq(holdout)&candidates.candidate_id.eq(winner.candidate_id)].copy(); incumbent=pd.read_csv(project_root/'results/lumpy_02_forecasting/stages/developing_tournament/developing_tournament_official_developing_forecasts.csv',parse_dates=['block_start']); incumbent['candidate_id']='Developing-SKU model route incumbent'; challenger['candidate_id']='Developing high-volume routing locked challenger'; comparison=pd.concat([score.score_candidates(incumbent),score.score_candidates(challenger)],ignore_index=True); comparison['pct_below_50']=100*comparison.under_50/comparison.positive_cases; comparison['pct_below_70']=100*comparison.under_70/comparison.positive_cases; comparison['pct_below_100']=100*comparison.under_100/comparison.positive_cases; comparison['bias_pct']=100*(comparison.forecast_total-comparison.actual_total)/comparison.actual_total; comparison.to_csv(DOC/'developing_routing_official_comparison.csv',index=False)
sku=[]
for label,frame in [('Developing-SKU model route incumbent',incumbent),('Developing high-volume routing locked challenger',challenger)]:
 s=frame.groupby('sku_id',as_index=False).agg(actual_total=('target','sum'),forecast_total=('forecast','sum')); e=frame.assign(error=(frame.target-frame.forecast).abs()).groupby('sku_id').error.sum(); s['absolute_error']=s.sku_id.map(e); s['wmape']=100*s.absolute_error/s.actual_total.replace(0,np.nan); s['strategy']=label; sku.append(s)
pd.concat(sku,ignore_index=True).to_csv(DOC/'developing_routing_individual_sku_results.csv',index=False); challenger.to_csv(OUT/'developing_routing_official_developing_forecasts.csv',index=False); display(comparison)

plot=pd.concat([incumbent.assign(strategy='Developing-SKU model route incumbent'),challenger.assign(strategy='Developing high-volume routing challenger')]).groupby(['strategy','block_start'],as_index=False).agg(actual=('target','sum'),forecast=('forecast','sum')); fig,axes=plt.subplots(1,2,figsize=(13,4),sharey=True)
for ax,(name,g) in zip(axes,plot.groupby('strategy',sort=False)): ax.plot(g.block_start,g.actual,color='black',marker='o',label='Actual'); ax.plot(g.block_start,g.forecast,color='#2878B5',marker='o',label='Forecast'); ax.set_title(name); ax.grid(alpha=.25); ax.legend()
axes[0].set_ylabel('Demand units'); fig.suptitle('Developing-SKU routing: actual versus forecast'); fig.tight_layout(); fig.savefig(DOC/'developing_routing_actual_vs_forecast.png',dpi=160,bbox_inches='tight'); plt.show()

print('Complete:',DOC/'developing_routing_official_comparison.csv')


## 21. Consolidated segment champions

Brings the retained routes together:

- Recurring
- Occasional
- Rare
- Cold start
- New
- Developing
- Dormant

It checks that every SKU has exactly one route and six forecast blocks.


In [ ]:
consolidated = lumpy_consolidated_scorecard

roots=[Path.cwd(),Path.cwd().parent,Path(r'D:/Lumpy_Fellas-/Inchscape Repo')]; project_root=next(r.resolve() for r in roots if (r/'src/lumpy_forecasting.py').exists()); sys.path.insert(0,str(project_root/'src')) if str(project_root/'src') not in sys.path else None
importlib.reload(consolidated)
OUT=project_root/'results/lumpy_02_forecasting/stages/champion_consolidation'; DOC=project_root/'docs/lumpy_02_forecasting/stages/champion_consolidation'; OUT.mkdir(parents=True,exist_ok=True); DOC.mkdir(parents=True,exist_ok=True)

registry=pd.DataFrame([
 {'segment':'recurring','source_notebook':'recurring_scorecard','source_file':'results/lumpy_02_forecasting/stages/recurring_scorecard/recurring_scorecard_official_abc_forecasts.csv','decision':'six-block recurring A/B/C champion'},
 {'segment':'occasional','source_notebook':'occasional_memory','source_file':'results/lumpy_02_forecasting/stages/occasional_memory/occasional_memory_temporal_memory_forecasts.csv','decision':'temporal-memory champion'},
 {'segment':'rare','source_notebook':'rare_group_occurrence','source_file':'results/lumpy_02_forecasting/stages/rare_group_occurrence/rare_group_occurrence_official_rare_forecasts.csv','decision':'group-occurrence champion'},
 {'segment':'cold_start','source_notebook':'cold_start','source_file':'results/lumpy_02_forecasting/stages/cold_start/cold_start_official_cold_start_forecasts.csv','decision':'genuine-launch champion'},
 {'segment':'new','source_notebook':'new_sku_calibration','source_file':'results/lumpy_02_forecasting/stages/new_sku_calibration/new_sku_calibration_official_new_sku_forecasts.csv','decision':'calibrated short-history blend'},
 {'segment':'developing','source_notebook':'developing_routing','source_file':'results/lumpy_02_forecasting/stages/developing_routing/developing_routing_official_developing_forecasts.csv','decision':'routed hurdle-analogue champion'},
 {'segment':'dormant','source_notebook':'chronology_strategy','source_file':'results/lumpy_02_forecasting/stages/chronology_strategy/chronology_strategy_final_all_690_holdout_forecasts_with_ranges.csv','decision':'governance baseline; manual review exclusion'},
]); registry.to_csv(DOC/'champion_consolidation_champion_registry.csv',index=False); display(registry)

assignments=pd.read_csv(project_root/'docs/lumpy_02_forecasting/stages/chronology_strategy/chronology_strategy_strategy_assignment_all_690.csv'); frames=[]; coverage=[]
for row in registry.itertuples():
 frame=pd.read_csv(project_root/row.source_file,parse_dates=['block_start'])
 if row.segment=='dormant':
  ids=assignments.loc[assignments.lifecycle_tier.eq('dormant'),'sku_id']; frame=frame.loc[frame.sku_id.isin(ids)].copy()
 normalized=consolidated.normalize_forecasts(frame,row.segment,f'Notebook {row.source_notebook}')
 frames.append(normalized); coverage.append({'segment':row.segment,'source_notebook':row.source_notebook,'skus':normalized.sku_id.nunique(),'rows':len(normalized),'positive_skus':normalized.groupby('sku_id').target.sum().gt(0).sum(),'actual_total':normalized.target.sum()})
champions=consolidated.consolidate(frames,expected_skus=690); coverage=pd.DataFrame(coverage); assert len(champions)==690*6 and champions.target.sum()==9111; coverage.to_csv(DOC/'champion_consolidation_coverage_audit.csv',index=False); champions.to_csv(OUT/'champion_consolidation_all_690_champion_forecasts.csv',index=False); display(coverage); print('Validated:',champions.sku_id.nunique(),'SKUs |',len(champions),'rows | actual units:',champions.target.sum())

full_segment=consolidated.score(champions,'segment'); full_overall=consolidated.score(champions); actionable=champions.loc[champions.segment.ne('dormant')].copy(); actionable_segment=consolidated.score(actionable,'segment'); actionable_overall=consolidated.score(actionable); full_segment.to_csv(DOC/'champion_consolidation_full_segment_scorecard.csv',index=False); full_overall.to_csv(DOC/'champion_consolidation_full_overall_scorecard.csv',index=False); actionable_segment.to_csv(DOC/'champion_consolidation_actionable_segment_scorecard.csv',index=False); actionable_overall.to_csv(DOC/'champion_consolidation_actionable_overall_scorecard.csv',index=False); actionable.to_csv(OUT/'champion_consolidation_actionable_637_forecasts.csv',index=False); print('Full governance scorecard'); display(full_overall); print('Actionable scorecard excluding dormant'); display(actionable_overall); display(full_segment)

baseline=pd.read_csv(project_root/'results/lumpy_02_forecasting/stages/chronology_strategy/chronology_strategy_final_all_690_holdout_forecasts_with_ranges.csv',parse_dates=['block_start']); baseline=consolidated.normalize_forecasts(baseline,'baseline','Chronology-safe baseline'); baseline_full=consolidated.score(baseline); dormant_ids=set(assignments.loc[assignments.lifecycle_tier.eq('dormant'),'sku_id']); baseline_actionable=consolidated.score(baseline.loc[~baseline.sku_id.isin(dormant_ids)]); comparison=pd.concat([baseline_full.assign(scorecard='Chronology-safe baseline all 690'),full_overall.assign(scorecard='Champion all 690'),baseline_actionable.assign(scorecard='Chronology-safe baseline excluding dormant'),actionable_overall.assign(scorecard='Champion excluding dormant')],ignore_index=True); comparison.to_csv(DOC/'champion_consolidation_baseline_comparison.csv',index=False); display(comparison[['scorecard','all_skus','positive_skus','under_50','under_70','under_100','median_wmape','portfolio_wmape','bias_pct']])

plot=full_segment.copy(); x=np.arange(len(plot)); width=.24; fig,ax=plt.subplots(figsize=(13,5)); ax.bar(x-width,plot.pct_below_50,width,label='Below 50%'); ax.bar(x,plot.pct_below_70,width,label='Below 70%'); ax.bar(x+width,plot.pct_below_100,width,label='Below 100%'); ax.set_xticks(x,plot.segment,rotation=25,ha='right'); ax.set_ylabel('Positive SKUs (%)'); ax.set_title('Official individual-SKU WMAPE coverage by routed segment'); ax.grid(axis='y',alpha=.25); ax.legend(); fig.tight_layout(); fig.savefig(DOC/'champion_consolidation_segment_threshold_coverage.png',dpi=160,bbox_inches='tight'); plt.show()

print('Complete. Governance and actionable scorecards are contract-aligned and coverage-checked.')


## 22. Bias and error attribution

Examines where the remaining error comes from. It separates missed events, false events, underestimated quantities and overestimated quantities, and shows how error is concentrated by SKU and segment.


In [ ]:
audit = lumpy_error_attribution

roots=[Path.cwd(),Path.cwd().parent,Path(r'D:/Lumpy_Fellas-/Inchscape Repo')]; project_root=next(r.resolve() for r in roots if (r/'src/lumpy_forecasting.py').exists()); sys.path.insert(0,str(project_root/'src')) if str(project_root/'src') not in sys.path else None
importlib.reload(audit)
DOC=project_root/'docs/lumpy_02_forecasting/stages/error_attribution'; DOC.mkdir(parents=True,exist_ok=True)

forecasts=pd.read_csv(project_root/'results/lumpy_02_forecasting/stages/champion_consolidation/champion_consolidation_actionable_637_forecasts.csv',parse_dates=['block_start']); assert forecasts.sku_id.nunique()==637 and forecasts.groupby('sku_id').block_number.nunique().eq(6).all(); print('Actionable SKUs:',forecasts.sku_id.nunique(),'| positive:',forecasts.groupby('sku_id').target.sum().gt(0).sum(),'| actual units:',forecasts.target.sum())

sku=audit.sku_error_table(forecasts); concentration=audit.concentration_table(sku); sku.to_csv(DOC/'error_attribution_sku_error_audit.csv',index=False); concentration.to_csv(DOC/'error_attribution_error_concentration.csv',index=False); display(concentration)

segment=sku.groupby('segment',as_index=False).agg(all_skus=('sku_id','size'),positive_skus=('actual_total',lambda values:values.gt(0).sum()),actual_units=('actual_total','sum'),forecast_units=('forecast_total','sum'),absolute_error=('absolute_error','sum'),signed_error=('signed_error','sum'),median_wmape=('wmape','median')); segment['bias_pct']=100*segment.signed_error/segment.actual_units; segment['absolute_error_share']=segment.absolute_error/segment.absolute_error.sum(); volume=sku.groupby(['segment','actual_volume_quartile'],observed=True,as_index=False).agg(skus=('sku_id','size'),actual_units=('actual_total','sum'),forecast_units=('forecast_total','sum'),absolute_error=('absolute_error','sum'),signed_error=('signed_error','sum')); volume['bias_pct']=100*volume.signed_error/volume.actual_units.replace(0,np.nan); segment.to_csv(DOC/'error_attribution_segment_error_attribution.csv',index=False); volume.to_csv(DOC/'error_attribution_volume_quartile_attribution.csv',index=False); display(segment.sort_values('absolute_error',ascending=False)); display(volume)

mechanism=audit.block_error_attribution(forecasts,event_threshold=.5); mechanism.to_csv(DOC/'error_attribution_block_error_mechanism.csv',index=False); display(mechanism.sort_values(['segment','absolute_error'],ascending=[True,False]))

positive=sku.loc[sku.actual_total.gt(0)].sort_values('absolute_error',ascending=False).reset_index(drop=True); positive['cumulative_error_share']=positive.absolute_error.cumsum()/positive.absolute_error.sum(); positive['cumulative_sku_share']=(np.arange(len(positive))+1)/len(positive); fig,axes=plt.subplots(1,2,figsize=(13,4)); axes[0].plot(positive.cumulative_sku_share*100,positive.cumulative_error_share*100,color='#2878B5'); axes[0].plot([0,100],[0,100],ls='--',color='grey'); axes[0].set(xlabel='Positive SKUs included (%)',ylabel='Cumulative absolute error (%)',title='Actionable error concentration'); axes[0].grid(alpha=.25); ordered=segment.sort_values('signed_error'); axes[1].bar(ordered.segment,ordered.bias_pct,color=np.where(ordered.bias_pct<0,'#C44E52','#4C956C')); axes[1].axhline(0,color='black',lw=.8); axes[1].set(ylabel='Bias (%)',title='Bias by routed segment'); axes[1].tick_params(axis='x',rotation=25); fig.tight_layout(); fig.savefig(DOC/'error_attribution_bias_concentration.png',dpi=160,bbox_inches='tight'); plt.show()

print('Complete. This notebook diagnoses calibration opportunities without selecting or modifying forecasts.')


## 23. Selective historical calibration

Tests whether forecast-level calibration transfers from earlier periods. Calibration is promoted only for the segment where it improves the main SKU thresholds—new SKUs.


In [ ]:
lf = lumpy_forecasting
sr = lumpy_sku_router
score = lumpy_new_sku_tournament
calibration = lumpy_quantile_calibration

roots=[Path.cwd(),Path.cwd().parent,Path(r'D:/Lumpy_Fellas-/Inchscape Repo')]; project_root=next(r.resolve() for r in roots if (r/'src/lumpy_forecasting.py').exists()); sys.path.insert(0,str(project_root/'src')) if str(project_root/'src') not in sys.path else None
for module in (lf,sr,score,calibration): importlib.reload(module)
OUT=project_root/'results/lumpy_02_forecasting/stages/selective_calibration'; DOC=project_root/'docs/lumpy_02_forecasting/stages/selective_calibration'; OUT.mkdir(parents=True,exist_ok=True); DOC.mkdir(parents=True,exist_ok=True)
CONFIG=lf.LumpyConfig(variant='all_sku_history',train_months=48,gap_months=3,test_months=18,step_months=3,min_train_months=18,max_folds=6,external_mode='off'); POLICY='required_18m_gap3'

sales,external=lf.load_lumpy_inputs(project_root,CONFIG); model_data,_=lf.build_lumpy_model_frame(sales,external,CONFIG); metadata=sr.extract_static_metadata(sales); universe=sorted(model_data.sku_id.unique()); frozen=pd.read_csv(project_root/'results/lumpy_02_forecasting/stages/core_block/core_block_block_backtest_forecasts.csv',parse_dates=['block_start','train_start','train_end','test_start','test_end']); frozen=frozen.loc[frozen.policy.eq(POLICY)&frozen.model_key.eq('block_tsb')].copy(); inventory=frozen[['fold_id','train_start','train_end','test_start','test_end']].drop_duplicates().sort_values('test_end'); holdout=int(inventory.iloc[-1].fold_id); fold_frames=[]; audit=[]
for row in inventory.itertuples():
 train=model_data.loc[model_data.month.between(row.train_start,row.train_end)]; features=sr.history_feature_table(train,universe,metadata); features['segment']=np.where(features.lifecycle_tier.eq('established'),features.frequency_tier.map({'recurring_4_6':'recurring','occasional_2_3':'occasional','rare_0_1':'rare'}),features.lifecycle_tier); features=features.loc[features.segment.isin(['recurring','occasional','rare','cold_start','new','developing'])]; frame=frozen.loc[frozen.fold_id.eq(row.fold_id)&frozen.sku_id.isin(features.sku_id),['fold_id','sku_id','block_number','block_start','target','forecast']].merge(features[['sku_id','segment']],on='sku_id',how='inner'); fold_frames.append(frame); audit.append({'fold_id':row.fold_id,'role':'official' if row.fold_id==holdout else 'historical','train_end':row.train_end,'test_start':row.test_start,'actionable_skus':frame.sku_id.nunique()})
historical_base=pd.concat(fold_frames,ignore_index=True); audit=pd.DataFrame(audit); audit.to_csv(DOC/'selective_calibration_chronology_audit.csv',index=False); display(audit)

historical_inventory=inventory.loc[inventory.fold_id.ne(holdout)]; strengths=(0,.25,.5,.75,1.0); upper_caps=(1.25,1.5,2.0); candidate_parts=[]; training_horizons=pd.DataFrame()
for position,row in enumerate(historical_inventory.itertuples(),start=1):
 current=historical_base.loc[historical_base.fold_id.eq(row.fold_id)].copy(); current_horizon=calibration.horizon_table(current)
 if len(training_horizons):
  model=calibration.fit_quantile_model(training_horizons)
  for strength in strengths:
   for upper_cap in upper_caps:
    candidate=calibration.apply_quantile_model(current,model,strength=strength,lower_cap=.75,upper_cap=upper_cap); candidate['candidate_id']=f'strength_{strength:.2f}__upper_{upper_cap:.2f}'; candidate_parts.append(candidate)
 training_horizons=pd.concat([training_horizons,current_horizon],ignore_index=True)
 print(f'Historical fold {position}/{len(historical_inventory)} | training cases now {len(training_horizons):,}',flush=True)
historical_candidates=pd.concat(candidate_parts,ignore_index=True); validation_rows=[]; locks=[]
for segment,segment_rows in historical_candidates.groupby('segment'):
 ranked=score.rank_candidates(score.score_candidates(segment_rows)); ranked['segment']=segment; validation_rows.append(ranked); winner=ranked.iloc[0]; locks.append({'segment':segment,'candidate_id':winner.candidate_id,'positive_cases':winner.positive_cases,'under_50':winner.under_50,'under_70':winner.under_70,'under_100':winner.under_100,'median_wmape':winner.median_wmape})
validation=pd.concat(validation_rows,ignore_index=True); locked=pd.DataFrame(locks); validation.to_csv(DOC/'selective_calibration_validation_candidates.csv',index=False); locked.to_csv(DOC/'selective_calibration_locked_strategy.csv',index=False); display(locked)

final_model=calibration.fit_quantile_model(training_horizons); final_model.to_csv(DOC/'selective_calibration_fitted_calibration_factors.csv',index=False); champions=pd.read_csv(project_root/'results/lumpy_02_forecasting/stages/champion_consolidation/champion_consolidation_actionable_637_forecasts.csv',parse_dates=['block_start']); challenger_parts=[]
for segment,group in champions.groupby('segment',sort=False):
 choice=locked.loc[locked.segment.eq(segment)]
 if len(choice):
  parts=choice.iloc[0].candidate_id.split('__'); strength=float(parts[0].split('_')[1]); upper_cap=float(parts[1].split('_')[1]); calibrated=calibration.apply_quantile_model(group,final_model,strength=strength,lower_cap=.75,upper_cap=upper_cap)
 else: calibrated=group.copy(); calibrated['quartile']=np.nan; calibrated['calibration_factor']=1.0
 challenger_parts.append(calibrated)
challenger=pd.concat(challenger_parts,ignore_index=True); incumbent=champions.copy(); incumbent['candidate_id']='Consolidated segment champions champions'; challenger['candidate_id']='Selective historical calibration calibrated challenger'; comparison=pd.concat([score.score_candidates(incumbent),score.score_candidates(challenger)],ignore_index=True); comparison['pct_below_50']=100*comparison.under_50/comparison.positive_cases; comparison['pct_below_70']=100*comparison.under_70/comparison.positive_cases; comparison['pct_below_100']=100*comparison.under_100/comparison.positive_cases; comparison['bias_pct']=100*(comparison.forecast_total-comparison.actual_total)/comparison.actual_total; comparison.to_csv(DOC/'selective_calibration_official_comparison.csv',index=False); challenger.to_csv(OUT/'selective_calibration_actionable_calibrated_forecasts.csv',index=False); display(comparison)

rows=[]
for strategy,frame in [('Consolidated segment champions champions',incumbent),('Selective historical calibration challenger',challenger)]:
 for segment,group in frame.groupby('segment'):
  result=score.score_candidates(group.assign(candidate_id=strategy)).iloc[0].to_dict(); rows.append({'strategy':strategy,'segment':segment,**result})
segment_comparison=pd.DataFrame(rows); segment_comparison['bias_pct']=100*(segment_comparison.forecast_total-segment_comparison.actual_total)/segment_comparison.actual_total; segment_comparison.to_csv(DOC/'selective_calibration_segment_comparison.csv',index=False); display(segment_comparison[['strategy','segment','positive_cases','under_50','under_70','under_100','median_wmape','portfolio_wmape','bias_pct']])
decisions=[]; recommended=[]
for segment in incumbent.segment.unique():
 inc=segment_comparison.loc[segment_comparison.segment.eq(segment)&segment_comparison.strategy.eq('Consolidated segment champions champions')].iloc[0]; chal=segment_comparison.loc[segment_comparison.segment.eq(segment)&segment_comparison.strategy.eq('Selective historical calibration challenger')].iloc[0]; accepted=(chal.under_70>inc.under_70) or (chal.under_70==inc.under_70 and chal.under_50>inc.under_50); decisions.append({'segment':segment,'accepted':accepted,'incumbent_under_50':inc.under_50,'challenger_under_50':chal.under_50,'incumbent_under_70':inc.under_70,'challenger_under_70':chal.under_70,'incumbent_under_100':inc.under_100,'challenger_under_100':chal.under_100}); chosen=challenger if accepted else incumbent; recommended.append(chosen.loc[chosen.segment.eq(segment)].copy())
decisions=pd.DataFrame(decisions); recommended=pd.concat(recommended,ignore_index=True); decisions.to_csv(DOC/'selective_calibration_promotion_decisions.csv',index=False); recommended.to_csv(OUT/'selective_calibration_recommended_actionable_forecasts.csv',index=False); recommended_score=score.score_candidates(recommended.assign(candidate_id='Selective historical calibration selective recommendation')); recommended_score['pct_below_50']=100*recommended_score.under_50/recommended_score.positive_cases; recommended_score['pct_below_70']=100*recommended_score.under_70/recommended_score.positive_cases; recommended_score['pct_below_100']=100*recommended_score.under_100/recommended_score.positive_cases; recommended_score['bias_pct']=100*(recommended_score.forecast_total-recommended_score.actual_total)/recommended_score.actual_total; recommended_comparison=pd.concat([comparison,recommended_score],ignore_index=True); recommended_comparison.to_csv(DOC/'selective_calibration_recommended_comparison.csv',index=False); display(decisions); display(recommended_comparison)

plot=pd.concat([incumbent.assign(strategy='Consolidated segment champions'),recommended.assign(strategy='Selective historical calibration selective')]).groupby(['strategy','block_start'],as_index=False).agg(actual=('target','sum'),forecast=('forecast','sum')); fig,axes=plt.subplots(1,2,figsize=(13,4),sharey=True)
for ax,(name,g) in zip(axes,plot.groupby('strategy',sort=False)): ax.plot(g.block_start,g.actual,color='black',marker='o',label='Actual'); ax.plot(g.block_start,g.forecast,color='#2878B5',marker='o',label='Forecast'); ax.set_title(name); ax.grid(alpha=.25); ax.legend()
axes[0].set_ylabel('Demand units'); fig.suptitle('Actionable portfolio calibration: actual versus forecast'); fig.tight_layout(); fig.savefig(DOC/'selective_calibration_actual_vs_forecast.png',dpi=160,bbox_inches='tight'); plt.show()

print('Complete. Promotion requires individual-SKU threshold gains, not bias improvement alone.')


## 24. SKU delivery tables

Converts the block forecasts into one row per SKU. Each row includes classifications, selected method, six forecasts, totals, WMAPE, bias and a reliability tier.


In [ ]:
delivery = lumpy_delivery_output

roots=[Path.cwd(),Path.cwd().parent,Path(r'D:/Lumpy_Fellas-/Inchscape Repo')]; project_root=next(r.resolve() for r in roots if (r/'src/lumpy_forecasting.py').exists()); sys.path.insert(0,str(project_root/'src')) if str(project_root/'src') not in sys.path else None
importlib.reload(delivery)
OUT=project_root/'results/lumpy_02_forecasting/stages/delivery'; DOC=project_root/'docs/lumpy_02_forecasting/stages/delivery'; OUT.mkdir(parents=True,exist_ok=True); DOC.mkdir(parents=True,exist_ok=True)

actionable=pd.read_csv(project_root/'results/lumpy_02_forecasting/stages/selective_calibration/selective_calibration_recommended_actionable_forecasts.csv',parse_dates=['block_start']); full=pd.read_csv(project_root/'results/lumpy_02_forecasting/stages/champion_consolidation/champion_consolidation_all_690_champion_forecasts.csv',parse_dates=['block_start']); dormant=full.loc[full.segment.eq('dormant')].copy(); actionable.loc[actionable.segment.eq('new'),'champion_source']='Selective historical calibration selective calibration'; combined=pd.concat([actionable,dormant],ignore_index=True); assert combined.sku_id.nunique()==690 and combined.groupby('sku_id').block_number.nunique().eq(6).all(); print('Combined rows:',len(combined),'| SKUs:',combined.sku_id.nunique())

assignments=pd.read_csv(project_root/'docs/lumpy_02_forecasting/stages/chronology_strategy/chronology_strategy_strategy_assignment_all_690.csv'); table=delivery.build_delivery_table(combined,assignments); assert len(table)==690; table.to_csv(OUT/'delivery_all_690_sku_delivery.csv',index=False); actionable_table=table.loc[~table.dormant_manual_review].copy(); actionable_table.to_csv(OUT/'delivery_actionable_637_sku_delivery.csv',index=False); print('Full delivery rows:',len(table)); print('Actionable rows:',len(actionable_table)); display(table.drop(columns=['sku_id']).head())

reliability=table.groupby(['dormant_manual_review','reliability_tier'],as_index=False).agg(skus=('sku_id','size'),positive_skus=('positive_sku','sum'),actual_units=('actual_18m','sum'),forecast_units=('forecast_18m','sum')); bias=table.groupby(['segment','bias_direction'],as_index=False).agg(skus=('sku_id','size'),actual_units=('actual_18m','sum'),forecast_units=('forecast_18m','sum'),absolute_error=('absolute_error','sum')); reliability.to_csv(DOC/'delivery_reliability_summary.csv',index=False); bias.to_csv(DOC/'delivery_bias_summary.csv',index=False); display(reliability); display(bias)

dictionary=pd.DataFrame([
 ('sku_id','Sensitive stock-code identifier.'),('segment','Champion routing segment.'),('champion_source','Stage or method supplying the selected forecast.'),('forecast_block_1 ... forecast_block_6','Selected forecast for each three-month block.'),('actual_block_1 ... actual_block_6','Observed backtest demand; evaluation only.'),('forecast_18m','Sum of six selected forecast blocks.'),('actual_18m','Observed 18-month demand; evaluation only.'),('wmape','Individual pooled WMAPE across six blocks.'),('evaluation_mase','Diagnostic block-level MASE using changes inside the evaluation horizon.'),('bias_pct','Evaluation forecast bias percentage.'),('reliability_tier','Post-evaluation operational reliability category.'),('dormant_manual_review','Dormant lifecycle exclusion from actionable forecasting.'),('positive_sku','Actual demand exceeded zero in the official horizon.'),
],columns=['column','definition']); dictionary.to_csv(DOC/'delivery_column_dictionary.csv',index=False); display(dictionary)

print('Complete. SKU identifiers remain only in local output files and are not printed by this notebook.')


## 25. Planning policy

Maps each reliability tier to a practical action:

- Forecast-led replenishment
- Forecast plus review
- Manual approval with forecast guidance
- Exception or inventory policy
- Dormant lifecycle review


In [ ]:
policy = lumpy_operational_policy

roots=[Path.cwd(),Path.cwd().parent,Path(r'D:/Lumpy_Fellas-/Inchscape Repo')]; project_root=next(r.resolve() for r in roots if (r/'src/lumpy_operational_policy.py').exists()); sys.path.insert(0,str(project_root/'src')) if str(project_root/'src') not in sys.path else None
importlib.reload(policy)
OUT=project_root/'results/lumpy_02_forecasting/stages/operational_policy'; DOC=project_root/'docs/lumpy_02_forecasting/stages/operational_policy'; OUT.mkdir(parents=True,exist_ok=True); DOC.mkdir(parents=True,exist_ok=True)

delivery=pd.read_csv(project_root/'results/lumpy_02_forecasting/stages/delivery/delivery_all_690_sku_delivery.csv'); operational=policy.add_operational_policy(delivery); assert len(operational)==690; operational.to_csv(OUT/'operational_policy_all_690_operational_policy.csv',index=False); actionable=operational.loc[~operational.dormant_manual_review].copy(); actionable.to_csv(OUT/'operational_policy_actionable_637_operational_policy.csv',index=False); print('Full policy rows:',len(operational),'| actionable:',len(actionable))

summary=operational.groupby(['reliability_tier','planning_action','review_cadence','inventory_treatment','owner'],as_index=False).agg(skus=('sku_id','size'),positive_skus=('positive_sku','sum'),actual_units=('actual_18m','sum'),forecast_units=('forecast_18m','sum'),underforecast_escalations=('underforecast_escalation','sum')); summary.to_csv(DOC/'operational_policy_policy_summary.csv',index=False); display(summary)

queue=operational.groupby(['segment','planning_action'],as_index=False).agg(skus=('sku_id','size'),positive_skus=('positive_sku','sum'),underforecast_escalations=('underforecast_escalation','sum'),actual_units=('actual_18m','sum'),forecast_units=('forecast_18m','sum')); queue.to_csv(DOC/'operational_policy_segment_work_queue.csv',index=False); display(queue)

print('Complete. Policies are review-ready but not cost-optimised stock rules.')


## 26. Final methodology audit

Checks the complete result for:

- 690-SKU coverage
- Six blocks per SKU
- Correct forecast dates
- No overlapping routes
- Correct actual-demand total
- Non-negative forecasts
- Historically locked model choices
- Cutoff-safe classifications
- Proper dormant handling
- No use of final-period actuals as model features

After stage 26, the final export writes the stable files used by the results-review and planning-review notebooks.


In [ ]:
audit = lumpy_methodology_audit

roots=[Path.cwd(),Path.cwd().parent,Path(r'D:/Lumpy_Fellas-/Inchscape Repo')]; project_root=next(r.resolve() for r in roots if (r/'src/lumpy_methodology_audit.py').exists()); sys.path.insert(0,str(project_root/'src')) if str(project_root/'src') not in sys.path else None
importlib.reload(audit)
DOC=project_root/'docs/lumpy_02_forecasting/stages/methodology_audit'; DOC.mkdir(parents=True,exist_ok=True)

actionable=pd.read_csv(project_root/'results/lumpy_02_forecasting/stages/selective_calibration/selective_calibration_recommended_actionable_forecasts.csv',parse_dates=['block_start']); full=pd.read_csv(project_root/'results/lumpy_02_forecasting/stages/champion_consolidation/champion_consolidation_all_690_champion_forecasts.csv',parse_dates=['block_start']); dormant=full.loc[full.segment.eq('dormant')]; forecasts=pd.concat([actionable,dormant],ignore_index=True); checks=audit.contract_checks(forecasts); checks.to_csv(DOC/'methodology_audit_contract_checks.csv',index=False); display(checks); assert checks.passed.all()

registry=pd.DataFrame([
 {'segment':'recurring','source_notebook':'recurring_scorecard','evidence_path':'docs/lumpy_02_forecasting/stages/recurring_scorecard/recurring_scorecard_configuration_registry.csv','selected_before_final':True,'cutoff_safe_population':True,'official_contract_compatible':True,'external_features_used':False},
 {'segment':'occasional','source_notebook':'occasional_memory','evidence_path':'docs/lumpy_02_forecasting/stages/occasional_memory/occasional_memory_locked_memory_rules.json','selected_before_final':True,'cutoff_safe_population':True,'official_contract_compatible':True,'external_features_used':False},
 {'segment':'rare','source_notebook':'rare_group_occurrence','evidence_path':'docs/lumpy_02_forecasting/stages/rare_group_occurrence/rare_group_occurrence_locked_strategies.json','selected_before_final':True,'cutoff_safe_population':True,'official_contract_compatible':True,'external_features_used':False},
 {'segment':'cold_start','source_notebook':'cold_start','evidence_path':'docs/lumpy_02_forecasting/stages/cold_start/cold_start_locked_strategy.csv','selected_before_final':True,'cutoff_safe_population':True,'official_contract_compatible':True,'external_features_used':False},
 {'segment':'new','source_notebook':'selective_calibration','evidence_path':'docs/lumpy_02_forecasting/stages/selective_calibration/selective_calibration_locked_strategy.csv','selected_before_final':True,'cutoff_safe_population':True,'official_contract_compatible':True,'external_features_used':False},
 {'segment':'developing','source_notebook':'developing_routing','evidence_path':'docs/lumpy_02_forecasting/stages/developing_routing/developing_routing_locked_strategy.csv','selected_before_final':True,'cutoff_safe_population':True,'official_contract_compatible':True,'external_features_used':False},
 {'segment':'dormant','source_notebook':'chronology_strategy','evidence_path':'docs/lumpy_02_forecasting/stages/chronology_strategy/chronology_strategy_strategy_assignment_all_690.csv','selected_before_final':True,'cutoff_safe_population':True,'official_contract_compatible':True,'external_features_used':False},
]); evidence=audit.evidence_checks(project_root,registry); evidence.to_csv(DOC/'methodology_audit_champion_evidence.csv',index=False); display(evidence); assert evidence.passed.all()

assignments=pd.read_csv(project_root/'docs/lumpy_02_forecasting/stages/chronology_strategy/chronology_strategy_strategy_assignment_all_690.csv'); population=pd.DataFrame([
 {'guardrail':'classification_frozen_at_cutoff','passed':assignments.sku_id.nunique()==690,'detail':'Lifecycle and frequency assignments cover all 690 SKUs.'},
 {'guardrail':'positive_sku_not_selector','passed':'positive_sku' not in assignments.columns,'detail':'Official positive demand is not present in the routing assignment table.'},
 {'guardrail':'dormant_retained_for_governance','passed':forecasts.loc[forecasts.segment.eq('dormant'),'sku_id'].nunique()==53,'detail':'Dormant SKUs are retained but operationally flagged.'},
 {'guardrail':'actionable_excludes_dormant','passed':not actionable.segment.eq('dormant').any(),'detail':'Actionable result excludes dormant manual-review population.'},
 {'guardrail':'official_actuals_not_model_features','passed':True,'detail':'Target, WMAPE, MASE and reliability are evaluation outputs; documented by SKU delivery tables.'},
]); population.to_csv(DOC/'methodology_audit_population_guardrails.csv',index=False); display(population); assert population.passed.all()

summary=pd.DataFrame([{'contract_checks':len(checks),'contract_passed':int(checks.passed.sum()),'champion_sources':len(evidence),'champion_sources_passed':int(evidence.passed.sum()),'population_guardrails':len(population),'population_guardrails_passed':int(population.passed.sum()),'overall_passed':bool(checks.passed.all() and evidence.passed.all() and population.passed.all())}]); summary.to_csv(DOC/'methodology_audit_audit_summary.csv',index=False); display(summary)


## Final export

We publish stable 02 outputs for the results review, planning review and future-forecast stages.


In [ ]:
final_score = lumpy_consolidated_scorecard

roots = [Path.cwd(), Path.cwd().parent, Path(r"D:/Lumpy_Fellas-/Inchscape Repo")]
project_root = next(root.resolve() for root in roots if (root / "src/lumpy_consolidated_scorecard.py").exists())
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

importlib.reload(final_score)

result_root = project_root / "results/lumpy_02_forecasting"
doc_root = project_root / "docs/lumpy_02_forecasting"
stage_results = result_root / "stages"
stage_docs = doc_root / "stages"
result_root.mkdir(parents=True, exist_ok=True)
doc_root.mkdir(parents=True, exist_ok=True)

actionable = pd.read_csv(
    stage_results / "selective_calibration/selective_calibration_recommended_actionable_forecasts.csv",
    parse_dates=["block_start"],
)
full_champions = pd.read_csv(
    stage_results / "champion_consolidation/champion_consolidation_all_690_champion_forecasts.csv",
    parse_dates=["block_start"],
)
dormant = full_champions.loc[full_champions.segment.eq("dormant")].copy()
final_forecasts = pd.concat([actionable, dormant], ignore_index=True).sort_values(
    ["segment", "sku_id", "block_number"]
)
actionable_forecasts = final_forecasts.loc[final_forecasts.segment.ne("dormant")].copy()

delivery_all = pd.read_csv(stage_results / "delivery/delivery_all_690_sku_delivery.csv")
delivery_actionable = pd.read_csv(stage_results / "delivery/delivery_actionable_637_sku_delivery.csv")
policy_all = pd.read_csv(stage_results / "operational_policy/operational_policy_all_690_operational_policy.csv")

headline = pd.concat([
    final_score.score(final_forecasts).assign(scorecard="full_690_governance"),
    final_score.score(actionable_forecasts).assign(scorecard="actionable_637"),
], ignore_index=True)
segments = final_score.score(actionable_forecasts, group_column="segment")

final_forecasts.to_csv(result_root / "lumpy_02_final_690_block_forecasts.csv", index=False)
actionable_forecasts.to_csv(result_root / "lumpy_02_actionable_637_block_forecasts.csv", index=False)
delivery_all.to_csv(result_root / "lumpy_02_all_690_sku_delivery.csv", index=False)
delivery_actionable.to_csv(result_root / "lumpy_02_actionable_637_sku_delivery.csv", index=False)
policy_all.to_csv(result_root / "lumpy_02_operational_policy.csv", index=False)
headline.to_csv(doc_root / "lumpy_02_headline_scorecard.csv", index=False)
segments.to_csv(doc_root / "lumpy_02_segment_scorecard.csv", index=False)

assert final_forecasts.sku_id.nunique() == 690
assert actionable_forecasts.sku_id.nunique() == 637
assert len(final_forecasts) == 4140

display(headline[[
    "scorecard", "all_skus", "positive_skus", "under_50", "under_70", "under_100",
    "median_wmape", "portfolio_wmape", "bias_pct",
]])
print("Standalone workflow complete. Stable outputs written to:", result_root)
